# SAADE–AUA v6.1 · УСЕ В ОДНОМУ — генерація REAL–TTS і перевірка адаптивного навчання

Цей ноутбук **створює справжні текстово зіставлені пари**: підтверджений реальний запис Common Voice → його транскрипція → українська модель Piper → синтезоване WAV того самого тексту. Потім запускає E1 Static / E2 Random / E3 Online Adaptive на однаковому наборі та 21 акустичному сценарії.

**Наукова мета:** перевірити ΔASRI = ASRI(E3) − ASRI(E2). Позитивний ефект не гарантований. Пілот із 5 доступними голосами Piper **не є доказом узагальнення на десятки незалежних синтетичних голосів або генераторів**.

Робочі шляхи: `audio_real_fake/AUA` тільки для читання; **всі нові WAV, моделі та звіти пишуться виключно** в `audio_real_fake/results/SAADE_AUA_2026_v6_MATCHED_TTS/`. Папку `Melnyk_dus` не використовуємо.

**Послідовність `Run All`:** налаштування → метадані → вибір текстів → отримання TTS-голосів → реальна генерація → аудит пар → E1/E2/E3 → графіки/аудіо → ZIP. **Hidden test закрито за замовчуванням.**


## 0. Один блок налаштувань

Стандартний режим **`matched_tts`**, без автоматичного переходу на старий пілот. Якщо TTS не встановлено або немає доступу до моделей, блок генерації повідомить точну причину й збереже таблицю завдань. Синтез не підмінюється шумом чи фальшивими WAV.

Для першого контрольованого запуску використовується до 600 текстів Common Voice (ліцензія `CC0-1.0`, статус `confirmed`, ідентифікований мовець). `Piper` завантажує відкриті українські голоси до **окремої директорії результатів**. Моделі з невідомими або несумісними умовами використання не слід публікувати; перед поширенням корпусу перевірте MODEL_CARD кожного голосу.


In [ ]:
from pathlib import Path
from dataclasses import replace
from zipfile import ZipFile, ZIP_DEFLATED
import sys, os, types, shutil, json, importlib.util, hashlib, time, subprocess, re
import numpy as np
import pandas as pd
from IPython.display import display, Audio, Markdown

PROJECT = Path('/home/jovyan/work/audio_real_fake')
AUA = PROJECT / 'AUA'
RESULTS_ROOT = PROJECT / 'results'
MATCHED_OUT = RESULTS_ROOT / 'SAADE_AUA_2026_v6_MATCHED_TTS'
DATA_MODE = 'matched_tts'   # НІЯКОГО тихого переходу до PILOT

# Реальна генерація: за доступу до Інтернету завантажує голоси з Hugging Face.
# Для офлайнового запуску вкажіть локальні шляхи у LOCAL_PIPER_VOICES.
RUN_TTS_GENERATION = True
AUTO_INSTALL_TTS_PACKAGES = True  # Встановити лише piper-tts / huggingface_hub, не змінюючи torch/CUDA
AUTO_DOWNLOAD_PIPER_VOICES = True
TTS_REPO = 'rhasspy/piper-voices'
VOICE_ALLOCATION = {
    'train': ('lada', 'mykyta', 'oleksa'),
    'validation': ('tetiana',),
    'hidden_test': ('ukrainian_tts',),
}
LOCAL_PIPER_VOICES = {}   # {'lada': '/absolute/path/to/uk_UA-lada-x_low.onnx', ...}
SELECT_ONLY_CC0 = True
VERIFY_REAL_WAV_SHA = True
TTS_MAX_UTTERANCES = 600
MAX_SOURCE_DURATION_SEC = 8.0
MIN_SOURCE_DURATION_SEC = 2.0
MAX_TTS_DURATION_SEC = 8.0
MIN_TTS_DURATION_SEC = 2.0
MAX_DURATION_RATIO = 2.0
SEED_SELECTION = 42

# Навчання головного дослідження (залиште True після успішної генерації).
RUN_TRAINING = True
SEEDS = (42, 123, 777)
MODELS = ('spectrogram_cnn', 'raw_wave_cnn', 'mfcc_svm')
EPOCHS = 15
UPDATE_EVERY = 3
ADAPT_MAX_PER_CLASS = 24
MAX_PARENT_FILES = 900
UNKNOWN_TRAIN_CAP = 150
TARGET_SEGMENTS = 1200
AUGMENTATION_PROBABILITY = .50
SHOW_AUDIO_AND_PLOTS = True
SHOW_FULL_REPORT_PLOTS = True
BOOTSTRAPS = 500

FREEZE_FOR_HIDDEN = False
RUN_HIDDEN_ONCE = False
HIDDEN_CONFIRMATION = ''

assert DATA_MODE == 'matched_tts'
assert AUA.is_dir() and (AUA/'01_audio').is_dir(), f'Немає AUA: {AUA}'
assert MATCHED_OUT.resolve().is_relative_to(RESULTS_ROOT.resolve())
assert not MATCHED_OUT.resolve().is_relative_to(AUA.resolve())
assert 'Melnyk_dus' not in str(MATCHED_OUT)
assert shutil.which('ffmpeg'), 'Потрібен системний ffmpeg'
for dep in ('numpy','pandas','scipy','sklearn','soundfile','torch','matplotlib','joblib','statsmodels'):
    if importlib.util.find_spec(dep) is None:
        raise ImportError(f'Відсутній {dep}; встановіть пакет у поточний kernel Jupyter')
MATCHED_OUT.mkdir(parents=True, exist_ok=True)
print('AUA (read-only):', AUA)
print('OUTPUT:', MATCHED_OUT)
print('Усі нові файли лише в audio_real_fake/results; Melnyk_dus не використовується.')

## 1. ВБУДОВАНЕ ЯДРО SAADE

Нижче розташовані п’ять згорнутих комірок із **повним вихідним кодом** модулів v5: обробка сигналу, підтверджене походження AUA, сегментація, візуалізації та online adaptive. Вони завантажуються **в пам’ять поточного kernel**, без створення `.py` у вашому проєкті. За потреби розгорніть їх для перевірки алгоритмів. Це частина одного ноутбука.

In [ ]:
# Код модуля: saade_final_core.py (вбудований, інших файлів не потрібно)
_module_source = r'''
"""SAADE: Scenario-Adaptive Audio Deepfake Evaluation for AUA.
User-facing executable research scaffold. Python 3.10+, uses local server media.
All writes are confined to output directory. Never edits AUA.
"""
from __future__ import annotations
import dataclasses, hashlib, importlib.util, json, math, os, re, shutil, subprocess, sys, warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import signal
from scipy.fft import dct
from sklearn.metrics import (balanced_accuracy_score, f1_score, roc_curve, roc_auc_score,
    confusion_matrix, accuracy_score)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

AUDIO_EXT={'.wav','.flac','.mp3','.ogg','.m4a','.aac'}
CATEGORIES=['noise','compression','reverberation','lowpass']
LEVELS={
    'noise':{1:20.,2:10.,3:0.},
    'compression':{1:64,2:32,3:16},
    'reverberation':{1:0.2,2:0.5,3:0.9},
    'lowpass':{1:7000.,2:5000.,3:3400.},
}
EXTRA_LEVELS={
    'noise':{4:15.,5:5.},
    'compression':{4:48,5:24},
    'reverberation':{4:0.35,5:0.7},
    'lowpass':{4:6000.,5:4200.},
}
# NEW PROTOCOL: five severity levels ordered from light to severe.
# V1_LEGACY levels 1..3 are preserved unchanged for reproducibility.
FIVE_LEVELS={
    'noise':{1:20.,2:15.,3:10.,4:5.,5:0.},
    'compression':{1:64,2:48,3:32,4:24,5:16},
    'reverberation':{1:0.2,2:0.35,3:0.5,4:0.7,5:0.9},
    'lowpass':{1:7000.,2:6000.,3:5000.,4:4200.,5:3400.},
}
VALID_SCENARIO_PROFILES=('v1_3level','v2_5level')

def available_levels(profile):
    if profile=='v1_3level': return (1,2,3)
    if profile=='v2_5level': return (1,2,3,4,5)
    raise ValueError(f'Unknown scenario_profile={profile}; choose {VALID_SCENARIO_PROFILES}')

def scenario_severity(category, level, profile='v1_3level'):
    if category=='baseline': return None
    if profile=='v2_5level': return FIVE_LEVELS[category][int(level)]
    if int(level) in LEVELS[category]: return LEVELS[category][int(level)]
    if int(level) in EXTRA_LEVELS[category]: return EXTRA_LEVELS[category][int(level)]
    raise ValueError((category,level,profile))

@dataclass
class Config:
    root: str='/home/jovyan/work/audio_real_fake/AUA'
    out: str='/home/jovyan/work/audio_real_fake/results/SAADE_AUA_2026_v5_FINAL'
    seed: int=42
    sample_rate: int=16000
    seconds: float=4.0
    min_seconds: float=2.0
    max_items: int=400  # 0=all available, equal counts per label when limiting
    max_per_group: int=30
    epochs: int=4
    batch_size: int=12
    lr: float=0.0005
    num_workers: int=0
    augmentation_probability: float=0.5
    use_cache: bool=True
    max_ambiguous_label_fraction: float=0.05
    allow_weak_groups: bool=False  # explicit override for exploratory only
    models: tuple=('spectrogram_cnn','raw_wave_cnn','mfcc_svm')
    repo_asvspoof5: str='/home/jovyan/work/asvspoof5'
    repo_aasist: str='/home/jovyan/work/aasist'
    wavlm_checkpoint: str='microsoft/wavlm-base-plus'
    bootstraps: int=200
    scenario_seed: int=1001   # fixed across training seeds; prevents comparison confounding
    scenario_cache_dir: str=''  # optional shared immutable cache across seed repeats
    scenario_profile: str='v1_3level' # v1_3level=13; v2_5level=21 ordered scenarios
    test_finalization: bool=False # frozen test locked and one-time evaluation gate

    def __post_init__(self):
        # Explicit safety invariant: this experiment never writes into AUA or Melnyk's workspace.
        output=Path(self.out).expanduser().resolve()
        dataset=Path(self.root).expanduser().resolve()
        if output == dataset or dataset in output.parents:
            raise ValueError('OUT may not be inside the read-only AUA data folder')
        if any('melnyk' in part.casefold() for part in output.parts):
            raise ValueError('Writing to Melnyk_dus or any Melnyk folder is prohibited')


def save_json(path, obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding='utf-8')


def normalize_path_key(s):
    return str(s).replace('\\','/').strip().lstrip('./')


def getval(row, *names):
    for name in names:
        v=row.get(name,'')
        if v is not None and not (isinstance(v,float) and np.isnan(v)):
            s=str(v).strip()
            if s and s.lower() not in ('nan','none','null'):
                return s
    return ''


def source_from_path(s):
    ss=str(s).lower().replace('\\','/')
    if '/own_processed/' in ss: return 'AUA_own'
    if '/common_voice/' in ss: return 'CommonVoice'
    if '/fleurs/' in ss: return 'FLEURS'
    if '/new-tts/' in ss: return 'AUA_new_TTS'
    if '/drssu/' in ss: return 'DRSSU'
    if '/synthetic/' in ss: return 'synthetic_other'
    return 'external_other'


def safe_key(s):
    return re.sub(r'[^\w.-]+','_',str(s), flags=re.UNICODE)[:160]


def _read_csv(path):
    if not Path(path).exists(): return pd.DataFrame()
    return pd.read_csv(path, dtype=str, keep_default_na=False, low_memory=False)


def build_catalog(cfg:Config):
    """Build conservative catalog from the 01_audio directory and the REAL AUA metadata.
    Never use 05_splits for sample counts. Suspect provenance/labels are flagged, not silently verified.
    """
    root=Path(cfg.root).expanduser().resolve()
    if not (root/'01_audio').is_dir():
        raise FileNotFoundError(f'Cannot find {root}/01_audio. Set cfg.root correctly.')
    aud=root/'01_audio'
    meta=_read_csv(aud/'metadata_audio.csv')
    own=_read_csv(aud/'own_processed'/'metadata_audio.csv')
    newtts=_read_csv(aud/'synthetic'/'new-tts'/'metadata_new-tts.csv')
    own_regions=_read_csv(aud/'own_master'/'metadata_audio.csv')
    master_by_speaker={}
    if not own_regions.empty:
        for _,rr in own_regions.iterrows():
            sid=getval(rr,'speaker_id')
            if sid: master_by_speaker.setdefault(sid,rr.to_dict())
    canonical_dirs=[aud/'own_processed',aud/'external_selected',aud/'synthetic']
    paths=[]
    for folder in canonical_dirs:
        if folder.exists():
            paths.extend(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in AUDIO_EXT)
    paths=sorted(set(paths))
    byname=defaultdict(list)
    for p in paths: byname[p.name].append(p)
    meta_by_path, meta_by_name={}, defaultdict(list)
    for _,rr in meta.iterrows():
        row=rr.to_dict()
        key=getval(row,'working_path')
        if key:
            meta_by_path[normalize_path_key(key)]=row
        name=getval(row,'file_name')
        if name: meta_by_name[Path(name).name].append(row)
    own_by_name={Path(getval(r.to_dict(),'working_path','file_name')).name:r.to_dict() for _,r in own.iterrows()}
    tts_by_name={Path(getval(r.to_dict(),'filename')).name:r.to_dict() for _,r in newtts.iterrows()}
    records=[]
    for p in paths:
        rp=p.relative_to(root).as_posix(); r=None
        for candidate in (str(p),rp):
            if normalize_path_key(candidate) in meta_by_path:
                r=meta_by_path[normalize_path_key(candidate)];break
        if r is None and len(meta_by_name.get(p.name,[]))==1:
            r=meta_by_name[p.name][0]
        r=r or {}; ownr=own_by_name.get(p.name,{}) if '/own_processed/' in '/'+rp else {}
        tr=tts_by_name.get(p.name,{}) if '/new-tts/' in '/'+rp else {}
        src=source_from_path(rp)
        # A named corpus can legitimately contain BOTH real and fake examples.
        # Class comes from canonical folder + explicit mode, NOT the corpus name.
        declared_source=getval(r,'source_dataset').lower()
        if 'drssu' in declared_source or '/drssu/' in '/'+rp.lower():src='DRSSU'
        elif 'fleurs' in declared_source:src='FLEURS'
        elif 'common voice' in declared_source or 'common_voice' in declared_source:src='CommonVoice'
        source_label=1 if '/synthetic/' in '/'+rp.replace('\\','/').lower() else 0
        mode=getval(r,'mode').lower()
        explicit_label=None
        if mode in ('real','bona fide','bona_fide','bonafide','genuine','human'):
            explicit_label=0
        elif mode in ('fake','spoof','synthetic','tts','vc','voice_conversion','generated'):
            explicit_label=1
        conflict=explicit_label is not None and explicit_label!=source_label
        label=source_label if not conflict else None
        sp=getval(ownr,'speaker_id') or getval(r,'speaker_id')
        basis=getval(r,'speaker_id_basis')
        srid=getval(ownr,'source_record_id') or getval(r,'source_record_id')
        shash=getval(ownr,'source_sha256') or getval(r,'source_sha256')
        voice=getval(tr,'voice_id')
        generator=getval(tr,'model_name')
        if src=='DRSSU' and label==1: generator='DRSSU_generator_unknown'  # may be refined from source metadata
        if not generator and label==1: generator=getval(r,'source_dataset') or src
        if sp:
            group=f'SPEAKER::{src}::{sp}'
            group_confidence='metadata_speaker'
        elif voice:
            group=f'VOICE::{src}::{generator}::{voice}'
            group_confidence='metadata_voice'
        elif shash:
            group=f'SOURCEHASH::{shash}'
            group_confidence='source_sha256'
        elif srid:
            group=f'SOURCE_RECORD::{src}::{srid}'
            group_confidence='source_record_id'
        else:
            group=f'UNVERIFIED_FILE::{rp}'
            group_confidence='weak_file_fallback'
        region=master_by_speaker.get(sp,{}) if src=='AUA_own' else {}
        transcript=getval(ownr,'transcription') or getval(r,'transcription')
        records.append(dict(path=str(p),relative_path=rp,source=src,source_dataset=getval(r,'source_dataset') or src,
             label=label,label_source='folder_and_metadata' if explicit_label is not None and not conflict else 'folder_inferred',
             label_conflict=conflict,speaker_id=sp,speaker_id_basis=basis,source_record_id=srid,
             source_sha256=shash,working_sha256=getval(ownr,'working_sha256') or getval(r,'working_sha256'),
             group_id=group,group_confidence=group_confidence,voice_id=voice,generator=generator,
             transcription=transcript,script_id=getval(ownr,'script_id'),
             speaker_origin_group=getval(region,'speaker_origin_group'),
             script_profile=getval(region,'script_profile'),
             dialect_authenticity=getval(region,'dialect_authenticity'),
             approval_status=getval(ownr,'approval_status') or getval(r,'approval_status') or getval(tr,'approval_status'),
             license=getval(r,'license') or getval(tr,'license_terms')))
    df=pd.DataFrame(records)
    if df.empty: raise ValueError('No canonical media found in 01_audio.')
    # Duration: prefer reliable WAV header; for other codecs use soundfile info or ffprobe
    df['duration_sec']=[audio_info_seconds(Path(p)) for p in df.path]
    df['usable']=(df.label.notna())&(df.duration_sec>=cfg.min_seconds)&(~df.label_conflict)
    df['warning']=''
    df.loc[df.label_conflict,'warning']='CONFLICT: metadata mode versus folder label'
    df.loc[df.duration_sec.isna(),'warning']='unreadable audio'
    df.loc[df.group_confidence.eq('weak_file_fallback'),'warning']='WEAK GROUP: potential speaker/content leakage'
    dups=df[df.working_sha256.ne('') & df.working_sha256.duplicated(keep=False)]
    if len(dups):
        df.loc[dups.index,'warning']=df.loc[dups.index,'warning'].astype(str)+'; duplicate working SHA'
    # dedup repeated file hashes deterministically
    df.loc[df.working_sha256.ne('') & df.working_sha256.duplicated(keep='first'),'usable']=False
    out=Path(cfg.out);out.mkdir(parents=True,exist_ok=True)
    df.to_csv(out/'01_catalog_all.csv',index=False)
    issues=df[~df.usable | df.warning.ne('')]
    issues.to_csv(out/'01_catalog_issues.csv',index=False)
    summary=(df.groupby(['source','label'],dropna=False).agg(files=('path','size'),hours=('duration_sec',lambda x:x.sum()/3600),
         speakers=('speaker_id',lambda x: x[x.ne('')].nunique()),weak_groups=('group_confidence',lambda x: (x=='weak_file_fallback').sum())).reset_index())
    summary.to_csv(out/'01_sources_summary.csv',index=False)
    print('Catalog:',len(df),'files; usable:',int(df.usable.sum()),'hours usable:',round(df.loc[df.usable,'duration_sec'].sum()/3600,3))
    print(summary.to_string(index=False))
    if df.label_conflict.any(): print('WARNING: conflicting labels:',int(df.label_conflict.sum()))
    if (df.group_confidence=='weak_file_fallback').any(): print('WARNING: unverified grouping:',int((df.group_confidence=='weak_file_fallback').sum()))
    if not (out/'01_catalog_all.csv').exists():raise AssertionError('Catalog not saved')
    return df


def audio_info_seconds(path):
    try:
        import soundfile as sf
        return float(sf.info(str(path)).duration)
    except Exception:
        try:
            r=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=nokey=1:noprint_wrappers=1',str(path)],capture_output=True,text=True,timeout=25)
            return float(r.stdout.strip()) if r.returncode==0 else np.nan
        except Exception:
            return np.nan


def get_experimental_subset(catalog:pd.DataFrame,cfg:Config):
    out=Path(cfg.out)
    d=catalog[catalog.usable & catalog.label.notna()].copy()
    d['label']=d.label.astype(int)
    if d.label.nunique()<2: raise ValueError('Both real=0 and fake=1 required after catalog filtering.')
    if cfg.max_items:
        rng=np.random.default_rng(cfg.seed)
        chosen=[]
        for lab in (0,1):
            x=d[d.label.eq(lab)].copy()
            n=int(cfg.max_items//2)
            if len(x)<n:
                warnings.warn(f'Only {len(x)} files for class {lab}; cannot reach {n}')
                n=len(x)
            # round-robin over source then group to cap dominance
            x=x.iloc[rng.permutation(len(x))]
            x['grank']=x.groupby('group_id').cumcount()
            x=x[x.grank<cfg.max_per_group]
            x['source_rank']=x.groupby('source').cumcount()
            x=x.sort_values(['source_rank','grank'])
            chosen.append(x.head(n).drop(columns=['grank','source_rank']))
        d=pd.concat(chosen,ignore_index=True)
    else:
        d=d.reset_index(drop=True)
    d.to_csv(out/'02_experimental_subset.csv',index=False)
    print('Subset',len(d),'label counts:',d.label.value_counts().to_dict(), 'group counts:',d.groupby('label').group_id.nunique().to_dict())
    return d


def make_splits(d:pd.DataFrame,cfg:Config):
    """Multiple group-aware candidates, never split one group between train/val/test.
    Attempts class balance. No reliance on possibly contaminated existing 05_splits.
    """
    if not cfg.allow_weak_groups and d.group_confidence.eq('weak_file_fallback').any():
        n=int(d.group_confidence.eq('weak_file_fallback').sum())
        raise ValueError(f'{n} samples have unverifiable speaker/source groups. Enrich metadata or set cfg.allow_weak_groups=True for EXPLORATORY run.')
    if min(d.groupby('label').group_id.nunique())<3:
        raise ValueError('Fewer than three independent groups in one label; cannot create safe train/val/test.')
    best=None;best_score=float('inf')
    X=np.zeros((len(d),1));groups=d.group_id.astype(str).to_numpy(); y=d.label.to_numpy()
    for i in range(150):
        try:
            first=GroupShuffleSplit(n_splits=1,test_size=.40,random_state=cfg.seed+i)
            tr,rest=next(first.split(X,y,groups))
            second=GroupShuffleSplit(n_splits=1,test_size=.50,random_state=cfg.seed+10000+i)
            va0,te0=next(second.split(X[rest],y[rest],groups[rest]))
            va,te=rest[va0],rest[te0]
        except Exception:continue
        if any(len(set(y[idx]))!=2 for idx in (tr,va,te)):continue
        counts=np.array([[np.sum(y[idx]==label) for label in (0,1)] for idx in (tr,va,te)],dtype=float)
        target=np.array([[.60,.60],[.20,.20],[.20,.20]])*np.bincount(y,minlength=2)
        penalty=float(np.square((counts-target)/np.maximum(1,target)).sum())
        if penalty<best_score:best=(tr,va,te);best_score=penalty
    if best is None:raise ValueError('Cannot produce class-complete group-disjoint splits. Check group size and metadata.')
    out=d.copy();out['split']=''
    for name,ix in zip(('train','validation','hidden_test'),best):out.iloc[ix,out.columns.get_loc('split')]=name
    groups_by={k:set(out[out.split.eq(k)].group_id) for k in ('train','validation','hidden_test')}
    for a in groups_by:
        for b in groups_by:
            if a!=b:assert not (groups_by[a]&groups_by[b]),f'Group leakage {a} {b}'
    # Exact same working hash must not straddle splits
    hashes=out[out.working_sha256.ne('')].groupby('working_sha256').split.nunique()
    if (hashes>1).any():raise AssertionError('Duplicate working SHA across splits; repair source groups.')
    # metadata-based source original links if known
    src=out[out.source_sha256.ne('')].groupby('source_sha256').split.nunique()
    if (src>1).any():raise AssertionError('Source SHA across splits; group by source_id or original hash.')
    out.to_csv(Path(cfg.out)/'03_locked_splits.csv',index=False)
    metrics=out.groupby(['split','label']).agg(n=('path','size'),groups=('group_id','nunique'),sources=('source','nunique')).reset_index()
    metrics.to_csv(Path(cfg.out)/'03_split_summary.csv',index=False)
    print(metrics.to_string(index=False))
    print('Group and exact hash checks: PASS; score',round(best_score,4))
    if len(set(out.source))>1:
        print('WARNING: class is confounded with corpus source. Report source-stratified results; see README.')
    return out


def frozen_hash(df):
    cols=['relative_path','split','label','group_id','source','speaker_id','working_sha256']
    cols=[*cols,*[c for c in ('segment_start_sec','parent_file_id','pair_id','text_key',
                                 'voice_id','source_sha256','generator','input_text_id') if c in df.columns]]
    s=df[cols].sort_values('relative_path').to_csv(index=False)
    return hashlib.sha256(s.encode('utf-8')).hexdigest()


def lock_hidden_test(df,cfg):
    out=Path(cfg.out); p=out/'03_SPLIT_LOCK.json';sha=frozen_hash(df)
    if p.exists():
        old=json.loads(p.read_text())
        if old['split_manifest_sha256']!=sha:raise RuntimeError('SPLITS CHANGED since first lock. New experiment ID required; do not reuse hidden test.')
    else:save_json(p,dict(split_manifest_sha256=sha,seed=cfg.seed,
      hidden_samples=int(df.split.eq('hidden_test').sum()),meaning='Freeze BEFORE adaptive validation; do not select or tune on test.'))
    print('Locked split SHA:',sha)
    return sha


@lru_cache(maxsize=512)
def read_wave(path,sr=16000,seconds=4.0,seed=0,train=False):
    import soundfile as sf
    p=Path(path)
    try:
        x,old_sr=sf.read(str(p),dtype='float32',always_2d=False)
    except Exception:
        # decoder fallback only; no mutation of source files
        cmd=['ffmpeg','-v','error','-i',str(p),'-ar',str(sr),'-ac','1','-f','f32le','pipe:1']
        r=subprocess.run(cmd,capture_output=True,timeout=45)
        if r.returncode: raise RuntimeError(f'ffmpeg cannot decode {p}: {r.stderr[:150]}')
        x=np.frombuffer(r.stdout,dtype='<f4').copy();old_sr=sr
    x=np.asarray(x,dtype=np.float32)
    if x.ndim==2:x=x.mean(axis=1)
    if old_sr!=sr:
        from math import gcd
        g=gcd(old_sr,sr);x=signal.resample_poly(x,sr//g,old_sr//g).astype('float32')
    if not np.isfinite(x).all():raise ValueError('NaN/Inf in decoded waveform '+str(p))
    need=int(sr*seconds)
    if len(x)>=need:
        off=int(np.random.default_rng(seed).integers(0,len(x)-need+1)) if train else (len(x)-need)//2
        x=x[off:off+need]
    else:x=np.pad(x,(0,need-len(x)))
    return np.ascontiguousarray(x,dtype=np.float32)


# V2_SEGMENT_PATCH: deterministic, non-overlapping fixed offset windows.
def read_fixed_segment(path, sr=16000, seconds=4.0, start_sec=None, seed=0, train=False):
    """Read exact window if start_sec is supplied; else legacy center/random crop.
    Segment windows must be assigned to a split via their parent BEFORE expansion.
    """
    if start_sec is None or (isinstance(start_sec, float) and np.isnan(start_sec)):
        return read_wave(str(path),sr,seconds,seed,train)
    import soundfile as sf
    start_sec=float(start_sec)
    if not np.isfinite(start_sec) or start_sec<0:raise ValueError('Invalid segment offset')
    length=int(round(seconds*sr))
    try:
        with sf.SoundFile(str(path)) as handle:
            old_sr=int(handle.samplerate)
            handle.seek(int(round(start_sec*old_sr)))
            x=handle.read(int(np.ceil(seconds*old_sr))+3,dtype='float32',always_2d=False)
    except Exception:
        cmd=['ffmpeg','-v','error','-ss',str(start_sec),'-i',str(path),'-t',str(seconds+0.05),
             '-ar',str(sr),'-ac','1','-f','f32le','pipe:1']
        run=subprocess.run(cmd,capture_output=True,timeout=50)
        if run.returncode:raise RuntimeError('Unable to decode segment: '+str(path))
        x=np.frombuffer(run.stdout,dtype='<f4').copy();old_sr=sr
    x=np.asarray(x,dtype=np.float32)
    if x.ndim>1:x=x.mean(axis=1)
    if old_sr!=sr:
        from math import gcd
        g=gcd(old_sr,sr);x=signal.resample_poly(x,sr//g,old_sr//g).astype('float32')
    if len(x)>=length:x=x[:length]
    else:x=np.pad(x,(0,length-len(x)))
    if not np.isfinite(x).all():raise ValueError('Nonfinite signal: '+str(path))
    return np.ascontiguousarray(x,dtype=np.float32)


def _ffmpeg_compress(x,sr,kbps):
    if not shutil.which('ffmpeg'):raise FileNotFoundError('ffmpeg required for true MP3 compression scenarios. Install ffmpeg; no silent fake compression.')
    enc=subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-f','f32le','-ar',str(sr),'-ac','1','-i','pipe:0',
        '-codec:a','libmp3lame','-b:a',f'{int(kbps)}k','-f','mp3','pipe:1'],input=np.asarray(x,dtype='<f4').tobytes(),capture_output=True,timeout=40)
    if enc.returncode:raise RuntimeError('MP3 encoding error: '+enc.stderr.decode(errors='replace')[:300])
    dec=subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-f','mp3','-i','pipe:0','-f','f32le','-ar',str(sr),'-ac','1','pipe:1'],input=enc.stdout,capture_output=True,timeout=40)
    if dec.returncode:raise RuntimeError('MP3 decoding error: '+dec.stderr.decode(errors='replace')[:300])
    out=np.frombuffer(dec.stdout,dtype='<f4').copy()
    if len(out)>=len(x):out=out[:len(x)]
    else:out=np.pad(out,(0,len(x)-len(out)))
    return out.astype('float32')


def transform(x,category,level,sr=16000,seed=42,severity_override=None):
    """Deterministic perturbation; class label remains unchanged."""
    if category=='baseline':return np.asarray(x,dtype=np.float32).copy()
    table=LEVELS if level in (1,2,3) else EXTRA_LEVELS
    if category not in table or level not in table[category]:raise ValueError((category,level))
    severity=table[category][level] if severity_override is None else severity_override
    x=np.asarray(x,dtype='float32');rng=np.random.default_rng(seed)
    if category=='noise':
        noise=rng.normal(0,1,len(x)).astype('float32')
        power=max(1e-7,float(np.mean(x*x)));npow=max(1e-7,float(np.mean(noise*noise)))
        result=x+noise*np.sqrt(power/(npow*10**(severity/10)))
    elif category=='compression':result=_ffmpeg_compress(x,sr,severity)
    elif category=='reverberation':
        # EXPLICIT: synthetic, NOT a measured physical RIR; RT60 is a nominal decay parameter.
        rt60=float(severity)
        n=min(int(sr*1.2),len(x));tt=np.arange(n)/sr
        rir=(rng.normal(size=n)*np.exp(-6.9078*tt/rt60)).astype('float32')
        rir[0]=max(2.0,float(np.max(np.abs(rir))))
        rir/=max(1e-8,float(np.sum(np.abs(rir))))
        result=signal.fftconvolve(x,rir,mode='full')[:len(x)].astype('float32')
        # preserve perceived level, avoid globally normalizing fake/noise too much
        rms0=np.sqrt(np.mean(x*x)+1e-10);rms1=np.sqrt(np.mean(result*result)+1e-10)
        result=result*(rms0/max(rms1,1e-8))
    elif category=='lowpass':
        cutoff=min(float(severity),sr*.49)
        sos=signal.butter(6,cutoff,fs=sr,btype='lowpass',output='sos')
        result=signal.sosfilt(sos,x)
    else:raise KeyError(category)
    return np.nan_to_num(result).astype('float32')


def deterministic_seed(*parts):
    s='|'.join(str(a) for a in parts).encode('utf-8');return int(hashlib.blake2b(s,digest_size=4).hexdigest(),16)


def scenario_manifest(df,split='validation',extra=False,profile='v1_3level'):
    d=df[df.split.eq(split)]
    rows=[]
    for r in d.itertuples():
        common=dict(path=r.path,relative_path=r.relative_path,group_id=r.group_id,speaker_id=r.speaker_id,
                    voice_id=str(getattr(r,'voice_id','')), pair_id=str(getattr(r,'pair_id','')),
                    text_key=str(getattr(r,'text_key','')), input_text_id=str(getattr(r,'input_text_id','')), 
                    segment_start_sec=float(getattr(r,'segment_start_sec',float('nan'))),
                    parent_file_id=str(getattr(r,'parent_file_id','')),
                    label=int(r.label),source=r.source,generator=r.generator,split=split)
        rows.append(dict(**common,category='baseline',level=0,value=np.nan,scenario_id=f'{r.relative_path}::baseline::0'))
        for c in CATEGORIES:
            levels=(list(FIVE_LEVELS[c].items()) if profile=='v2_5level' else
                    list(LEVELS[c].items())+(list(EXTRA_LEVELS[c].items()) if extra else []))
            for lev,value in levels:
                rows.append(dict(**common,category=c,level=lev,value=value,scenario_id=f'{r.relative_path}::{c}::{lev}'))
    return pd.DataFrame(rows)


def mel_filter(sr=16000,n_fft=256,n_mels=40):
    def hz_to_mel(f):return 2595*np.log10(1+f/700)
    def mel_to_hz(m):return 700*(10**(m/2595)-1)
    hz=mel_to_hz(np.linspace(hz_to_mel(0),hz_to_mel(sr/2),n_mels+2))
    bins=hz/(sr/2)*(n_fft//2)
    f=np.arange(n_fft//2+1);out=np.zeros((n_mels,len(f)),dtype='float32')
    for i in range(n_mels):
        l,c,r=bins[i:i+3]
        out[i]=np.maximum(0,np.minimum((f-l)/max(1e-8,c-l),(r-f)/max(1e-8,r-c)))
    return out


def mfcc_features(wave,sr=16000):
    wave=np.asarray(wave,dtype='float32')
    _,_,z=signal.stft(wave,fs=sr,nperseg=512,noverlap=352,boundary='zeros')
    power=np.abs(z)**2; fb=mel_filter(sr,512,40)
    m=np.log(np.maximum(1e-8,fb@power)); cc=dct(m,type=2,norm='ortho',axis=0)[:20]
    delta=np.diff(cc,axis=1,prepend=cc[:,0:1])
    return np.concatenate([cc.mean(axis=1),cc.std(axis=1),delta.mean(axis=1),delta.std(axis=1)]).astype('float32')


def metric_row(y,scores,threshold=None):
    y=np.asarray(y,dtype=int);scores=np.asarray(scores,dtype=float)
    if threshold is None:threshold=0.0
    pred=(scores>=threshold).astype(int)
    out=dict(n=len(y),real=int(sum(y==0)),fake=int(sum(y==1)),threshold=float(threshold),
            balanced_accuracy=float(balanced_accuracy_score(y,pred)),accuracy=float(accuracy_score(y,pred)),
            f1_fake=float(f1_score(y,pred,zero_division=0)))
    cm=confusion_matrix(y,pred,labels=[0,1]);tn,fp,fn,tp=cm.ravel()
    out.update(false_alarm_rate=float(fp/max(1,tn+fp)),miss_rate=float(fn/max(1,tp+fn)))
    if len(set(y))==2:
        fpr,tpr,ths=roc_curve(y,scores);fnr=1-tpr;idx=np.nanargmin(abs(fnr-fpr))
        out['eer']=float((fpr[idx]+fnr[idx])/2);out['roc_auc']=float(roc_auc_score(y,scores))
    else:out.update(eer=np.nan,roc_auc=np.nan)
    return out


def best_threshold(y,scores):
    y=np.asarray(y).astype(int);scores=np.asarray(scores,float)
    if len(np.unique(y))<2:raise ValueError('Cannot calibrate threshold with one class')
    fpr,tpr,ths=roc_curve(y,scores)
    idx=int(np.nanargmax(tpr-fpr))
    return float(ths[idx])


def asri_table(preds,expected_levels=3):
    """Scenario-balanced ASRI: average over four perturbation families x requested severities.
    Scale max(0, 2*BA-1). Missing cells => NaN, not a fake passing score.
    """
    levels=tuple(range(1,int(expected_levels)+1))
    x=preds[preds.category.isin(CATEGORIES)&preds.level.isin(levels)]
    rows=[]
    for key,g in x.groupby(['model','regime']):
        cell=[]
        for c in CATEGORIES:
            for level in levels:
                gg=g[(g.category==c)&(g.level==level)]
                if gg.empty or gg.label.nunique()!=2:continue
                mm=metric_row(gg.label,gg.score,float(gg.threshold.iloc[0]))
                cell.append((c,level,max(0.,2*mm['balanced_accuracy']-1)))
        complete=len(cell)==4*len(levels)
        value=float(np.mean([j[2] for j in cell])) if complete else np.nan
        rows.append(dict(model=key[0],regime=key[1],asri=value,complete_cells=len(cell),required_cells=4*len(levels),scenario_profile=('v2_5level' if len(levels)==5 else 'v1_3level')))
    return pd.DataFrame(rows)


def get_scenario_wave(r,cfg):
    """Deterministic disk cache for costly MP3/reverb scenario waveforms.
    Cache key includes source size and mtime so updated server files invalidate entries.
    Original AUA files remain untouched."""
    path=Path(r.path)
    if cfg.use_cache:
        st=path.stat()
        payload='|'.join(map(str,[r.relative_path,st.st_size,st.st_mtime_ns,r.category,r.level,
                                  cfg.scenario_seed,cfg.seconds,cfg.sample_rate,cfg.scenario_profile,getattr(r,'segment_start_sec',None)]))
        digest=hashlib.sha256(payload.encode()).hexdigest()
        folder=Path(cfg.scenario_cache_dir or cfg.out)/'cache_scenario_waves'/digest[:2]
        cached=folder/(digest+'.npy')
        if cached.exists():
            wav=np.load(cached,allow_pickle=False)
            if wav.ndim==1 and len(wav)==int(cfg.seconds*cfg.sample_rate):return wav
    wav=read_fixed_segment(r.path,cfg.sample_rate,cfg.seconds,
            start_sec=getattr(r,'segment_start_sec',None),seed=cfg.scenario_seed)
    seed=deterministic_seed(cfg.scenario_seed,r.relative_path,r.category,r.level,cfg.scenario_profile)
    if r.category!='baseline':
        severity=scenario_severity(r.category,r.level,cfg.scenario_profile)
        wav=transform(wav,r.category,r.level,cfg.sample_rate,seed,severity_override=severity)
    if cfg.use_cache:
        folder.mkdir(parents=True,exist_ok=True)
        temp=folder/(digest+'.partial.npy')
        np.save(temp,wav)
        os.replace(temp,cached)
    return wav


def run_scenario_predictions(df,model,model_name,regime,cfg,threshold=0.0,split='validation',extra=False):
    scen=scenario_manifest(df,split,extra=extra,profile=cfg.scenario_profile)
    rows=[]
    for i,r in enumerate(scen.itertuples()):
        wav=get_scenario_wave(r,cfg)
        score=float(model_score(model,model_name,wav,cfg))
        rows.append(dict(scenario_id=r.scenario_id,path=r.path,relative_path=r.relative_path,
            group_id=r.group_id,source=r.source,generator=r.generator,speaker_id=r.speaker_id,
            voice_id=str(getattr(r,'voice_id','')),pair_id=str(getattr(r,'pair_id','')),
            text_key=str(getattr(r,'text_key','')),input_text_id=str(getattr(r,'input_text_id','')),
            label=r.label,category=r.category,level=r.level,value=r.value,split=split,
            segment_start_sec=getattr(r,'segment_start_sec',None),
            parent_file_id=getattr(r,'parent_file_id',''),
            model=model_name,regime=regime,score=score,threshold=float(threshold),
            pred=int(score>=threshold)))
        if (i+1)%500==0:print('  scenarios scored:',i+1,'/',len(scen))
    return pd.DataFrame(rows)


def summarize_predictions(preds,cfg,report_tag=None):
    summary=[]
    for (model,regime,category,level),g in preds.groupby(['model','regime','category','level']):
        if g.label.nunique()!=2:continue
        row=metric_row(g.label,g.score,float(g.threshold.iloc[0]))
        row.update(model=model,regime=regime,category=category,level=int(level))
        summary.append(row)
    s=pd.DataFrame(summary)
    if not s.empty:
        suffix=('_'+safe_key(report_tag)) if report_tag else ''
        s.to_csv(Path(cfg.out)/f'08_scenario_metrics{suffix}.csv',index=False)
        a=asri_table(preds,expected_levels=len(available_levels(cfg.scenario_profile)))
        a.to_csv(Path(cfg.out)/f'09_ASRI{suffix}.csv',index=False)
    else:a=pd.DataFrame()
    return s,a


def validation_difficulty(p):
    """Per-category mean validation classification error, balanced equally over classes & levels."""
    scenario_levels=sorted(set(p.loc[p.category.isin(CATEGORIES),'level'].astype(int))-{0})
    v=p[p.category.isin(CATEGORIES)&p.level.isin(scenario_levels)]
    weights={}
    for c in CATEGORIES:
        v1=v[v.category==c]
        if v1.empty: weights[c]=.25;continue
        losses=[]
        for lev in scenario_levels:
            g=v1[v1.level==lev]
            if g.empty:continue
            for lab in (0,1):
                g2=g[g.label==lab]
                if len(g2):losses.append(float((g2.pred!=g2.label).mean()))
        weights[c]=float(np.mean(losses)) if losses else .25
    # smoothing so every category retained; no test access
    a=np.array([weights[c]+.05 for c in CATEGORIES],float);a/=a.sum()
    return dict(zip(CATEGORIES,a.tolist()))


def paired_group_bootstrap_delta(preds,model,regime_a='random',regime_b='adaptive',B=200,seed=42):
    x=preds[(preds.model==model)&(preds.regime.isin([regime_a,regime_b]))]
    a=x[x.regime==regime_a];b=x[x.regime==regime_b]
    key=['scenario_id','group_id','category','level','label']
    paired=a[key+['score','threshold']].merge(b[key+['score','threshold']],on=key,suffixes=('_a','_b'))
    if paired.empty:return dict(status='NO_PAIRED_DATA')
    group_ids=paired.group_id.unique();rng=np.random.default_rng(seed)
    if len(group_ids)<3:return dict(status='TOO_FEW_GROUPS',n_groups=len(group_ids))
    scenario_levels=sorted(set(paired.loc[paired.category.isin(CATEGORIES),'level'].astype(int))-{0})
    def mean_delta(sub):
        sub=sub[sub.category.isin(CATEGORIES)&sub.level.isin(scenario_levels)]
        deltas=[]
        for c in CATEGORIES:
            for lev in scenario_levels:
                g=sub[(sub.category==c)&(sub.level==lev)]
                if g.empty or g.label.nunique()<2:continue
                v1=balanced_accuracy_score(g.label,(g.score_a>=g.threshold_a).astype(int))
                v2=balanced_accuracy_score(g.label,(g.score_b>=g.threshold_b).astype(int))
                deltas.append(max(0,2*v2-1)-max(0,2*v1-1))
        return float(np.mean(deltas)) if len(deltas)==4*len(scenario_levels) else np.nan
    observed=mean_delta(paired);boots=[]
    grouped={g:paired[paired.group_id==g] for g in group_ids}
    for _ in range(B):
        sampled=rng.choice(group_ids,size=len(group_ids),replace=True)
        sub=pd.concat([grouped[g] for g in sampled],ignore_index=True)
        v=mean_delta(sub)
        if np.isfinite(v):boots.append(v)
    if len(boots)<max(20,B//10):return dict(status='INSUFFICIENT_BOOTSTRAPS',n_valid=len(boots),observed=observed)
    return dict(status='OK',model=model,delta_asri=observed,ci95_lower=float(np.quantile(boots,.025)),
        ci95_upper=float(np.quantile(boots,.975)),n_valid=len(boots),n_groups=len(group_ids))


def render_charts(metric_table,asri_table_df,cfg):
    import matplotlib.pyplot as plt
    out=Path(cfg.out)
    if not metric_table.empty:
        x=metric_table[metric_table.category.isin(CATEGORIES)].copy()
        for (model,regime),g in x.groupby(['model','regime']):
            pv=g.pivot(index='category',columns='level',values='balanced_accuracy').reindex(CATEGORIES)
            fig,ax=plt.subplots(figsize=(7,4));im=ax.imshow(pv.values,vmin=0,vmax=1,cmap='viridis')
            ax.set_xticks(range(len(pv.columns)),pv.columns);ax.set_yticks(range(len(pv.index)),pv.index)
            ax.set_xlabel('Severity (1–3)');ax.set_title(f'{model}: {regime} — Balanced Accuracy')
            for i in range(len(pv.index)):
                for j in range(len(pv.columns)):
                    v=pv.iloc[i,j]
                    if np.isfinite(v):ax.text(j,i,f'{v:.2f}',ha='center',va='center',color='white' if v<.5 else 'black')
            fig.colorbar(im,ax=ax,label='Balanced Accuracy');fig.tight_layout()
            fig.savefig(out/f'chart_heatmap_{safe_key(model)}_{safe_key(regime)}.png',dpi=160);plt.close(fig)
    if not asri_table_df.empty:
        z=asri_table_df.pivot(index='model',columns='regime',values='asri')
        fig,ax=plt.subplots(figsize=(8,4));z.plot(kind='bar',ax=ax,ylim=(0,1),rot=0)
        ax.set_title('ASRI: scenario-balanced robustness');ax.set_ylabel('ASRI [0,1]')
        fig.tight_layout();fig.savefig(out/'chart_ASRI.png',dpi=170);plt.close(fig)


def zip_results(cfg,include_models=False):
    from zipfile import ZipFile,ZIP_DEFLATED
    out=Path(cfg.out); destination=out/'SAADE_RESULTS.zip'
    with ZipFile(destination,'w',ZIP_DEFLATED,compresslevel=6) as z:
        for p in out.rglob('*'):
            if not p.is_file() or p==destination:continue
            if p.suffix.lower() in ('.csv','.json','.md','.txt','.png','.svg','.log') or (include_models and p.suffix=='.pt'):
                z.write(p,p.relative_to(out))
    print('Report ZIP:',destination,'bytes:',destination.stat().st_size)
    return destination

# ==========================================================
# Trainable models and evaluation adapters
# ==========================================================

def seed_everything(seed):
    import torch
    import random
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False


def _nn_components():
    import torch
    import torch.nn as nn
    return torch,nn


def create_model(name,cfg):
    """Actual named models. Lightweight baselines are NEVER presented as AASIST/RawNet2.
    Optional official AASIST/RawNet2 need downloaded source checkout.
    """
    if name=='mfcc_svm':
        return make_pipeline(StandardScaler(),SVC(C=1.0,kernel='rbf',class_weight='balanced'))
    torch,nn=_nn_components()
    if name=='spectrogram_cnn':
        class SpectrogramCNN(nn.Module):
            def __init__(self):
                super().__init__()
                self.register_buffer('window',torch.hann_window(512))
                self.encoder=nn.Sequential(
                    nn.Conv2d(1,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),
                    nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
                    nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.AdaptiveAvgPool2d((1,1)))
                self.head=nn.Sequential(nn.Flatten(),nn.Dropout(.2),nn.Linear(64,1))
            def forward(self,x):
                z=torch.stft(x,n_fft=512,hop_length=256,window=self.window,return_complex=True)
                spec=torch.log1p(z.abs().square()).unsqueeze(1)
                return self.head(self.encoder(spec)).view(-1)
        return SpectrogramCNN()
    if name=='raw_wave_cnn':
        class RawWaveCNN(nn.Module):
            def __init__(self):
                super().__init__()
                self.layers=nn.Sequential(
                    nn.Conv1d(1,16,101,stride=10,padding=50),nn.BatchNorm1d(16),nn.LeakyReLU(.1),
                    nn.Conv1d(16,32,11,stride=4,padding=5),nn.BatchNorm1d(32),nn.LeakyReLU(.1),
                    nn.Conv1d(32,64,7,stride=4,padding=3),nn.BatchNorm1d(64),nn.LeakyReLU(.1),
                    nn.Conv1d(64,128,5,stride=4,padding=2),nn.BatchNorm1d(128),nn.LeakyReLU(.1),
                    nn.AdaptiveAvgPool1d(1))
                self.head=nn.Sequential(nn.Flatten(),nn.Dropout(.25),nn.Linear(128,1))
            def forward(self,x):return self.head(self.layers(x.unsqueeze(1))).view(-1)
        return RawWaveCNN()
    if name=='aasist':
        p=Path(cfg.repo_aasist).expanduser().resolve()
        modfile=p/'models'/'AASIST.py'
        config=p/'config'/'AASIST.conf'
        if not modfile.exists() or not config.exists():
            raise FileNotFoundError('Official AASIST not installed. Clone https://github.com/clovaai/aasist into cfg.repo_aasist; see README.')
        spec=importlib.util.spec_from_file_location('_saade_official_aasist',str(modfile))
        module=importlib.util.module_from_spec(spec);sys.modules[spec.name]=module;spec.loader.exec_module(module)
        d_args=json.loads(config.read_text())['model_config']
        backbone=module.Model(d_args)
        class OfficialAASIST(nn.Module):
            def __init__(self,backbone):super().__init__();self.backbone=backbone
            def forward(self,x):
                logits=self.backbone(x)
                if isinstance(logits,tuple):logits=logits[-1]
                if logits.ndim!=2 or logits.shape[1]!=2:raise ValueError('Unexpected official AASIST output shape')
                return logits[:,1]-logits[:,0]  # trained against fake=1 in SAADE
        return OfficialAASIST(backbone)
    if name=='rawnet2':
        p=Path(cfg.repo_asvspoof5).expanduser().resolve()/'Baseline-RawNet2'/'model.py'
        if not p.exists():raise FileNotFoundError('Official RawNet2 missing. Clone https://github.com/asvspoof-challenge/asvspoof5 into cfg.repo_asvspoof5.')
        spec=importlib.util.spec_from_file_location('_saade_official_rawnet2',str(p))
        module=importlib.util.module_from_spec(spec);sys.modules[spec.name]=module;spec.loader.exec_module(module)
        device='cuda' if torch.cuda.is_available() else 'cpu'
        backbone=module.RawNet2(device)
        class OfficialRawNet2(nn.Module):
            def __init__(self,backbone):super().__init__();self.backbone=backbone
            def forward(self,x):
                logits=self.backbone(x)
                if isinstance(logits,tuple):logits=logits[-1]
                if logits.ndim!=2 or logits.shape[1]!=2:raise ValueError('Unexpected official RawNet2 output shape')
                return logits[:,1]-logits[:,0]  # trained against fake=1 in SAADE
        return OfficialRawNet2(backbone)
    if name=='wavlm':
        try:
            from transformers import WavLMModel
        except ImportError as e:
            raise ImportError('Install transformers compatible with your PyTorch version; see README.') from e
        backbone=WavLMModel.from_pretrained(cfg.wavlm_checkpoint)
        for p in backbone.parameters():p.requires_grad=False
        class FrozenWavLMHead(nn.Module):
            def __init__(self,backbone):
                super().__init__();self.backbone=backbone
                self.head=nn.Sequential(nn.LayerNorm(backbone.config.hidden_size),nn.Linear(backbone.config.hidden_size,1))
            def forward(self,x):
                self.backbone.eval()
                with torch.no_grad():rep=self.backbone(input_values=x).last_hidden_state.mean(dim=1)
                return self.head(rep).view(-1)
            def train(self,mode=True):
                super().train(mode);self.backbone.eval();return self
        return FrozenWavLMHead(backbone)
    raise KeyError(f'Unknown model name: {name}')


class AudioDataset:
    def __init__(self,records,cfg,train=False,regime='static',adaptive_weights=None):
        self.records=records.reset_index(drop=True)
        self.cfg=cfg;self.train=train;self.regime=regime;self.adaptive_weights=adaptive_weights
        self.epoch=0
    def __len__(self):return len(self.records)
    def __getitem__(self,i):
        row=self.records.iloc[i]
        seed=deterministic_seed(self.cfg.seed,self.epoch,i,row.relative_path)
        x=read_fixed_segment(row.path,self.cfg.sample_rate,self.cfg.seconds,
                start_sec=row.get('segment_start_sec',None),seed=seed,train=self.train)
        if self.train and self.regime!='static':
            rng=np.random.default_rng(seed+123)
            if rng.random()<self.cfg.augmentation_probability:
                if self.regime=='adaptive':
                    assert self.adaptive_weights is not None
                    weights=np.array([self.adaptive_weights[c] for c in CATEGORIES],float)
                    cat=str(rng.choice(CATEGORIES,p=weights/weights.sum()))
                else:cat=str(rng.choice(CATEGORIES))
                lev=int(rng.choice(available_levels(self.cfg.scenario_profile)))
                severity=scenario_severity(cat,lev,self.cfg.scenario_profile)
                x=transform(x,cat,lev,self.cfg.sample_rate,seed,severity_override=severity)
        return np.asarray(x,dtype='float32'),int(row.label)


def _batch_tensors(batch):
    import torch
    xs,ys=zip(*batch)
    return torch.from_numpy(np.stack(xs)),torch.tensor(ys,dtype=torch.float32)


def get_validation_scores(model,model_name,df,cfg):
    scores=[];labels=[]
    for r in df.itertuples():
        w=read_fixed_segment(r.path,cfg.sample_rate,cfg.seconds,
             start_sec=getattr(r,'segment_start_sec',None),seed=cfg.scenario_seed,train=False)
        scores.append(float(model_score(model,model_name,w,cfg)))
        labels.append(int(r.label))
    return np.array(labels,int),np.array(scores,float)


def train_one(df,cfg,name,regime='static',adaptive_weights=None,force=False):
    """Train all models for SAME epoch count and seed per model x regime.
    Threshold determined strictly on CLEAN validation, hidden test not read.
    """
    out=Path(cfg.out)/'models';out.mkdir(parents=True,exist_ok=True)
    modelpath=out/f'{safe_key(name)}_{safe_key(regime)}.pt'
    metapath=out/f'{safe_key(name)}_{safe_key(regime)}_meta.json'
    signature=dict(split_hash=frozen_hash(df),epochs=cfg.epochs,seed=cfg.seed,
                   seconds=cfg.seconds,sample_rate=cfg.sample_rate,lr=cfg.lr,
                   batch_size=cfg.batch_size,augmentation_probability=cfg.augmentation_probability,
                   adaptive_weights=adaptive_weights,scenario_profile=cfg.scenario_profile,
                   scenario_seed=cfg.scenario_seed,model=name)
    if not force and modelpath.exists() and metapath.exists():
        meta=json.loads(metapath.read_text())
        if meta.get('training_signature')!=signature:
            raise RuntimeError('Existing checkpoint belongs to DIFFERENT data or hyperparameters: '
                +str(modelpath)+'. Use a NEW cfg.out directory (preferred), or force=True after deliberate review.')
        print('Reusing checkpoint:',name,regime)
        if name=='mfcc_svm':
            import joblib
            m=joblib.load(modelpath)
        else:
            import torch
            m=create_model(name,cfg)
            m.load_state_dict(torch.load(modelpath,map_location='cpu',weights_only=True))
            m.to('cuda' if torch.cuda.is_available() else 'cpu').eval()
        return m,meta
    seed_everything(cfg.seed+deterministic_seed(name)%1000)
    tr=df[df.split.eq('train')].reset_index(drop=True)
    val=df[df.split.eq('validation')].reset_index(drop=True)
    if tr.label.nunique()!=2 or val.label.nunique()!=2:raise ValueError('Two classes required in training and validation.')
    model=create_model(name,cfg)
    if name=='mfcc_svm':
        dataset=AudioDataset(tr,cfg,train=True,regime=regime,adaptive_weights=adaptive_weights)
        xx=[];yy=[]
        for epoch in range(cfg.epochs):
            dataset.epoch=epoch
            for i in range(len(dataset)):
                wave,label=dataset[i];xx.append(mfcc_features(wave,cfg.sample_rate));yy.append(label)
        model.fit(np.stack(xx),yy)
        y,s=get_validation_scores(model,name,val,cfg)
        threshold=best_threshold(y,s);score=metric_row(y,s,threshold)
        import joblib
        joblib.dump(model,modelpath)
    else:
        import torch
        from torch import nn
        from torch.utils.data import DataLoader
        device='cuda' if torch.cuda.is_available() else 'cpu'
        model=model.to(device)
        ds=AudioDataset(tr,cfg,train=True,regime=regime,adaptive_weights=adaptive_weights)
        loader=DataLoader(ds,batch_size=cfg.batch_size,shuffle=True,num_workers=cfg.num_workers,
              collate_fn=_batch_tensors,drop_last=False)
        opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=cfg.lr)
        n0=int((tr.label==0).sum());n1=int((tr.label==1).sum())
        pos=torch.tensor([n0/max(1,n1)],device=device,dtype=torch.float32)
        criterion=nn.BCEWithLogitsLoss(pos_weight=pos)
        best_auc=-np.inf;best_state=None;best_metrics={};best_thr=0.0
        for epoch in range(cfg.epochs):
            ds.epoch=epoch
            model.train();losses=[]
            for batch_x,batch_y in loader:
                batch_x=batch_x.to(device);batch_y=batch_y.to(device)
                opt.zero_grad(set_to_none=True)
                output=model(batch_x).view(-1)
                loss=criterion(output,batch_y)
                if not torch.isfinite(loss):raise FloatingPointError('NaN training loss')
                loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),5.0);opt.step()
                losses.append(float(loss.detach().cpu()))
            y,s=get_validation_scores(model,name,val,cfg)
            threshold=best_threshold(y,s);scores=metric_row(y,s,threshold)
            print(f'  {name} {regime} epoch {epoch+1}/{cfg.epochs}: loss={np.mean(losses):.4f} val AUC={scores["roc_auc"]:.3f} EER={scores["eer"]:.3f}')
            if scores['roc_auc']>best_auc:
                best_auc=scores['roc_auc'];best_metrics=scores;best_thr=threshold
                best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        model.load_state_dict(best_state);model.eval()
        torch.save(best_state,modelpath)
        score=best_metrics;threshold=best_thr
    meta=dict(name=name,regime=regime,seed=cfg.seed,threshold=float(threshold),
             clean_val_metrics=score,training_records=len(tr),val_records=len(val),
             augmentation_weights=adaptive_weights,epochs=cfg.epochs,training_signature=signature)
    save_json(metapath,meta)
    return model,meta


def model_score(model,name,wave,cfg):
    if name=='mfcc_svm':
        feature=mfcc_features(wave,cfg.sample_rate).reshape(1,-1)
        score=model.decision_function(feature)
        return float(np.asarray(score).ravel()[0])
    import torch
    device=next(model.parameters()).device
    x=torch.from_numpy(np.asarray(wave,dtype='float32')).unsqueeze(0).to(device)
    model.eval()
    with torch.inference_mode():s=model(x).view(-1)
    return float(s[0].detach().cpu())


def summarize_source_results(preds,cfg):
    lines=[]
    for (model,regime,split,source),g in preds.groupby(['model','regime','split','source']):
        m={'model':model,'regime':regime,'split':split,'source':source,'n':len(g),'class_count':g.label.nunique()}
        if g.label.nunique()==2:m.update(metric_row(g.label,g.score,g.threshold.iloc[0]))
        else:
            # Single-class detection rates are valid but not balanced accuracy or EER
            m['class_recall']=float((g.pred==g.label).mean())
            m['warning']='ONLY_ONE_CLASS_IN_SOURCE; EER/BA undefined'
        lines.append(m)
    res=pd.DataFrame(lines);res.to_csv(Path(cfg.out)/'10_source_strata.csv',index=False)
    return res


def ensure_test_gate(cfg):
    if not cfg.test_finalization:
        raise RuntimeError('Hidden test is locked. Set cfg.test_finalization=True ONLY AFTER model selection and scenario policy are frozen. No exploratory hidden-test tuning.')
    p=Path(cfg.out)/'03_SPLIT_LOCK.json'
    if not p.exists():raise RuntimeError('No frozen split manifest.')


def make_readme(cfg,df,validation_df=None,test_df=None):
    out=Path(cfg.out);t=[
        '# SAADE–AUA: experiment report',
        f'Root: `{cfg.root}`',f'Seed: {cfg.seed}; sample rate: {cfg.sample_rate}; crop length: {cfg.seconds}s',
        f'Raw canonical audio catalog: {len(df)} rows',
        f'Scenario profile: {cfg.scenario_profile}; scenarios/audio: {1+4*len(available_levels(cfg.scenario_profile))}',
        '## Methodology limitations',
        '- The uploaded audit ZIP contains reports and notebooks, not the audio payload. This experiment runs against the server-side AUA directory.',
        '- Source-specific recording artefacts may be confounded with real/fake label; source-stratified results mandatory.',
        '- Synthetic recordings have generator provenance; unknown/weak speaker groups must not be interpreted as leak-free.',
        '- Own dialectal script_profile must never substitute for true speaker_origin_group.',
        '- Synthetic reverb uses exponentially decaying simulated RIR, not physical room measurements.',
        '- Compression is real MP3 round-trip via ffmpeg.',
        '- ASRI is a *proposed experimental index*, not an established standard; always report EER and ROC-AUC.',
        '- The 5-level v2 protocol is a NEW experimental configuration. Its levels are ordered and are NOT equivalent to legacy v1 level IDs.',
        '- Accuracy threshold selected only from clean validation; hidden test locked throughout model selection.',
        '- Cluster bootstrap resamples source/speaker group, not audio variants as independent observations.',
        '- Metadata approval does not establish legal redistribution permission. No recordings are bundled in result ZIP.',
        '- Official AASIST/RawNet2 are optional distinct model architectures; lightweight CNN baselines are not mislabeled as them.',
        '', '## Output files',
        '01_catalog_all.csv, 01_catalog_issues.csv, 02_experimental_subset.csv, 03_locked_splits.csv,',
        '04_val_predictions_*.csv, 05_adaptive_weights.json, 07_hidden_predictions.csv,',
        '08_scenario_metrics.csv, 09_ASRI.csv, 10_source_strata.csv, 11_bootstrap.csv, charts and SAADE_RESULTS.zip.'
    ]
    (out/'README_RESULTS.md').write_text('\n'.join(t),encoding='utf-8')
    return out/'README_RESULTS.md'


def transcript_leakage_report(df,cfg):
    # Detector only. Cross-split repeated text can be legitimate, but needs expert decision.
    d=df[df.transcription.astype(str).str.len()>=25].copy()
    d['transcript_key']=d.transcription.fillna('').map(lambda t: hashlib.sha256(' '.join(str(t).lower().split()).encode()).hexdigest())
    leak_ids=d.groupby('transcript_key').split.nunique()
    repeated=d[d.transcript_key.isin(leak_ids[leak_ids>1].index)]
    repeated.to_csv(Path(cfg.out)/'03_transcription_cross_split_REVIEW.csv',index=False)
    return repeated


def generator_holdout_report(df,cfg):
    """Feasibility for an additional leave-one-generator-out study.
    Actual training must rebuild GROUPED train/validation without the held-out generator.
    """
    fake=df[(df.label==1)].copy()
    rows=[]
    for gen,g in fake.groupby('generator'):
        if not gen or len(g)<10:continue
        other=fake[fake.generator.ne(gen)]
        rows.append(dict(heldout_generator=gen,heldout_n=len(g),heldout_groups=g.group_id.nunique(),
             remaining_fake_n=len(other),remaining_fake_generators=other.generator.nunique(),
             feasibility='CHECK' if other.generator.nunique()<2 or g.group_id.nunique()<2 else 'POSSIBLE'))
    res=pd.DataFrame(rows);res.to_csv(Path(cfg.out)/'03_generator_holdout_feasibility.csv',index=False)
    return res


def mcdm_rank(summary_df,asri_df,cfg):
    """Exploratory MCDM robustness ranking. Explicit criteria, no hallucinated significance.
    Uses mean EER/AUC and ASRI plus worst-case BA; min/max polarity declared.
    """
    from scipy.stats import rankdata
    if summary_df.empty or asri_df.empty:return pd.DataFrame()
    rows=[]
    for (model,regime),g in summary_df.groupby(['model','regime']):
        z=g[g.category.isin(CATEGORIES)]
        a=asri_df[(asri_df.model==model)&(asri_df.regime==regime)]
        if z.empty or a.empty:continue
        rows.append(dict(model=model,regime=regime,ASRI=float(a.asri.iloc[0]),
            worst_BA=float(z.balanced_accuracy.min()),mean_AUC=float(z.roc_auc.mean()),
            mean_EER=float(z.eer.mean())))
    df=pd.DataFrame(rows).dropna()
    if len(df)<2:return df
    cols=['ASRI','worst_BA','mean_AUC','mean_EER']
    benefits=df[cols].copy();benefits['mean_EER']=1-benefits['mean_EER']
    lo,hi=benefits.min(),benefits.max()
    norm=(benefits-lo)/(hi-lo).replace(0,np.nan)
    norm=norm.fillna(.5).clip(0,1)
    W=np.full(len(cols),1/len(cols));A=norm.values
    df['WSM']=(A*W).sum(axis=1)
    dpos=np.linalg.norm((A-1)*W,axis=1);dneg=np.linalg.norm(A*W,axis=1)
    df['TOPSIS']=dneg/(dneg+dpos+1e-12)
    df['DnE']=1-np.linalg.norm((A-1)*W,axis=1)/np.linalg.norm(np.ones(len(cols))*W)
    df['DnM']=1-np.sum(abs(A-1)*W,axis=1)
    ranks=np.stack([rankdata(-A[:,i],method='average') for i in range(len(cols))],axis=1)
    df['SRD']=1-np.abs(ranks-1).sum(axis=1)/(len(cols)*max(1,len(df)-1))
    df['MCDM_consensus']=df[['WSM','TOPSIS','DnE','DnM','SRD']].mean(axis=1)
    df.sort_values('MCDM_consensus',ascending=False).to_csv(Path(cfg.out)/'12_MCDM_exploratory.csv',index=False)
    return df.sort_values('MCDM_consensus',ascending=False)


def gee_cluster_effects(preds,cfg):
    """Exploratory clustered GEE logistic model; interpretable main effects,
    not a replacement for paired bootstrap or holdout validity. Requires statsmodels.
    """
    try:
        import statsmodels.api as sm
        import statsmodels.formula.api as smf
    except ImportError:
        print('statsmodels unavailable: skipping GEE');return None
    d=preds[preds.category.isin(CATEGORIES)&preds.level.isin(available_levels(cfg.scenario_profile))].copy()
    if d.empty or d.group_id.nunique()<10:return None
    d['correct']=(d.label==d.pred).astype(int)
    d['level']=d.level.astype(str)
    try:
        fit=smf.gee('correct ~ C(model) + C(regime) + C(category) + C(level)',
            groups='group_id',data=d, family=sm.families.Binomial()).fit()
        summary=fit.summary().as_text()
        (Path(cfg.out)/'13_GEE_clustered_main_effects.txt').write_text(summary,encoding='utf-8')
        return fit
    except Exception as exc:
        print('GEE did not converge / unavailable:',str(exc)[:180]);return None


def save_locked_protocol(cfg,df,models_info):
    """Freeze exactly which checkpoints and configurations will be evaluated on hidden test."""
    out=Path(cfg.out); p=out/'06_FINAL_PROTOCOL.json'
    def sha(path):
        h=hashlib.sha256()
        with open(path,'rb') as f:
            for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
        return h.hexdigest()
    models=[]
    for (name,regime),meta in models_info.items():
        cp=out/'models'/f'{safe_key(name)}_{safe_key(regime)}.pt'
        if not cp.exists():raise FileNotFoundError(cp)
        models.append(dict(model=name,regime=regime,threshold=meta['threshold'],
            checkpoint=str(cp),checkpoint_sha256=sha(cp)))
    obj=dict(dataset_hash=frozen_hash(df),config={k:v for k,v in dataclasses.asdict(cfg).items() if k!='test_finalization'},
        models=sorted(models,key=lambda m:(m['model'],m['regime'])))
    obj=json.loads(json.dumps(obj,default=str)) # canonicalize tuples/ndarrays before comparing to JSON
    if p.exists():
        old=json.loads(p.read_text())
        if old!=obj:raise RuntimeError('Frozen final protocol changed. Start new experiment directory, not re-use test.')
    else:save_json(p,obj)
    return obj


def verify_locked_protocol(cfg,df):
    p=Path(cfg.out)/'06_FINAL_PROTOCOL.json'
    if not p.exists():raise RuntimeError('Finalize training and lock final protocol FIRST.')
    obj=json.loads(p.read_text());
    if obj['dataset_hash']!=frozen_hash(df):raise RuntimeError('Dataset split changed since protocol lock')
    for rec in obj['models']:
        h=hashlib.sha256()
        with open(rec['checkpoint'],'rb') as f:
            for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
        if h.hexdigest()!=rec['checkpoint_sha256']:raise RuntimeError('Checkpoint changed since protocol freeze: '+rec['checkpoint'])
    return obj
'''
_module_name = 'saade_final_core'
_mod = types.ModuleType(_module_name)
_mod.__file__ = '<SAADE_ALL_IN_ONE>/' + _module_name + '.py'
sys.modules[_module_name] = _mod
exec(compile(_module_source, _mod.__file__, 'exec'), _mod.__dict__)
print('Підключено: saade_final_core')

In [ ]:
# Код модуля: saade_final_data.py (вбудований, інших файлів не потрібно)
_module_source = r'''
"""SAADE v3 — AUA authoritative metadata adapter and conservative parent splits.

This module reads *only* canonical data under AUA/01_audio and AUA source metadata.
It never edits the dataset. Audio provenance and human-speaker identity are distinct:
S1108/S1109 with source_group_unknown are NOT human/synthetic voice IDs.
Unknown-identity corpora are train-only in the default documented pilot protocol.
No script claims that a class label is independently expert verified.
"""
from __future__ import annotations
from pathlib import Path
from collections import defaultdict
from dataclasses import replace
import hashlib, json, re, warnings
import numpy as np
import pandas as pd
import saade_final_core as C

BAD_VALUES={'','nan','none','null','unknown','n/a','na'}

def clean(v):
    if v is None: return ''
    s=str(v).strip()
    return '' if s.lower() in BAD_VALUES else s

def get(row,*cols):
    for col in cols:
        if col in row and clean(row[col]): return clean(row[col])
    return ''

def read_csv(p):
    p=Path(p)
    if not p.is_file(): return pd.DataFrame()
    return pd.read_csv(p,dtype=str,keep_default_na=False,low_memory=False)

def key_path(s): return str(s).replace('\\','/').strip().lstrip('./')

def table_by(df,col):
    d=defaultdict(list)
    if col not in df: return d
    for _,r in df.iterrows():
        k=key_path(clean(r[col]))
        if k: d[k].append(r.to_dict())
    return d

def one_entry(mapping,key):
    rows=mapping.get(key,[])
    return rows[0] if len(rows)==1 else {}

def csv_source_candidates(root, patterns):
    """Find potential original metadata without assuming every file describes speakers."""
    out=[]
    for sub in patterns:
        folder=root/'00_sources_and_licenses'/'audio'/sub
        if folder.is_dir():
            out.extend([str(p) for p in folder.rglob('*') if p.suffix.lower() in ('.csv','.tsv','.json','.jsonl','.parquet')])
    return sorted(out)

def source_key(src_value,path):
    s=(str(src_value)+' '+str(path)).lower()
    if 'common_voice' in s or 'common voice' in s: return 'CommonVoice'
    if 'fleurs' in s: return 'FLEURS'
    if 'drssu' in s: return 'DRSSU'
    if 'new-tts' in s or 'new_tts' in s: return 'AUA_new_TTS'
    if '/own_processed/' in str(path).lower().replace('\\','/'): return 'AUA_own'
    return 'unknown_source'

def valid_speaker(speaker,basis,source):
    """Do not treat synthetic/technical group labels as verified speaker identity."""
    basis=basis.lower()
    if not speaker: return False
    if any(t in basis for t in ('unknown','source_group','fallback','synthetic_collection','batch')):return False
    if source in ('DRSSU','FLEURS') and not basis:return False
    if source=='CommonVoice': return 'client_id' in basis
    if source=='AUA_own': return True
    return bool('speaker' in basis or 'identity' in basis or 'verified' in basis)

def build_verified_catalog(cfg, require_source_metadata=True):
    root=Path(cfg.root).expanduser().resolve();aud=root/'01_audio';out=Path(cfg.out)
    if out.resolve()==root or root in out.resolve().parents:
        raise ValueError('Output folder must not be placed inside AUA')
    if any('melnyk' in part.casefold() for part in out.resolve().parts):
        raise ValueError('Do not write to Melnyk workspace')
    out.mkdir(parents=True,exist_ok=True)
    if not aud.is_dir():raise FileNotFoundError(f'Cannot find {aud}')
    general=read_csv(aud/'metadata_audio.csv')
    own=read_csv(aud/'own_processed'/'metadata_audio.csv')
    tts=read_csv(aud/'synthetic'/'new-tts'/'metadata_new-tts.csv')
    drssu=read_csv(root/'00_sources_and_licenses'/'audio'/'drssu'/'metadata.csv')
    # Direct metadata matches first; basename match only when globally unambiguous.
    by_work=table_by(general,'working_path');by_name=table_by(general,'file_name')
    own_by_name=table_by(own,'file_name');own_by_work=table_by(own,'working_path')
    tts_by_filename=table_by(tts,'filename')
    # The AUA field source_record_id is `drssu:<selection_rank>`; the
    # source CSV's `file_name` instead contains `generated/4308.wav`, etc.
    # Join on selection_rank and verify source SHA, never on the folder name.
    drssu_rank=table_by(drssu,'selection_rank')
    # Report potential source speaker metadata for manual linkage without inventing IDs.
    candidates=csv_source_candidates(root,['fleurs','common_voice','drssu'])
    C.save_json(out/'00_available_metadata_sources.json',dict(root=str(root),
        general_rows=len(general),own_rows=len(own),tts_rows=len(tts),drssu_rows=len(drssu),candidate_files=candidates[:300]))
    allowed=[aud/'own_processed',aud/'external_selected',aud/'synthetic']
    files=sorted({p for d in allowed if d.is_dir() for p in d.rglob('*')
                  if p.is_file() and p.suffix.lower() in C.AUDIO_EXT})
    rows=[]
    for p in files:
        rp=p.relative_to(root).as_posix()
        name=p.name
        g=one_entry(by_work,key_path(rp)) or one_entry(by_name,name)
        o=one_entry(own_by_work,key_path(rp)) or one_entry(own_by_name,name)
        t=one_entry(tts_by_filename,name)
        source=source_key(get(g,'source_dataset'),rp)
        source_record=get(o,'source_record_id') or get(g,'source_record_id')
        src_sha=get(o,'source_sha256') or get(g,'source_sha256')
        working_sha=get(o,'working_sha256') or get(g,'working_sha256')
        sp=get(o,'speaker_id') or get(g,'speaker_id')
        sp_basis=get(g,'speaker_id_basis') or get(o,'speaker_id_basis')
        mode=get(g,'mode').lower()
        note=[]
        dr={}
        if source=='DRSSU':
            # AUA metadata uses drssu:selection_rank, NOT a verified voice ID.
            rank=source_record.split(':')[-1] if source_record else ''
            dr=one_entry(drssu_rank,rank)
            # Missing rank is never silently inferred from a file name.
            if not dr:note.append('DRSSU_SOURCE_ROW_NOT_LINKED')
        label=None;label_basis='NOT_VERIFIED'
        if source=='DRSSU' and dr:
            flag=get(dr,'generated').lower()
            # Class comes from the source register, NOT from its filename.
            if flag in ('true','1','yes'):
                label=1;label_basis='DRSSU_source_metadata_generated_true'
            elif flag in ('false','0','no'):
                label=0;label_basis='DRSSU_source_metadata_generated_false'
            else:note.append('DRSSU_GENERATED_FLAG_INVALID')
            # A missing or conflicting source SHA must not be treated as
            # independently corroborated provenance. Refuse the class.
            original_sha=get(dr,'sha256')
            if not original_sha or not src_sha:
                note.append('DRSSU_SOURCE_SHA_MISSING')
                label=None;label_basis='NOT_VERIFIED'
            elif original_sha.lower()!=src_sha.lower():
                note.append('DRSSU_SOURCE_SHA_CONFLICT')
                label=None;label_basis='NOT_VERIFIED'
        elif source=='AUA_new_TTS' and t:
            # A generator-specific register establishes that this is TTS even
            # when voice_id is unavailable. Unknown voice remains TRAIN ONLY.
            if get(t,'model_name'):
                label=1
                label_basis=('TTS_source_generation_register' if get(t,'voice_id')
                             else 'TTS_source_generation_register_voice_unknown')
                if not get(t,'voice_id'):
                    note.append('TTS_VOICE_ID_UNKNOWN_TRAIN_ONLY')
            else:
                note.append('TTS_MODEL_NAME_MISSING')
        elif source in ('CommonVoice','FLEURS') and g:
            label=0;label_basis=f'{source}_dataset_origin_real_assumed'
        elif source=='AUA_own' and o:
            if get(o,'source_path') or get(o,'source_record_id'):
                label=0;label_basis='AUA_own_original_record_mapping'
            else:note.append('OWN_ORIGINAL_MAPPING_MISSING')
        else:
            note.append('SOURCE_METADATA_MISSING')
        if source=='unknown_source':note.append('UNKNOWN_SOURCE')
        # Explicit mode only when it really denotes class, not READC etc.
        class_mode={'real':0,'human':0,'bona_fide':0,'genuine':0,
                    'fake':1,'synthetic':1,'spoof':1,'tts':1,'vc':1,'generated':1}.get(mode)
        if class_mode is not None and label is not None and class_mode!=label:
            note.append('EXPLICIT_MODE_CLASS_CONFLICT')
            label=None
        known_speaker=valid_speaker(sp,sp_basis,source)
        voice=get(t,'voice_id');generator=get(t,'model_name')
        input_text_id=get(t,'input_text_id')
        voice_confirmed=bool(source=='AUA_new_TTS' and voice and generator)
        if source=='DRSSU':generator='UNKNOWN_DRSSU_GENERATOR'
        elif label==1 and not generator:generator='unknown_generator'
        if known_speaker:
            group=f'SPK::{source}::{sp}'
            confidence='metadata_speaker_verified_basis' if source=='CommonVoice' else 'metadata_speaker'
            identity='speaker_confirmed'
            split_role='eligible'
        elif voice_confirmed:
            group=f'VOICE::{source}::{generator}::{voice}'
            confidence='metadata_voice'
            identity='synthetic_voice_confirmed'
            split_role='eligible'
        else:
            # Conservative source-wise grouping, never pretend each DRSSU file is a new person.
            # Unknown voices cannot enter validation/test under exploratory protocol.
            group=f'UNKNOWN_IDENTITY::{source}'
            confidence='source_identity_unknown'
            identity='unknown_identity'
            split_role='train_only'
            note.append('IDENTITY_UNKNOWN_TRAIN_ONLY')
        if source=='DRSSU' and not dr and require_source_metadata:label=None
        duration=C.audio_info_seconds(p)
        usable=(label in (0,1) and np.isfinite(duration) and duration>=float(cfg.min_seconds)
                and not any('CONFLICT' in s for s in note))
        if not usable: note.append('UNUSABLE_OR_UNVERIFIED')
        rows.append(dict(path=str(p),relative_path=rp,source=source,source_dataset=get(g,'source_dataset') or source,
             label=label,label_source=label_basis,label_conflict=any('CONFLICT' in s for s in note),
             speaker_id=sp if known_speaker else '',technical_speaker_id=sp,
             speaker_id_basis=sp_basis,source_record_id=source_record,source_sha256=src_sha,
             working_sha256=working_sha,group_id=group,group_confidence=confidence,
             identity_status=identity,split_role=split_role,voice_id=voice,generator=generator,
             input_text_id=input_text_id,transcription=get(o,'transcription') or get(g,'transcription'),
             script_id=get(o,'script_id'),speaker_origin_group=get(o,'speaker_origin_group'),
             script_profile=get(o,'script_profile'),dialect_authenticity=get(o,'dialect_authenticity'),
             approval_status=get(o,'approval_status') or get(g,'approval_status') or get(t,'approval_status'),
             license=get(g,'license') or get(t,'license_terms'),duration_sec=duration,
             usable=bool(usable),warning='; '.join(note),primary_meta_found=bool(g),
             primary_source_meta_found=bool(dr) if source=='DRSSU' else bool(t) if source=='AUA_new_TTS' else bool(g)))
    x=pd.DataFrame(rows)
    if x.empty:raise ValueError('No audio found in canonical audio folders')
    # Dedup by exact working SHA, and by original source SHA within DRSSU.
    # AUA contains 10 repeat DRSSU source records with distinct working WAV
    # hashes; they must not be counted as 10 independent source recordings.
    nonempty=x.working_sha256.fillna('').ne('')
    dupl=nonempty & x.working_sha256.duplicated(keep='first')
    x.loc[dupl,'usable']=False
    x.loc[dupl,'warning']=x.loc[dupl,'warning'].astype(str)+'; DUPLICATE_WORKING_SHA'
    drmask=(x.source.eq('DRSSU') & x.source_sha256.fillna('').astype(str).str.strip().ne(''))
    dr_dup=drmask & x.loc[:, 'source_sha256'].where(drmask).duplicated(keep='first')
    x.loc[dr_dup, 'usable']=False
    x.loc[dr_dup, 'warning']=x.loc[dr_dup,'warning'].astype(str)+'; DRSSU_DUPLICATE_SOURCE_SHA'
    # A SHA conflict is not proof of fake/real identity; keep it out of experiments.
    x.to_csv(out/'01_catalog_all.csv',index=False)
    x[~x.usable | x.warning.ne('')].to_csv(out/'01_catalog_issues.csv',index=False)
    summary=x.groupby(['source','label','label_source','identity_status','split_role'],dropna=False).agg(
        files=('path','size'),hours=('duration_sec',lambda s:pd.to_numeric(s,errors='coerce').sum()/3600),
        groups=('group_id','nunique'),usable=('usable','sum')).reset_index()
    summary.to_csv(out/'01_sources_summary.csv',index=False)
    crosstab=pd.crosstab(x.source,x.label,dropna=False)
    crosstab.to_csv(out/'01_source_class_table.csv')
    C.save_json(out/'01_data_diagnostics.json',dict(total=len(x),usable=int(x.usable.sum()),
         class_labels_according_to_provenance=int(x.label.notna().sum()),
         metadata_unlinked=int((~x.primary_meta_found).sum()),
         unknown_identity_files=int(x.identity_status.eq('unknown_identity').sum()),
         label_source_counts=x.label_source.value_counts(dropna=False).to_dict(),
         warning='Dataset origin real/fake is not expert audio validation; source-vs-class confounding may remain.'))
    dr=x[x.source.eq('DRSSU')]
    provenance=dict(
        DRSSU_total=int(len(dr)),
        DRSSU_linked=int(dr.primary_source_meta_found.sum()) if len(dr) else 0,
        DRSSU_labelled_fake=int(dr.label.eq(1).sum()) if len(dr) else 0,
        DRSSU_source_sha_conflicts=int(dr.warning.fillna('').str.contains('DRSSU_SOURCE_SHA_CONFLICT').sum()) if len(dr) else 0,
        DRSSU_duplicate_source_sha=int(dr.warning.fillna('').str.contains('DRSSU_DUPLICATE_SOURCE_SHA').sum()) if len(dr) else 0,
        DRSSU_split_role_train_only=int(dr.split_role.eq('train_only').sum()) if len(dr) else 0,
        warning='Known source sha duplicates and technical speaker identities are not independent data.'
    )
    C.save_json(out/'01_DRSSU_PROVENANCE.json',provenance)
    print('DRSSU provenance:',provenance)
    print('AUTHORITATIVE CATALOG:',len(x),'usable',x.usable.sum())
    print(summary.to_string(index=False,max_rows=20))
    return x


def prepare_parent_splits(catalog,cfg,max_parent_files=850,unknown_train_cap=150,policy='exploratory'):
    """Create group-safe split on VERIFIED speaker/voice groups; append unknown-identity TRAIN ONLY.

    A DRSSU `source_group_unknown` is NOT a valid independent synthetic voice ID.
    Paired source/class prerequisite for strict publication mode.
    """
    out=Path(cfg.out);out.mkdir(parents=True,exist_ok=True)
    v=catalog[catalog.usable & catalog.label.isin([0,1])].copy()
    v['label']=v.label.astype(int)
    eligible=v[v.split_role.eq('eligible')].copy()
    trainonly=v[v.split_role.eq('train_only')].copy()
    if eligible.empty or eligible.label.nunique()!=2:
        raise ValueError('Not enough metadata-confirmed REAL/FAKE groups for validation/test. Check TTS voice register.')
    src=pd.crosstab(v.source,v.label)
    for val in (0,1):
        if val not in src:src[val]=0
    matched=src[(src[0]>0)&(src[1]>0)].index.tolist()
    if policy=='strict_source' and not matched:
        raise ValueError('STRICT SOURCE MATCH FAILED: no confirmed source contains both REAL and FAKE. '
                         'Provide comparable real/fake source data; do not present exploratory as final.')
    # Strict mode uses matched sources only, including identity-known subset.
    if policy=='strict_source':
        eligible=eligible[eligible.source.isin(matched)].copy()
        trainonly=trainonly.iloc[:0].copy()
    cap=int(max_parent_files)
    if cap>0:
        # Set cap for split-eligible and train-only separately so unverified voices never dominate validation.
        elig_cap=max(50,cap-int(unknown_train_cap))
        local=replace(cfg,max_items=elig_cap)
        eligible=C.get_experimental_subset(eligible,local)
    # min groups for val/test from each label
    cnt=eligible.groupby('label').group_id.nunique()
    if len(cnt)!=2 or cnt.min()<3:
        raise ValueError('Need at least 3 independently identified speaker/voice groups PER CLASS')
    split=C.make_splits(eligible,cfg)
    if not trainonly.empty and unknown_train_cap>0:
        # Select per-source examples without granting speaker independence.
        def order_key(z):
            return z.sort_values(['source','source_record_id','relative_path'])
        trainonly=order_key(trainonly)
        chunks=[]
        for _,a in trainonly.groupby('source',sort=True):
            chunks.append(a.head(max(1,int(unknown_train_cap)//max(1,trainonly.source.nunique()))))
        trainonly=pd.concat(chunks,ignore_index=True)
        trainonly['split']='train'
        split=pd.concat([split,trainonly],ignore_index=True)
    # Guard source and working hashes, speaker/voice group IDs
    for col in ('group_id','source_sha256','working_sha256','path'):
        if col not in split:continue
        z=split.copy()
        z=z[z[col].fillna('').astype(str).str.strip().ne('')]
        leaks=z.groupby(col).split.nunique()
        if leaks.gt(1).any():
            raise AssertionError(f'Cross-split leakage for {col}: {int(leaks.gt(1).sum())} groups')
    split['parent_file_id']=split.relative_path
    split.to_csv(out/'03_parent_splits.csv',index=False)
    # Cross-split TTS prompt reuse: diagnostic only; identical input text is NOT a verified voice leak.
    if 'input_text_id' in split.columns:
        tx=split[split.input_text_id.fillna('').astype(str).str.strip().ne('')]
        repeat=tx.groupby('input_text_id').agg(
            splits=('split',lambda a:' | '.join(sorted(set(a)))),
            n_splits=('split','nunique'),files=('path','size'),voices=('voice_id','nunique')).reset_index()
        repeat.to_csv(out/'03_TTS_TEXT_OVERLAP.csv',index=False)
        C.save_json(out/'03_TTS_TEXT_OVERLAP_SUMMARY.json',dict(
            cross_split_text_ids=int((repeat.n_splits>1).sum()),
            note='Cross-split TTS input texts can influence text-generalization. They are NOT automatically speaker leaks.'))
    report=(split.groupby(['split','source','label','identity_status'],dropna=False)
                .agg(parent_files=('path','size'),groups=('group_id','nunique'))
                .reset_index())
    report.to_csv(out/'03_parent_split_summary.csv',index=False)
    C.save_json(out/'03_split_provenance.json',dict(policy=policy,matched_sources=matched,
         unknown_voice_files_train_only=int(split.split_role.eq('train_only').sum()),
         train_only_unknown_identity=True,
         warning='Group-safe split conditional on metadata. Does NOT resolve source/class confounding or hidden identity repetition.'))
    return split,report


def data_readiness_report(catalog, parent_splits, cfg, policy):
    out=Path(cfg.out)
    both=(pd.crosstab(catalog[catalog.usable].source,catalog[catalog.usable].label)
          .reindex(columns=[0,1],fill_value=0))
    pairs=both[(both[0]>0)&(both[1]>0)]
    check=[
        ('provenance_source_links',int((catalog.primary_meta_found).sum()),len(catalog),'CHECK' if (~catalog.primary_meta_found).any() else 'PASS'),
        ('label_unverified',int(catalog.label.isna().sum()),0,'PASS' if catalog.label.notna().all() else 'CHECK'),
        ('unknown_identity_in_validation_test',int(((parent_splits.split!='train') & parent_splits.split_role.eq('train_only')).sum()),0, 'PASS' if not ((parent_splits.split!='train') & parent_splits.split_role.eq('train_only')).any() else 'FAIL'),
        ('matched_source_both_classes',len(pairs),1,'PASS' if len(pairs)>0 else 'FAIL'),
        ('source_group_unknown_ID_reused_as_speaker',int((parent_splits.identity_status.eq('unknown_identity') & parent_splits.group_confidence.str.contains('metadata_speaker')).sum()),0,'PASS'),
        ('source_class_confounding_resolved',len(pairs),1,'PASS' if len(pairs)>0 else 'FAIL')]
    data=pd.DataFrame(check,columns=['item','actual','required','status'])
    data.to_csv(out/'DATA_READINESS.csv',index=False)
    C.save_json(out/'DATA_READINESS.json',dict(strict_ready=bool(len(pairs)>0 and policy=='strict_source'),
        purpose='SAADE diagnostics and pilot ML; not expert evaluation',
        publication_warning='Matched source is necessary, not sufficient; capture-channel / speaker / text / generator confounds still require checks.'))
    return data
'''
_module_name = 'saade_final_data'
_mod = types.ModuleType(_module_name)
_mod.__file__ = '<SAADE_ALL_IN_ONE>/' + _module_name + '.py'
sys.modules[_module_name] = _mod
exec(compile(_module_source, _mod.__file__, 'exec'), _mod.__dict__)
print('Підключено: saade_final_data')

In [ ]:
# Код модуля: saade_final_methods.py (вбудований, інших файлів не потрібно)
_module_source = r'''
"""SAADE v2 extensions: source-confounding diagnostics, safe segment expansion,
seeds, scenario-specific statistics and a one-shot hidden-test gate.

This module does NOT modify the user's original AUA media.
"""
from __future__ import annotations
import dataclasses
import hashlib
import json
import math
import os
import re
import time
import warnings
from collections import defaultdict
from pathlib import Path
from dataclasses import replace
import numpy as np
import pandas as pd
import saade_final_core as S


def source_balance_audit(catalog, cfg, tag='all'):
    """Assess provenance/label confounding WITHOUT claiming true matching.

    A source must contain both authentic and synthetic examples; this is a
    necessary but not sufficient condition for matched capture conditions.
    """
    x=catalog[catalog.label.isin([0,1])].copy()
    if x.empty:raise ValueError('No labeled recordings')
    x['label']=x.label.astype(int)
    src=x.groupby(['source','label']).agg(files=('path','size'),
                    independent_groups=('group_id','nunique'),
                    minutes=('duration_sec',lambda a:float(pd.to_numeric(a,errors='coerce').sum()/60))).reset_index()
    wide=src.pivot(index='source',columns='label',values='files').fillna(0)
    for lab in (0,1):
        if lab not in wide.columns: wide[lab]=0
    wide=wide.rename(columns={0:'real_count',1:'fake_count'}).reset_index()
    wide['both_labels']=wide.real_count.gt(0)&wide.fake_count.gt(0)
    out=Path(cfg.out)/'v2_quality';out.mkdir(parents=True,exist_ok=True)
    src.to_csv(out/f'{tag}_source_label_details.csv',index=False)
    wide.to_csv(out/f'{tag}_source_class_balance.csv',index=False)
    x['label_name']=x.label.map({0:'REAL',1:'FAKE'})
    matches=[]
    if 'transcription' in x.columns:
        x['transcript_key']=x.transcription.fillna('').astype(str).str.lower().str.replace(r'\W+','',regex=True)
        tx=x[x.transcript_key.str.len()>15]
        g=tx.groupby('transcript_key').agg(labels=('label','nunique'),paths=('relative_path','size')).reset_index()
        mixed=set(g[g.labels.eq(2)].transcript_key)
        if mixed:
            for _,r in tx[tx.transcript_key.isin(mixed)].iterrows():
                matches.append(dict(transcript_group=r.transcript_key[:55],label=r.label,
                                    source=r.source,relative_path=r.relative_path,group_id=r.group_id))
    matched=pd.DataFrame(matches,columns=['transcript_group','label','source','relative_path','group_id'])
    matched.to_csv(out/f'{tag}_same_text_candidates.csv',index=False)
    warnings_list=[]
    if not wide.both_labels.any():
        warnings_list.append('NO_SOURCE_WITH_BOTH_CLASSES: cross-source confounding unresolved')
    if matched.empty:
        warnings_list.append('NO_MATCHED_TEXT_PAIRS: matched utterance content not confirmed')
    if x.group_confidence.astype(str).eq('weak_file_fallback').any():
        warnings_list.append('UNVERIFIABLE_GROUPS: cannot claim speaker/generator independence')
    report=dict(total_files=len(x),matched_sources=wide[wide.both_labels].source.tolist(),
                matched_text_groups=int(matched.transcript_group.nunique()),warnings=warnings_list,
                source_purity=float((wide[['real_count','fake_count']].min(axis=1)==0).mean()))
    S.save_json(out/f'{tag}_confounding_report.json',report)
    return wide,matched,report


def check_and_prepare_parent_split(catalog,cfg,max_items=400,policy='exploratory'):
    """Split *parents*, not audio windows. Strict mode fails when source/class confounded.
    Mutually exclusive by metadata group_id; no weak IDs are silently accepted.
    """
    if policy not in ('strict_source','exploratory'):raise ValueError('Policy must be strict_source or exploratory')
    df=catalog[catalog.usable & catalog.label.isin([0,1])].copy()
    df['label']=df.label.astype(int)
    df=df[~df.group_confidence.eq('weak_file_fallback')].copy()
    wide,_,report=source_balance_audit(df,cfg,tag='candidate')
    if policy=='strict_source':
        good=set(wide[wide.both_labels].source)
        if not good:
            raise ValueError('STRICT_SOURCE blocked: no source has both REAL and FAKE. Supply matched audio + verified metadata.')
        df=df[df.source.isin(good)].copy()
        if df.label.nunique()<2: raise ValueError('Strict paired sources do not provide both labels')
    else:
        if not wide.both_labels.any():
            warnings.warn('EXPLORATORY ONLY: source predicts class. Cannot claim source-invariant deepfake detection.')
    max_items=max_items or cfg.max_items
    local=replace(cfg,max_items=max_items)
    subset=S.get_experimental_subset(df,local)
    parent_splits=S.make_splits(subset,cfg)
    if 'parent_file_id' not in parent_splits:parent_splits['parent_file_id']=parent_splits.relative_path
    out=Path(cfg.out)/'v2_quality';out.mkdir(exist_ok=True,parents=True)
    parent_splits.to_csv(out/'03_PARENT_SPLITS.csv',index=False)
    return parent_splits,report


def expand_nonoverlapping_segments(parent_splits,cfg,target=1200,max_per_parent=30,min_segments_per_file=1):
    """Expand fixed non-overlapping windows AFTER parent group/split assignment.

    The parent/source group_id and original split remain unchanged. This DOES NOT
    invent additional independent speakers. Target is a cap, never a promise.
    """
    if target<1:raise ValueError('target must be positive')
    required={'path','relative_path','group_id','label','duration_sec','split'}
    if missing:=required-set(parent_splits.columns):raise ValueError('Missing: '+str(sorted(missing)))
    rows=[];step=float(cfg.seconds)
    for r in parent_splits.itertuples():
        duration=float(r.duration_sec)
        n=int(duration//step)
        if n<=0:
            if duration>=float(cfg.min_seconds):
                starts=[0.]
            else:continue
        else:
            n=min(n,max_per_parent)
            # distinct, non-overlapping starts, but cover the full recording when capped
            available=int(duration//step)
            indices=np.linspace(0,available-1,n,dtype=int)
            indices=np.unique(indices)
            starts=(indices*step).tolist()
        for seg_index,t0 in enumerate(starts):
            d=r._asdict().copy()
            parent=d.get('parent_file_id') or str(r.relative_path)
            d['parent_file_id']=parent
            d['media_relative_path']=str(r.relative_path)
            d['segment_start_sec']=round(float(t0),6)
            d['segment_index']=int(seg_index)
            d['relative_path']=str(r.relative_path)+f'::segment_{seg_index:03d}_t{t0:.2f}'
            d['duration_sec']=min(step,duration-t0)
            rows.append(d)
    allseg=pd.DataFrame(rows)
    if allseg.empty:raise ValueError('No segments can be extracted')
    # deterministic, label-stratified cap; retain all unique groups in already-assigned splits
    target=int(target)
    if len(allseg)>target:
        rng=np.random.default_rng(cfg.seed)
        chosen=[]
        for label in (0,1):
            x=allseg[allseg.label.eq(label)].copy()
            quota=min(len(x),target//2)
            # round-robin by original parent (prevents long recording from dominating)
            x=x.iloc[rng.permutation(len(x))].copy()
            x['rank']=x.groupby('parent_file_id').cumcount()
            x=x.sort_values(['rank','parent_file_id','segment_index'])
            chosen.append(x.head(quota).drop(columns='rank'))
        allseg=pd.concat(chosen,ignore_index=True)
    # grouped split and parent tests
    gcounts=allseg.groupby('group_id').split.nunique()
    if gcounts.gt(1).any():raise AssertionError('GROUP LEAKAGE after segment expansion')
    pc=allseg.groupby('parent_file_id').split.nunique()
    if pc.gt(1).any():raise AssertionError('PARENT LEAKAGE after segment expansion')
    # same master file & label may exist in different splits when group_id was different: catch
    fp=allseg.groupby('path').split.nunique()
    if fp.gt(1).any():raise AssertionError('SAME SOURCE FILE split across partitions')
    out=Path(cfg.out)/'v2_quality';out.mkdir(parents=True,exist_ok=True)
    allseg.to_csv(out/'04_SEGMENTS_GROUP_SAFE.csv',index=False)
    # Canonical locked splits for viewer, prediction code, and frozen protocol.
    allseg.to_csv(Path(cfg.out)/'03_locked_splits.csv',index=False)
    counts=allseg.groupby(['split','label']).agg(segments=('path','size'),
                   independent_groups=('group_id','nunique'),
                   parent_files=('parent_file_id','nunique'),
                   duration_min=('duration_sec',lambda a:float(a.sum()/60))).reset_index()
    counts.to_csv(out/'04_SEGMENT_COUNTS.csv',index=False)
    report={'target':target,'actual':len(allseg),'independent_groups':int(allseg.group_id.nunique()),
            'parent_files':int(allseg.parent_file_id.nunique()),'status':'PASS' if len(allseg)>=800 else 'CHECK_INSUFFICIENT',
            'warning':'Number of segments is NOT number of independent speech sources'}
    S.save_json(out/'04_SEGMENT_REPORT.json',report)
    return allseg,counts,report


def generator_independence(splits,cfg,tag='validation'):
    d=splits.copy()
    d['gen_id']=d.generator.fillna('').astype(str).str.strip().replace({'':'unknown'})
    fake=d[d.label.eq(1)].copy()
    cross=fake.groupby(['gen_id','split']).agg(segments=('path','size'),voices=('group_id','nunique')).reset_index()
    out=Path(cfg.out)/'v2_quality';out.mkdir(parents=True,exist_ok=True)
    cross.to_csv(out/'05_GENERATOR_SPLIT.csv',index=False)
    leak=cross.groupby('gen_id').split.nunique()
    known=leak[~leak.index.str.contains('unknown',case=False)]
    report={'known_generator_count':len(known),'generators_in_multiple_splits':known[known.gt(1)].index.tolist(),
            'true_generator_holdout':bool(len(known)>=3 and known.max()==1),
            'warning':'If generator appears in train and test, claims of unseen-generator generalization are not justified'}
    S.save_json(out/'05_GENERATOR_INDEPENDENCE.json',report)
    return cross,report


def seed_replicates(base_cfg,split_df,seeds=(42,123,777),models=None,force=False):
    """Run E1/E2/E3 3–5 seeds on the SAME frozen parent split and scenario transform seed.

    Use separate subdirectories to avoid overwriting checkpoints. VERY EXPENSIVE on GPU.
    """
    if len(seeds)<3 or len(seeds)>5:raise ValueError('Publication protocol requires 3–5 seeds')
    names=tuple(models or base_cfg.models)
    parent=Path(base_cfg.out);shared_cache=parent
    all_pred=[];all_asri=[];protocols=[]
    for seed in seeds:
        cfg=replace(base_cfg,seed=int(seed),out=str(parent/'replicates'/f'seed_{seed}'),
                    scenario_cache_dir=str(shared_cache),test_finalization=False)
        Path(cfg.out).mkdir(parents=True,exist_ok=True)
        # Frozen dataset hash depends only on splits, never on random seed
        S.lock_hidden_test(split_df,cfg)
        base=[];models_store={};modelinfo={}
        for name in names:
            model,meta=S.train_one(split_df,cfg,name,'static',force=force)
            models_store[(name,'static')]=model;modelinfo[(name,'static')]=meta
            p=S.run_scenario_predictions(split_df,model,name,'static',cfg,threshold=meta['threshold'],split='validation')
            base.append(p)
        base=pd.concat(base,ignore_index=True)
        weights_by_name={name:S.validation_difficulty(base[base.model.eq(name)]) for name in names}
        weights={cat:float(np.mean([weights_by_name[n][cat] for n in names])) for cat in S.CATEGORIES}
        z=sum(weights.values());weights={k:v/z for k,v in weights.items()}
        S.save_json(Path(cfg.out)/'05_adaptive_weights.json',weights)
        full=[base]
        for name in names:
            for regime in ('random','adaptive'):
                model,meta=S.train_one(split_df,cfg,name,regime,
                      adaptive_weights=weights if regime=='adaptive' else None,force=force)
                models_store[(name,regime)]=model;modelinfo[(name,regime)]=meta
                pred=S.run_scenario_predictions(split_df,model,name,regime,cfg,threshold=meta['threshold'],split='validation')
                full.append(pred)
        preds=pd.concat(full,ignore_index=True)
        preds['training_seed']=seed
        preds.to_csv(Path(cfg.out)/'04_val_predictions_ALL.csv',index=False)
        _,asri=S.summarize_predictions(preds,cfg,report_tag='validation')
        asri['training_seed']=seed
        all_pred.append(preds);all_asri.append(asri)
        protoc=S.save_locked_protocol(cfg,split_df,modelinfo)
        protocols.append({'seed':seed,'output':str(cfg.out),'dataset_hash':protoc['dataset_hash']})
        del models_store
    table=pd.concat(all_asri,ignore_index=True)
    table.to_csv(parent/'V2_REPLICATE_ASRI.csv',index=False)
    detail=pd.concat(all_pred,ignore_index=True)
    detail.to_csv(parent/'V2_REPLICATE_VALIDATION_PREDICTIONS.csv',index=False)
    stats=table.groupby(['model','regime']).asri.agg(['count','mean','std','min','max']).reset_index()
    stats.to_csv(parent/'V2_REPLICATE_SUMMARY.csv',index=False)
    S.save_json(parent/'V2_REPLICATE_PROTOCOLS.json',protocols)
    return table,stats,detail


def interaction_statistics(predictions,cfg,tag='validation'):
    """Fit GEE with explicit model × regime and category × severity interactions.
    Returns diagnostic text; not an automatic claim of statistical significance.
    """
    import statsmodels.formula.api as smf
    import statsmodels.api as sm
    d=predictions.copy()
    d=d[d.category.isin(S.CATEGORIES)].copy()
    if d.empty:raise ValueError('No acoustic scenarios')
    d['correct']=(d.pred.astype(int)==d.label.astype(int)).astype(int)
    d['level']=d.level.astype(str)
    d['cluster_id']=d.group_id.astype(str)
    out=Path(cfg.out)/'v2_quality';out.mkdir(parents=True,exist_ok=True)
    formula='correct ~ C(model)*C(regime) + C(category)*C(level)'
    result={'formula':formula,'clusters':int(d.cluster_id.nunique()),'observations':len(d)}
    if d.cluster_id.nunique()<30:
        result['status']='CHECK_TOO_FEW_CLUSTERS'
        S.save_json(out/f'{tag}_interaction_status.json',result)
        return result
    try:
        fit=smf.gee(formula,groups='cluster_id',data=d,family=sm.families.Binomial()).fit(maxiter=100)
        summary=fit.summary().as_text()
        (out/f'{tag}_GEE_interactions.txt').write_text(summary,encoding='utf-8')
        result['status']='FIT';result['converged']=bool(fit.converged)
        result['model_regime_terms']=[n for n in fit.params.index if 'C(model)' in n and 'C(regime)' in n]
        result['warning']='Check sparse cells, separation, and cluster dependence; inspect full summary and CIs.'
    except Exception as exc:
        result['status']='FAILED';result['error']=str(exc)[:400]
    S.save_json(out/f'{tag}_interaction_status.json',result)
    return result


def reserve_hidden_evaluation(cfg,split_df,expected_seed_protocols=None):
    """Strong fail-closed one-shot marker. If interrupted, researcher must document recovery.
    Important: it cannot prevent a human manually modifying the output directory.
    """
    S.verify_locked_protocol(cfg,split_df)
    lock=S.frozen_hash(split_df)
    folder=Path(cfg.out);folder.mkdir(parents=True,exist_ok=True)
    marker=folder/'HIDDEN_TEST_ONE_SHOT_STARTED.json'
    rec={'dataset_sha256':lock,'started_utc':pd.Timestamp.utcnow().isoformat(),
         'protocol':'SAADE v2','note':'Do not rerun hidden test or change hyperparameters after reading results.'}
    try:
        with marker.open('x',encoding='utf8') as f:
            json.dump(rec,f,ensure_ascii=False,indent=2)
    except FileExistsError:
        raise RuntimeError('Hidden-test evaluation has already been started. Do NOT rerun. See HIDDEN_TEST_ONE_SHOT_STARTED.json')
    return marker


def final_technical_summary(cfg,splits,preds=None,source_report=None,segment_report=None):
    out=Path(cfg.out)/'v2_quality';out.mkdir(parents=True,exist_ok=True)
    rows=[]
    def add(key,target,actual,status,details=''):
        rows.append(dict(check=key,target=target,actual=actual,status=status,details=details))
    c=splits.groupby(['split','label']).size().to_dict()
    add('Class-complete three splits','both labels in train/validation/hidden',str(c),
        'PASS' if all(c.get((x,y),0)>0 for x in ['train','validation','hidden_test'] for y in (0,1)) else 'FAIL')
    add('Group leakage','0',sum((splits.groupby('group_id').split.nunique()>1).values),
        'PASS' if not (splits.groupby('group_id').split.nunique()>1).any() else 'FAIL')
    add('Original-file leakage','0',sum((splits.groupby('path').split.nunique()>1).values),
        'PASS' if not (splits.groupby('path').split.nunique()>1).any() else 'FAIL')
    add('800–1500 segments','800–1500',len(splits),'PASS' if 800<=len(splits)<=1500 else 'CHECK',
        'Segment count is not independent sample count')
    if source_report:
        add('Matched source classes','≥1 source both classes',len(source_report.get('matched_sources',[])),
            'PASS' if source_report.get('matched_sources') else 'FAIL',
            'A mixed source is necessary, not sufficient, for matched acoustic conditions')
    if preds is not None and not preds.empty:
        levels=set(preds.level.unique());required=set(S.available_levels(cfg.scenario_profile))|{0}
        add('Scenario level coverage',sorted(required),sorted(levels),
            'PASS' if required.issubset(levels) else 'FAIL')
    d=pd.DataFrame(rows);d.to_csv(out/'V2_ACCEPTANCE.csv',index=False)
    return d


def hidden_all_repeats(base_cfg,split_df,seeds=(42,123,777),models=None):
    """ONE final heldout evaluation of ALL frozen seed checkpoints.

    Caller MUST invoke reserve_hidden_evaluation BEFORE this function. Predictions
    are analyzed per seed — multiple seeds are not independent audio samples.
    """
    out=Path(base_cfg.out)
    if not (out/'HIDDEN_TEST_ONE_SHOT_STARTED.json').exists():
        raise RuntimeError('Hidden test reservation missing; call reserve_hidden_evaluation first.')
    if len(seeds)<3 or len(seeds)>5:raise ValueError('Need 3–5 frozen seeds')
    names=tuple(models or base_cfg.models)
    seed_asri=[];all_parts=[];coverage=[]
    for seed in seeds:
        cfg=replace(base_cfg,seed=int(seed),out=str(out/'replicates'/f'seed_{seed}'),
                    scenario_cache_dir=str(out),test_finalization=True)
        S.verify_locked_protocol(cfg,split_df)
        train_dir=Path(cfg.out)
        weights=json.loads((train_dir/'05_adaptive_weights.json').read_text(encoding='utf-8'))
        parts=[]
        for name in names:
            for regime in ('static','random','adaptive'):
                # Only validated frozen weights are loaded; train_one reuses checkpoints.
                model,meta=S.train_one(split_df,cfg,name,regime,
                    adaptive_weights=weights if regime=='adaptive' else None,force=False)
                result=S.run_scenario_predictions(split_df,model,name,regime,cfg,
                   threshold=meta['threshold'],split='hidden_test')
                result['training_seed']=seed
                parts.append(result)
        pred=pd.concat(parts,ignore_index=True)
        pred.to_csv(train_dir/'07_hidden_predictions_ALL.csv',index=False)
        _,asri=S.summarize_predictions(pred,cfg,report_tag='hidden_test')
        asri['training_seed']=seed
        seed_asri.append(asri)
        all_parts.append(pred)
        coverage.append({'seed':int(seed),'scenarios':len(pred),'base_clips':
                         int(split_df.split.eq('hidden_test').sum()),
                         'distinct_groups':int(split_df.loc[split_df.split.eq('hidden_test'),'group_id'].nunique())})
    allpred=pd.concat(all_parts,ignore_index=True)
    allpred.to_csv(out/'07_hidden_predictions_ALL_SEEDS.csv',index=False)
    asri_by_seed=pd.concat(seed_asri,ignore_index=True)
    asri_by_seed.to_csv(out/'07_hidden_asri_by_seed.csv',index=False)
    asri_summary=asri_by_seed.groupby(['model','regime']).asri.agg(['count','mean','std','min','max']).reset_index()
    asri_summary.to_csv(out/'07_hidden_asri_summary.csv',index=False)
    S.save_json(out/'07_hidden_coverage.json',coverage)
    return allpred,asri_by_seed,asri_summary
'''
_module_name = 'saade_final_methods'
_mod = types.ModuleType(_module_name)
_mod.__file__ = '<SAADE_ALL_IN_ONE>/' + _module_name + '.py'
sys.modules[_module_name] = _mod
exec(compile(_module_source, _mod.__file__, 'exec'), _mod.__dict__)
print('Підключено: saade_final_methods')

In [ ]:
# Код модуля: saade_final_visuals.py (вбудований, інших файлів не потрібно)
_module_source = r'''
"""SAADE v2 visual and audio companion for the user-supplied AUA research notebook.

Usage:
    import saade_visuals as V
    V.show_audio_gallery(splits.iloc[0], cfg, show_players=True)
    V.report_scenario_tests(validation_predictions, cfg, tag='validation')

No source data are modified. Only figures/CSV are written to <cfg.out>/visuals/<tag>/.
No raw audio is exported automatically; notebook playback remains local to JupyterLab.
This module never opens hidden-test data unless the caller passes those predictions.
"""
from __future__ import annotations
import math, re, hashlib, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

import saade_final_core as S

CATEGORY_UA={
    'baseline':'Оригінал',
    'noise':'Гаусівський шум',
    'compression':'MP3-компресія',
    'reverberation':'Реверберація (синтетична)',
    'lowpass':'Низькочастотний фільтр',
}
CATEGORY_EN={'baseline':'Baseline', 'noise':'Noise', 'compression':'MP3',
             'reverberation':'Reverb', 'lowpass':'Low-pass'}
REGIME_EN={'static':'E1 · Static','random':'E2 · Random','adaptive':'E3 · Adaptive'}
METRIC_INFO={
    'balanced_accuracy':('Balanced Accuracy','0–1; більше = краще',0,1),
    'f1_fake':('F1 для FAKE','0–1; більше = краще',0,1),
    'roc_auc':('ROC-AUC','0–1; більше = краще',0,1),
    'eer':('EER','0–1; менше = краще',0,1),
    'false_alarm_rate':('False Alarm Rate','частка REAL помилково як FAKE',0,1),
    'miss_rate':('Miss Rate','частка FAKE помилково як REAL',0,1),
}


def scenario_keys(cfg):
    return [('baseline',0)]+[(c,k) for c in S.CATEGORIES for k in S.available_levels(cfg.scenario_profile)]


def label_for_scenario(category, level, cfg):
    if category=='baseline':return '00 · Original / baseline'
    severity=S.scenario_severity(category,int(level),cfg.scenario_profile)
    unit={'noise':'dB SNR','compression':'kbps MP3','reverberation':'s RT60*','lowpass':'Hz'}[category]
    return f'{CATEGORY_EN[category]} L{int(level)} · {severity:g} {unit}'


def all_scenario_metadata(cfg):
    rows=[]
    for i,(cat,lev) in enumerate(scenario_keys(cfg)):
        value=S.scenario_severity(cat,lev,cfg.scenario_profile)
        rows.append(dict(order=i,category=cat,category_ua=CATEGORY_UA[cat],level=int(lev),
                         parameter=value,description=label_for_scenario(cat,lev,cfg),
                         scenario_key=f'{cat}::{lev}',
                         scenario_profile=cfg.scenario_profile))
    return pd.DataFrame(rows)


def _out(cfg,tag='validation'):
    folder=Path(cfg.out)/'visuals'/S.safe_key(tag)
    folder.mkdir(parents=True,exist_ok=True)
    return folder


def _save(fig,path,show=True):
    fig.savefig(path,dpi=160,bbox_inches='tight')
    if show:
        plt.show()
    plt.close(fig)
    return path


def _row_as_series(record):
    if isinstance(record,pd.DataFrame):return record.iloc[0]
    if isinstance(record,pd.Series):return record
    if isinstance(record,dict):return pd.Series(record)
    return pd.Series(record._asdict())


def pick_examples(splits, split='validation'):
    """Picks 1 genuine and 1 fake only from requested split (default VALIDATION)."""
    d=splits[splits.split.eq(split)].copy()
    if d.empty:raise ValueError(f'Empty split: {split}')
    results=[]
    for label in [0,1]:
        x=d[d.label.eq(label)].sort_values(['source','relative_path'])
        if not x.empty:results.append(x.iloc[0])
    return results


def waves_for_record(record,cfg):
    """Generate the exact scenario audio that is fed to the models, respecting profile and cache."""
    r=_row_as_series(record)
    required=['path','relative_path']
    missing=[k for k in required if k not in r or not str(r[k]).strip()]
    if missing:raise ValueError('Audio row missing: '+','.join(missing))
    def attr(name,default=''):
        return r.get(name,default)
    class R:
        pass
    waves={}
    for cat,lev in scenario_keys(cfg):
        rr=R()
        rr.path=str(attr('path'))
        rr.segment_start_sec=attr('segment_start_sec',None) # V2_SEGMENT_VISUAL
        rr.relative_path=str(attr('relative_path'))
        rr.category=cat
        rr.level=lev
        rr.label=int(attr('label',0))
        rr.source=str(attr('source'))
        rr.group_id=str(attr('group_id'))
        rr.speaker_id=str(attr('speaker_id'))
        rr.generator=str(attr('generator'))
        rr.split=str(attr('split','validation'))
        waves[(cat,lev)]=S.get_scenario_wave(rr,cfg)
    return waves


def _spec(x,sr):
    _,_,z=signal.stft(x,fs=sr,nperseg=min(512,len(x)),noverlap=min(384,max(0,len(x)//2)),
                      boundary=None)
    return 20*np.log10(np.maximum(1e-7,np.abs(z)))


def _wave_stats(waves,cfg):
    baseline=np.asarray(waves[('baseline',0)],float)
    x0=baseline[:len(baseline)]
    rows=[]
    for order,(cat,lev) in enumerate(scenario_keys(cfg)):
        x=np.asarray(waves[(cat,lev)],float)
        rms=float(np.sqrt(np.mean(x*x)+1e-12))
        error=x-x0
        diff_rms=float(np.sqrt(np.mean(error*error)+1e-12))
        corr=float(np.corrcoef(x,x0)[0,1]) if np.std(x)>1e-8 and np.std(x0)>1e-8 else np.nan
        rows.append(dict(order=order,category=cat,level=lev,
             description=label_for_scenario(cat,lev,cfg),duration_sec=len(x)/cfg.sample_rate,
             rms=rms,peak=float(np.max(np.abs(x))),change_rms=diff_rms,
             correlation_with_original=corr,
             inferred_snr_db=(10*np.log10((np.mean(x0*x0)+1e-12)/(np.mean(error*error)+1e-12))
                if cat=='noise' else np.nan)))
    return pd.DataFrame(rows)


def show_audio_gallery(record,cfg,show_players=True,show_figures=True,tag='validation',max_players=None):
    """All 13 (v1) or 21 (v2) real playable scenario audio items and figures.

    Audio player labels represent local transforms. Normalization is ONLY for audition;
    the exact waveform fed to a model remains unaltered in generated figures and scores.
    """
    r=_row_as_series(record)
    label=int(r.get('label',0)); group=str(r.get('group_id',''))
    ident=hashlib.sha256(str(r.get('relative_path','')).encode()).hexdigest()[:10]
    print(f'КЛАС: {"FAKE / synthetic" if label==1 else "REAL / bona fide"}; '
          f'джерело: {r.get("source", "?")}; фрагмент ID: {ident}; '
          f'профіль: {cfg.scenario_profile}')
    print('Відтворення сегмента:',r.get('segment_start_sec','центр запису'),'с; тривалість',cfg.seconds,'с')
    waves=waves_for_record(r,cfg)
    info=_wave_stats(waves,cfg)
    path=_out(cfg,tag)
    info.to_csv(path/f'audio_scenario_stats_{ident}.csv',index=False)
    from IPython.display import display, Audio, Markdown
    display(info[['description','rms','peak','change_rms','correlation_with_original','inferred_snr_db']].round(4))

    keys=scenario_keys(cfg)
    ncol=3
    nrow=math.ceil(len(keys)/ncol)
    if show_figures:
        t=np.arange(int(cfg.seconds*cfg.sample_rate))/cfg.sample_rate
        fig,axs=plt.subplots(nrow,ncol,figsize=(16,2.3*nrow),sharex=True,sharey=True,squeeze=False)
        for ax,(cat,lev) in zip(axs.flat,keys):
            x=waves[(cat,lev)]
            ax.plot(t[:len(x)][::32],x[::32],linewidth=.55)
            ax.set_title(label_for_scenario(cat,lev,cfg),fontsize=9)
            ax.grid(alpha=.2)
        for ax in axs.flat[len(keys):]:ax.axis('off')
        fig.suptitle(f'ВСІ {len(keys)} СЦЕНАРІЇ · waveform · ID {ident}',y=1.003,fontsize=15)
        fig.supxlabel('Час, с');fig.supylabel('Амплітуда PCM');fig.tight_layout()
        _save(fig,path/f'audio_all_waveforms_{ident}.png',show=True)

        fig,axs=plt.subplots(nrow,ncol,figsize=(16,2.75*nrow),sharex=True,sharey=True,squeeze=False)
        for ax,(cat,lev) in zip(axs.flat,keys):
            x=waves[(cat,lev)]
            f,tt,zz=signal.stft(x,fs=cfg.sample_rate,nperseg=512,noverlap=384)
            ax.pcolormesh(tt,f/1000,20*np.log10(np.maximum(1e-7,np.abs(zz))),
                          shading='auto',cmap='magma',vmin=-85,vmax=-20)
            ax.set_ylim(0,8);ax.set_title(label_for_scenario(cat,lev,cfg),fontsize=9)
        for ax in axs.flat[len(keys):]:ax.axis('off')
        fig.suptitle(f'ВСІ {len(keys)} СЦЕНАРІЇ · спектрограми STFT · ID {ident}',y=1.003,fontsize=15)
        fig.supxlabel('Час, с');fig.supylabel('Частота, кГц');fig.tight_layout()
        _save(fig,path/f'audio_all_spectrograms_{ident}.png',show=True)

    if show_players:
        nshow=len(keys) if max_players is None else max(1,min(int(max_players),len(keys)))
        print(f'▶ ПРОСЛУХОВУВАННЯ: {nshow}/{len(keys)} сценаріїв. Звук нормалізується тільки у плеєрі.')
        for cat,lev in keys[:nshow]:
            display(Markdown('**'+label_for_scenario(cat,lev,cfg)+'**'))
            display(Audio(waves[(cat,lev)],rate=cfg.sample_rate,normalize=True))
    return info


def draw_scenario_guide(cfg,tag='validation'):
    guide=all_scenario_metadata(cfg)
    out=_out(cfg,tag)
    guide.to_csv(out/'00_scenario_guide.csv',index=False)
    print(f'Протокол {cfg.scenario_profile}: {len(guide)} умов; {len(guide)-1} акустичних впливів.')
    from IPython.display import display
    display(guide[['description','category_ua','level','parameter']])
    fig,axs=plt.subplots(1,4,figsize=(16,3),sharex=False)
    for ax,c in zip(axs,S.CATEGORIES):
        g=guide[guide.category.eq(c)];x=g.level.to_numpy();y=g.parameter.to_numpy(dtype=float)
        ax.plot(x,y,marker='o');ax.grid(alpha=.3)
        ax.set_title(CATEGORY_UA[c]);ax.set_xlabel('Рівень впливу');
        unit={'noise':'SNR, dB ↓','compression':'MP3, kbps ↓','reverberation':'RT60*, c ↑','lowpass':'Частота зрізу, Hz ↓'}[c]
        ax.set_ylabel(unit);ax.set_xticks(x)
    fig.suptitle('Параметри 4 типів спотворень (ordered v2 / legacy v1)')
    fig.tight_layout();_save(fig,out/'00_scenario_levels.png')
    return guide


def _with_scenarios(df,cfg):
    a=df.copy()
    a['category']=a['category'].astype(str)
    a['level']=pd.to_numeric(a['level'],errors='coerce').astype('Int64')
    order={k:i for i,k in enumerate(scenario_keys(cfg))}
    a['key']=list(zip(a.category,a.level.fillna(-1).astype(int)))
    a['scenario_order']=a['key'].map(order)
    a['scenario_label']=[label_for_scenario(c,l,cfg) if (c,l) in order else f'{c} L{l}'
                         for c,l in zip(a.category,a.level.fillna(-1))]
    return a


def _coerce_predictions(preds,cfg):
    if preds is None or len(preds)==0:raise ValueError('Empty prediction table; run E1/E2/E3 or load CSV.')
    needed={'scenario_id','label','score','threshold','model','regime','category','level','split'}
    missing=needed-set(preds.columns)
    if missing:raise ValueError(f'Predictions missing columns: {sorted(missing)}')
    x=preds.copy()
    for k in ['label','score','threshold','level']:
        x[k]=pd.to_numeric(x[k],errors='coerce')
    x=x[x.label.isin([0,1]) & x.score.notna() & x.threshold.notna() & x.level.notna()].copy()
    x['level']=x['level'].astype(int)
    x['label']=x['label'].astype(int)
    x['pred']=(x.score>=x.threshold).astype(int)
    x['correct']=(x.pred==x.label).astype(int)
    x=_with_scenarios(x,cfg)
    if x.scenario_order.isna().any():
        unknown=x.loc[x.scenario_order.isna(),['category','level']].drop_duplicates()
        raise ValueError('Incompatible scenario protocol. Unexpected (category,level): '+str(unknown.to_dict('records')))
    return x


def scenario_metric_table(predictions,cfg):
    """One row per model, regime, category, severity, with BOTH classes required.
    Uses threshold locked for each model from clean validation, not reoptimized per scenario.
    """
    p=_coerce_predictions(predictions,cfg)
    rows=[]
    for (model,regime,cat,level),g in p.groupby(['model','regime','category','level'],sort=False):
        meta=dict(model=model,regime=regime,category=cat,level=int(level),
                  scenario_label=label_for_scenario(cat,level,cfg),
                  n=len(g),n_real=int((g.label==0).sum()),n_fake=int((g.label==1).sum()),
                  n_groups=int(g.group_id.nunique()) if 'group_id' in g.columns else np.nan,
                  split=' | '.join(sorted(g.split.astype(str).unique())))
        if g.label.nunique()!=2:
            meta.update(status='CHECK_ONE_CLASS',**{k:np.nan for k in METRIC_INFO})
        elif g.score.nunique()<2:
            values=S.metric_row(g.label,g.score,float(g.threshold.iloc[0]))
            meta.update(status='CHECK_CONSTANT_SCORES',**values)
        else:
            meta.update(status='OK',**S.metric_row(g.label,g.score,float(g.threshold.iloc[0])))
        rows.append(meta)
    r=pd.DataFrame(rows)
    r=_with_scenarios(r,cfg).sort_values(['model','regime','scenario_order'])
    return r


def plot_source_class_balance(predictions,cfg,tag='validation'):
    p=_coerce_predictions(predictions,cfg)
    x=p[p.category.eq('baseline')].drop_duplicates(['scenario_id'])
    group=x.groupby(['source','label']).size().unstack(fill_value=0).reindex(columns=[0,1],fill_value=0)
    group.columns=['REAL','FAKE']
    out=_out(cfg,tag)
    group.to_csv(out/'source_class_balance.csv')
    fig,ax=plt.subplots(figsize=(max(8,min(16,len(group)*1.5)),4))
    group.plot.bar(stacked=True,ax=ax,rot=35)
    ax.set_xlabel('Джерело');ax.set_ylabel('Незалежні базові фрагменти')
    ax.set_title('Баланс REAL/FAKE за джерелом · перевірка source confounding')
    ax.legend(title='Клас');fig.tight_layout();_save(fig,out/'source_class_balance.png')
    return group.reset_index()


def plot_scenario_heatmaps(metrics,cfg,tag='validation',show=True):
    """Show ALL scenarios for ALL models x regimes for accuracy, EER, F1, miss/FA rates."""
    out=_out(cfg,tag)
    profiles=[]
    order=[label_for_scenario(c,l,cfg) for c,l in scenario_keys(cfg)]
    for metric in ['balanced_accuracy','eer','f1_fake','roc_auc','false_alarm_rate','miss_rate']:
        x=metrics.copy()
        x['configuration']=x['model']+' · '+x['regime'].map(lambda s:REGIME_EN.get(s,s))
        mat=x.pivot_table(index='configuration',columns='scenario_label',values=metric,aggfunc='first',dropna=False)
        mat=mat.reindex(columns=order)
        mat.to_csv(out/f'table_ALL_scenarios_{metric}.csv')
        fig,ax=plt.subplots(figsize=(max(13,0.75*len(order)),max(4,0.48*len(mat)+2)))
        image=ax.imshow(mat.values,aspect='auto',cmap='viridis' if metric not in ('eer','miss_rate','false_alarm_rate') else 'magma_r',vmin=0,vmax=1)
        ax.set_xticks(range(len(order)),order,rotation=55,ha='right',fontsize=8)
        ax.set_yticks(range(len(mat)),mat.index,fontsize=9)
        ax.set_title(f'{METRIC_INFO[metric][0]} за ВСІМА {len(order)} сценаріями · {tag}')
        ax.set_xlabel('Акустичний сценарій');ax.set_ylabel('Архітектура / режим')
        for i in range(len(mat)):
            for j in range(len(order)):
                v=mat.iloc[i,j]
                if pd.notna(v):ax.text(j,i,f'{v:.2f}',ha='center',va='center',fontsize=7,
                                     color='white' if (.2<v<.7) else 'black')
        fig.colorbar(image,ax=ax,label=METRIC_INFO[metric][1],fraction=.02,pad=.01)
        fig.tight_layout(); _save(fig,out/f'heatmap_ALL_scenarios_{metric}.png',show)
        profiles.append(mat)
    return profiles


def plot_per_category_curves(metrics,cfg,tag='validation',show=True):
    """For each acoustic category: lines for E1 E2 E3, separately for each model."""
    out=_out(cfg,tag)
    for metric in ['balanced_accuracy','eer']:
        for c in S.CATEGORIES:
            d=metrics[metrics.category.eq(c)].copy()
            if d.empty:continue
            names=sorted(d.model.unique())
            fig,axes=plt.subplots(1,len(names),figsize=(5*len(names),4),squeeze=False,sharey=True)
            for ax,name in zip(axes[0],names):
                z=d[d.model.eq(name)]
                for regime in ['static','random','adaptive']:
                    g=z[z.regime.eq(regime)].sort_values('level')
                    if not g.empty:ax.plot(g.level,g[metric],marker='o',label=REGIME_EN[regime])
                ax.set_title(name);ax.set_xticks(S.available_levels(cfg.scenario_profile))
                ax.set_xlabel('Рівень акустичного впливу');ax.set_ylim(0,1);ax.grid(alpha=.25)
            axes[0][0].set_ylabel(METRIC_INFO[metric][0]+' [0–1]')
            axes[0][-1].legend(fontsize=8,loc='best')
            fig.suptitle(f'{CATEGORY_UA[c]} · {METRIC_INFO[metric][0]} · {tag}',y=1.02)
            fig.tight_layout();_save(fig,out/f'curves_{c}_{metric}.png',show)


def plot_baseline_degradation(metrics,cfg,tag='validation',show=True):
    """All scenarios relative to SAME model/regime baseline on SAME evaluation split."""
    out=_out(cfg,tag)
    d=metrics.copy()
    d['configuration']=d.model+' · '+d.regime
    base=d[d.category.eq('baseline')][['configuration','balanced_accuracy']].rename(columns={'balanced_accuracy':'ba_baseline'})
    d=d.merge(base,on='configuration',how='left',validate='many_to_one')
    d['delta_ba_vs_clean']=d.balanced_accuracy-d.ba_baseline
    d=d[d.category.ne('baseline')]
    d.to_csv(out/'table_all_degradation_vs_clean.csv',index=False)
    scenarios=[label_for_scenario(c,l,cfg) for c,l in scenario_keys(cfg)[1:]]
    mat=d.pivot_table(index='configuration',columns='scenario_label',values='delta_ba_vs_clean',aggfunc='first')
    mat=mat.reindex(columns=scenarios)
    fig,ax=plt.subplots(figsize=(max(12,.7*len(scenarios)),max(4,.45*len(mat)+2)))
    extent=np.nanmax(abs(mat.to_numpy(dtype=float))) if np.isfinite(mat.to_numpy(dtype=float)).any() else .5
    extent=max(.05,float(extent))
    im=ax.imshow(mat.values,aspect='auto',cmap='coolwarm_r',vmin=-extent,vmax=extent)
    ax.set_xticks(range(len(scenarios)),scenarios,rotation=55,ha='right',fontsize=8)
    ax.set_yticks(range(len(mat)),mat.index)
    ax.set_title('Втрата / приріст Balanced Accuracy відносно CLEAN baseline')
    for i in range(len(mat)):
        for j in range(len(scenarios)):
            v=mat.iloc[i,j]
            if pd.notna(v):ax.text(j,i,f'{v:+.2f}',ha='center',va='center',fontsize=7)
    fig.colorbar(im,ax=ax,label='Δ Balanced Accuracy (scenario − baseline)',fraction=.02,pad=.01)
    fig.tight_layout();_save(fig,out/'heatmap_delta_BA_vs_baseline.png',show)
    return d


def plot_asri(asri,cfg,tag='validation',show=True):
    out=_out(cfg,tag)
    if asri is None or asri.empty:return
    table=asri.pivot(index='model',columns='regime',values='asri')
    table=table.reindex(columns=[k for k in ['static','random','adaptive'] if k in table.columns])
    table.to_csv(out/'table_asri.csv')
    fig,ax=plt.subplots(figsize=(max(7,len(table)*2.5),4.3))
    table.plot.bar(ax=ax,rot=0,ylim=(0,1))
    ax.set_title(f'ASRI за {4*len(S.available_levels(cfg.scenario_profile))} акустичними сценаріями · {tag}')
    ax.set_ylabel('ASRI [0,1]');ax.set_xlabel('Архітектура');ax.grid(axis='y',alpha=.25)
    fig.tight_layout();_save(fig,out/'ASRI_comparison.png',show)


def plot_error_confusions(predictions,cfg,model='spectrogram_cnn',regime='adaptive',tag='validation',show=True):
    """For ONE selected model/regime show confusion matrices for ALL 13/21 scenarios."""
    p=_coerce_predictions(predictions,cfg)
    d=p[p.model.eq(model)&p.regime.eq(regime)]
    if d.empty:
        print('No model/regime predictions:',model,regime);return pd.DataFrame()
    keys=scenario_keys(cfg)
    ncols=4;nrows=math.ceil(len(keys)/ncols)
    fig,axs=plt.subplots(nrows,ncols,figsize=(14,3.2*nrows),squeeze=False)
    rows=[]
    for ax,(cat,lev) in zip(axs.flat,keys):
        z=d[d.category.eq(cat)&d.level.eq(lev)]
        tn=int(((z.label==0)&(z.pred==0)).sum());fp=int(((z.label==0)&(z.pred==1)).sum())
        fn=int(((z.label==1)&(z.pred==0)).sum());tp=int(((z.label==1)&(z.pred==1)).sum())
        cm=np.array([[tn,fp],[fn,tp]])
        ax.imshow(cm,cmap='Blues',vmin=0,vmax=max(1,cm.max()))
        ax.set_xticks([0,1],['REAL','FAKE']);ax.set_yticks([0,1],['REAL','FAKE'])
        ax.set_xlabel('Прогноз');ax.set_ylabel('Еталон')
        for i in range(2):
            for j in range(2):ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=11)
        ax.set_title(label_for_scenario(cat,lev,cfg),fontsize=9)
        rows.append(dict(category=cat,level=lev,n=len(z),tn=tn,fp=fp,fn=fn,tp=tp))
    for ax in axs.flat[len(keys):]:ax.axis('off')
    fig.suptitle(f'ВСІ confusion matrices: {model} / {regime} / {tag}',y=1.001,fontsize=15)
    fig.tight_layout();_save(fig,_out(cfg,tag)/f'confusion_ALL_{S.safe_key(model)}_{S.safe_key(regime)}.png',show)
    result=pd.DataFrame(rows)
    result.to_csv(_out(cfg,tag)/f'confusion_ALL_{S.safe_key(model)}_{S.safe_key(regime)}.csv',index=False)
    return result


def show_one_record_decisions(predictions,record,cfg,tag='validation',show=True):
    """All scenario predictions for ALL model/regime configurations for ONE source file."""
    p=_coerce_predictions(predictions,cfg);r=_row_as_series(record)
    ref=str(r.relative_path)
    z=p[p.relative_path.astype(str).eq(ref)]
    if z.empty:
        print('No predictions for this sample in this evaluation split.');return pd.DataFrame()
    colorder=[label_for_scenario(c,l,cfg) for c,l in scenario_keys(cfg)]
    z=z.copy();z['config']=z.model+' · '+z.regime
    pivot=z.pivot_table(index='config',columns='scenario_label',values='correct',aggfunc='first').reindex(columns=colorder)
    nrows=max(3,int(.6*len(pivot)+1))
    fig,ax=plt.subplots(figsize=(max(13,.65*len(colorder)),nrows))
    im=ax.imshow(pivot.values,aspect='auto',cmap='RdYlGn',vmin=0,vmax=1)
    ax.set_xticks(range(len(colorder)),colorder,rotation=65,ha='right',fontsize=8)
    ax.set_yticks(range(len(pivot)),pivot.index,fontsize=9)
    for i in range(len(pivot)):
        for j in range(len(colorder)):
            v=pivot.iloc[i,j]
            if pd.notna(v):ax.text(j,i,'✓' if int(v)==1 else '✗',ha='center',va='center',fontsize=11)
    ax.set_title(f'Правильність прогнозу для ОДНОГО аудіозапису · {tag}\n'
                 'Зелений = правильно · червоний = помилка; score не зіставляти між архітектурами')
    fig.tight_layout();id10=hashlib.sha256(ref.encode()).hexdigest()[:10]
    _save(fig,_out(cfg,tag)/f'one_audio_{id10}_model_decisions.png',show)
    table=z[['model','regime','scenario_label','label','score','threshold','pred','correct']].sort_values(['model','regime','scenario_label'])
    table.to_csv(_out(cfg,tag)/f'one_audio_{id10}_scenario_decisions.csv',index=False)
    from IPython.display import display
    display(table.head(40).round(4))
    return table


def scenario_failure_examples(predictions,cfg,tag='validation',max_examples=150):
    """Hard examples (score crosses threshold incorrectly); path is kept server-local, never audio."""
    p=_coerce_predictions(predictions,cfg)
    p['wrong_margin']=np.where(p.correct.eq(0),abs(p.score-p.threshold),0)
    failures=p[p.correct.eq(0)].sort_values('wrong_margin',ascending=False)
    cols=['model','regime','category','level','relative_path','source','generator','group_id','label','pred','score','threshold','wrong_margin']
    cols=[c for c in cols if c in failures.columns]
    fail=failures[cols].head(max_examples)
    fail.to_csv(_out(cfg,tag)/'failure_examples_for_review.csv',index=False)
    return fail


def report_scenario_tests(predictions,cfg,tag='validation',show=True,confusion_model=None,
                          confusion_regime='adaptive',include_source_plot=True):
    """Report scenario-wise results for every model and regime on current split.

    Does not train models or open hidden-test audio. All charts and tables are saved to
    a tag-specific folder; validation and hidden cannot overwrite one another.
    """
    p=_coerce_predictions(predictions,cfg)
    observed=sorted(p.split.astype(str).unique())
    print('ТИП ОЦІНЮВАННЯ:', tag, '| фактичні split:', observed)
    if tag=='validation' and observed!=['validation']:
        warnings.warn('The validation dashboard received non-validation rows. Review before interpreting.')
    if tag=='hidden_test' and observed!=['hidden_test']:
        warnings.warn('The hidden dashboard received non-hidden data. Review before final publication.')
    out=_out(cfg,tag)
    metrics=scenario_metric_table(p,cfg)
    metrics.to_csv(out/'ALL_SCENARIO_METRICS.csv',index=False)
    asri=S.asri_table(p,expected_levels=len(S.available_levels(cfg.scenario_profile)))
    asri.to_csv(out/'ASRI_BY_MODEL_REGIME.csv',index=False)
    required=len(scenario_keys(cfg))
    counts=metrics.groupby(['model','regime']).agg(
        scenarios=('scenario_label','nunique'),
        min_n=('n','min'),max_n=('n','max'),
        min_real=('n_real','min'),min_fake=('n_fake','min'),
        n_failed_cells=('status',lambda x:(x=='CHECK_ONE_CLASS').sum()),
        n_constant_score_warnings=('status',lambda x:(x=='CHECK_CONSTANT_SCORES').sum())).reset_index()
    counts['expected_scenarios']=required
    counts['complete']=(counts.scenarios.eq(required)&counts.min_real.gt(0)&counts.min_fake.gt(0)&counts.n_failed_cells.eq(0))
    counts.to_csv(out/'ALL_SCENARIO_COVERAGE.csv',index=False)
    from IPython.display import display
    print(f'Покриття ВСІХ сценаріїв ({required}) для кожної моделі × режиму:')
    display(counts)
    print('ASRI:')
    display(asri.sort_values(['model','regime']))
    summary=metrics[metrics.category.ne('baseline')].groupby(['model','regime']).agg(
        mean_balanced_accuracy=('balanced_accuracy','mean'),worst_balanced_accuracy=('balanced_accuracy','min'),
        mean_EER=('eer','mean'),mean_F1_fake=('f1_fake','mean'),mean_ROC_AUC=('roc_auc','mean')).reset_index()
    display(summary.round(4));summary.to_csv(out/'MODEL_SUMMARY_ALL_SCENARIOS.csv',index=False)
    plot_asri(asri,cfg,tag,show)
    plot_scenario_heatmaps(metrics,cfg,tag,show)
    plot_per_category_curves(metrics,cfg,tag,show)
    plot_baseline_degradation(metrics,cfg,tag,show)
    if include_source_plot:plot_source_class_balance(p,cfg,tag)
    if confusion_model is None:
        confusion_model='spectrogram_cnn' if 'spectrogram_cnn' in p.model.unique() else str(p.model.iloc[0])
    if confusion_regime not in p.regime.unique():confusion_regime=str(p.regime.iloc[0])
    plot_error_confusions(p,cfg,confusion_model,confusion_regime,tag,show)
    failures=scenario_failure_examples(p,cfg,tag)
    print(f'Помилкових рішень (усі сценарії × моделі): {(p.correct==0).sum():,}. '
          f'Реєстр прикладів: {len(failures)} рядків; аудіофайли не копіюються.')
    print('Графіки / таблиці:',out)
    return metrics,asri,counts


def plot_roc_for_selected(predictions,cfg,model='spectrogram_cnn',regime='adaptive',tag='validation',show=True):
    """ROC baseline vs strongest severity in each of 4 groups; labels need two classes."""
    from sklearn.metrics import roc_curve,roc_auc_score
    p=_coerce_predictions(predictions,cfg)
    z=p[p.model.eq(model)&p.regime.eq(regime)]
    if z.empty:return
    max_level=max(S.available_levels(cfg.scenario_profile))
    fig,axs=plt.subplots(2,2,figsize=(12,9),squeeze=False)
    for ax,cat in zip(axs.flat,S.CATEGORIES):
        for scenario,level in [('baseline',0),(cat,max_level)]:
            d=z[z.category.eq(scenario)&z.level.eq(level)]
            if d.label.nunique()!=2:continue
            fpr,tpr,_=roc_curve(d.label,d.score)
            auc=roc_auc_score(d.label,d.score)
            ax.plot(fpr,tpr,label=f'{scenario} L{level} · AUC {auc:.3f}')
        ax.plot([0,1],[0,1],linestyle='--',alpha=.4)
        ax.set_title(CATEGORY_UA[cat]);ax.set_xlabel('FPR');ax.set_ylabel('TPR');ax.legend(fontsize=8)
        ax.grid(alpha=.2)
    fig.suptitle(f'ROC: baseline проти найсильнішого рівня · {model}/{regime}/{tag}')
    fig.tight_layout();_save(fig,_out(cfg,tag)/f'roc_baseline_vs_worst_{S.safe_key(model)}_{S.safe_key(regime)}.png',show)


def make_interactive_player(splits,cfg,split='validation'):
    """Optional ipywidgets explorer: select sample, category, level; play exact transformed wave."""
    try:
        import ipywidgets as W
        from IPython.display import Audio, display, clear_output
    except ImportError:
        print('Для інтерактивного меню встановіть `%pip install ipywidgets`. '
              'Без нього функція show_audio_gallery() уже показує всі плеєри.')
        return None
    d=splits[splits.split.eq(split)].copy().sort_values(['label','source','relative_path'])
    if d.empty:raise ValueError(f'No records in {split}')
    opts=[(f'{"FAKE" if int(r.label) else "REAL"} · {r.source} · {str(r.relative_path).split("/")[-1][:65]}',int(i))
         for i,r in d.iterrows()]
    clip=W.Dropdown(options=opts,description='Файл:',layout=W.Layout(width='90%'))
    scenarios=scenario_keys(cfg)
    scenario_opts=[(label_for_scenario(cat,lev,cfg),(cat,lev)) for cat,lev in scenarios]
    case=W.Dropdown(options=scenario_opts,description='Сценарій:',layout=W.Layout(width='90%'))
    go=W.Button(description='▶ Показати й відтворити',button_style='info')
    output=W.Output(layout=W.Layout(border='1px solid #ddd',padding='6px'))
    def render(_):
        with output:
            clear_output(wait=True)
            row=d.loc[clip.value]
            cat,level=case.value
            waves=waves_for_record(row,cfg)
            original=waves[('baseline',0)]
            changed=waves[(cat,level)]
            print(f'ID: {row.relative_path}; REAL/FAKE: {int(row.label)}; scenario={label_for_scenario(cat,level,cfg)}')
            display(Audio(changed,rate=cfg.sample_rate,normalize=True))
            t=np.arange(len(changed))/cfg.sample_rate
            fig,axes=plt.subplots(2,1,figsize=(12,5))
            axes[0].plot(t[::24],original[::24],label='Baseline',alpha=.8)
            axes[0].plot(t[::24],changed[::24],label='Scenario',alpha=.7)
            axes[0].set_title('Waveform: original vs scenario');axes[0].legend()
            f,tt,zz=signal.stft(changed,fs=cfg.sample_rate,nperseg=512,noverlap=384)
            axes[1].pcolormesh(tt,f/1000,20*np.log10(np.maximum(1e-7,abs(zz))),
                                shading='auto',cmap='magma')
            axes[1].set_ylim(0,8);axes[1].set_xlabel('Time, s');axes[1].set_ylabel('Frequency, kHz')
            axes[1].set_title('Spectrogram of selected scenario')
            fig.tight_layout();plt.show();plt.close(fig)
    go.on_click(render)
    display(W.VBox([clip,case,go,output]));return (clip,case,go,output)
'''
_module_name = 'saade_final_visuals'
_mod = types.ModuleType(_module_name)
_mod.__file__ = '<SAADE_ALL_IN_ONE>/' + _module_name + '.py'
sys.modules[_module_name] = _mod
exec(compile(_module_source, _mod.__file__, 'exec'), _mod.__dict__)
print('Підключено: saade_final_visuals')

In [ ]:
# Код модуля: saade_final_adaptive.py (вбудований, інших файлів не потрібно)
_module_source = r'''
"""SAADE v5: reproducible scenario-adaptive training for thesis experiments.

Key invariants:
* AUA and Melnyk_dus are read-only and are never output directories.
* parent/speaker groups are split BEFORE segmentation;
* an adaptation fold is carved exclusively from TRAIN speaker/voice groups;
* adaptation fold selects SAADE augmentation distribution and thresholds;
* validation is an untouched comparison fold; hidden_test never read here;
* E1/E2/E3 share identical training draws, epochs, seeds and augmentation budget;
* real/fake class balancing is within each partition, no file deletion;
* model files and all outputs live in cfg.out unique for this protocol.
"""
from __future__ import annotations
from dataclasses import replace
from pathlib import Path
from collections import defaultdict
import hashlib, json, math, os, subprocess, warnings
import numpy as np
import pandas as pd
import saade_final_core as S
import saade_final_methods as M

PARTITIONS=('train','adaptation','validation','hidden_test')


def _out(cfg):
    root=Path(cfg.root).expanduser().resolve()
    out=Path(cfg.out).expanduser().resolve()
    if out==root or root in out.parents or any('melnyk' in p.casefold() for p in out.parts):
        raise ValueError('Forbidden output path (AUA or Melnyk folder).')
    if not out.is_relative_to(Path('/home/jovyan/work/audio_real_fake').resolve()):
        # Permit a local fixture for developer tests, but do not silently target a different work tree.
        if str(root).startswith('/home/jovyan/work'):
            raise ValueError('Production outputs must stay inside /home/jovyan/work/audio_real_fake/')
    out.mkdir(parents=True,exist_ok=True)
    return out


def create_adaptation_fold(parents,cfg,seed=42,target_fraction=.22,require_two_groups=True):
    """Hold out known identities from *training* as adaptation; never touch validation/test.

    This is group-disjoint by group_id (source/voice), not proof of independent generators.
    Unknown-identity train_only material remains strictly training-only.
    """
    out=_out(cfg)
    df=parents.copy()
    if 'split' not in df:raise ValueError('Missing preassigned split')
    if 'group_id' not in df or 'identity_status' not in df:
        raise ValueError('Missing provenance grouping')
    if (df.split=='adaptation').any(): raise ValueError('Adaptation fold already exists: use fresh parents')
    rng=np.random.default_rng(seed)
    group_moves=[]
    for label in (0,1):
        train=df[(df.split=='train')&(df.label.astype(int)==label)&(df.split_role=='eligible')]
        info=train.groupby('group_id').agg(n=('path','size')).reset_index()
        if require_two_groups and len(info)<2:
            raise ValueError(f'Class {label}: need >=2 VERIFIED train groups to reserve adaptation. Found {len(info)}')
        if len(info)<2:
            warnings.warn(f'Class {label}: no independent adaptation possible; cannot run E3 safely')
            continue
        # Reserve at least 1 group and keep >=1 group in model-training data.
        idx=rng.permutation(len(info)); ordered=info.iloc[idx]
        quota=max(1,int(round(len(train)*float(target_fraction))))
        selected=[];count=0
        for r in ordered.itertuples():
            if len(selected)>=len(info)-1:break
            selected.append(r.group_id);count+=r.n
            if count>=quota:break
        group_moves+=selected
    if not group_moves:raise ValueError('No adaptation groups selected')
    df.loc[(df.split=='train')&df.group_id.isin(group_moves),'split']='adaptation'
    for partition in ('train','adaptation','validation'):
        part=df[df.split==partition]
        if part.label.nunique()!=2:
            raise ValueError(f'{partition} missing a class after adaptation carve-out')
    if df.groupby('group_id').split.nunique().max()>1:
        raise AssertionError('Group leakage after adaptation carving')
    for hcol in ['source_sha256','working_sha256','path']:
        if hcol not in df:continue
        h=df[df[hcol].fillna('').astype(str).str.strip().ne('')].groupby(hcol).split.nunique()
        if (h>1).any():raise AssertionError(f'{hcol} across folds')
    df.to_csv(out/'V5_01_PARENT_FOLDS.csv',index=False)
    S.save_json(out/'V5_01_ADAPTATION_PROVENANCE.json',dict(seed=seed,fraction=target_fraction,
      identities_moved=group_moves,selection='metadata verified train groups only',
      held_out_validation_untouched=True,hidden_test_untouched=True))
    return df


def create_matched_adaptation_fold(parents,cfg,seed=42,target_fraction=.22):
    """Group- AND pair-disjoint adaptation using connected components.

    Every text-pair connects the human speaker group to a synthetic voice group.
    All rows reachable through either shared group or pair must stay together.
    If there are insufficient independent components, fail rather than split pairs.
    """
    out=_out(cfg)
    df=parents.copy().reset_index(drop=True)
    if not {'split','pair_id','group_id','label'}.issubset(df.columns):
        raise ValueError('Matched parents require split,pair_id,group_id,label')
    if df.pair_id.isna().any() or df.pair_id.astype(str).str.strip().eq('').any():
        raise ValueError('Every matched parent must have a pair_id')
    if df.groupby('pair_id').split.nunique().gt(1).any():
        raise AssertionError('Matched pair crosses existing folds')
    rng=np.random.default_rng(int(seed))
    train=df[df.split.eq('train')]
    if train.empty:raise ValueError('No paired training rows')
    # Bipartite union-find: pair nodes and voice/speaker group nodes.
    parent={}
    def find(x):
        parent.setdefault(x,x)
        while parent[x]!=x:
            parent[x]=parent[parent[x]];x=parent[x]
        return x
    def union(a,b):
        a=find(a);b=find(b)
        if a!=b:parent[b]=a
    for r in train.itertuples():
        union('PAIR:'+str(r.pair_id),'GROUP:'+str(r.group_id))
    pair_component={pid:find('PAIR:'+str(pid)) for pid in train.pair_id.unique()}
    train=train.copy();train['component_id']=train.pair_id.map(pair_component)
    components=train.groupby('component_id').agg(rows=('path','size'),pairs=('pair_id','nunique'),
                              labels=('label','nunique')).reset_index()
    if len(components)<2:
        raise ValueError('Only one independent connected speaker/voice component in train; '
                         'cannot build pair-safe adaptation. Provide more independent TTS voices.')
    if components.labels.lt(2).any():
        raise ValueError('At least one connected component has one class; inspect pair registry')
    components=components.iloc[rng.permutation(len(components))]
    quota=max(1,int(round(len(train)*float(target_fraction))))
    picked=[];count=0
    for rr in components.itertuples():
        if len(picked)>=len(components)-1:break
        picked.append(rr.component_id);count+=int(rr.rows)
        if count>=quota:break
    move_pairs=set(train[train.component_id.isin(picked)].pair_id)
    df.loc[df.split.eq('train')&df.pair_id.isin(move_pairs),'split']='adaptation'
    for col in ['pair_id','group_id','source_sha256','working_sha256','path']:
        if col not in df.columns:continue
        present=df[col].fillna('').astype(str).str.strip().ne('')
        if df.loc[present].groupby(col).split.nunique().gt(1).any():
            raise AssertionError(f'Leakage in matched split: {col}')
    if df.groupby(['split','pair_id']).label.nunique().ne(2).any():
        raise AssertionError('Split has an incomplete REAL/FAKE pair')
    for part in ('train','adaptation','validation','hidden_test'):
        if df[df.split.eq(part)].label.nunique()!=2:
            raise ValueError(f'{part}: not both classes')
    df.to_csv(out/'V5_01_PARENT_FOLDS.csv',index=False)
    S.save_json(out/'V5_01_ADAPTATION_PROVENANCE.json',dict(method='connected_components_speaker_voice_pair',
        train_components_before=len(components),components_to_adaptation=len(picked),
        adaptation_pairs=len(move_pairs),seed=int(seed),validation_untouched=True,hidden_untouched=True))
    return df


def balance_per_partition(segments,cfg,real_pct=.50,seed=42):
    """Random *downsampling* of segment rows without altering their split or parent link.

    Training statistics are computed on segments, not independent speakers. Outputs
    include exactly reproducible retained/excluded manifests. For text-matched pairs
    use balance_pairs() instead: do not sever parent-real/TTS coupling.
    """
    out=_out(cfg)
    if not 0<float(real_pct)<1:raise ValueError('real_pct must be in (0,1)')
    if segments.groupby('group_id').split.nunique().gt(1).any():
        raise AssertionError('Preexisting group leakage')
    rng=np.random.default_rng(seed)
    kept=[];removed=[];report=[]
    for part, sub in segments.groupby('split',sort=False):
        a=sub[sub.label.astype(int)==0];b=sub[sub.label.astype(int)==1]
        if len(a)==0 or len(b)==0:
            raise ValueError(f'{part}: only one class; cannot balance')
        # For strict 50/50 n is min count, otherwise fill largest possible exact ratio approx.
        size=min(int(len(a)/real_pct),int(len(b)/(1-real_pct)))
        if abs(real_pct-.5)<1e-12:
            n_real=n_fake=min(len(a),len(b));size=2*n_real
        else:
            n_real=min(len(a),int(round(size*real_pct)))
            n_fake=min(len(b),int(round(size*(1-real_pct))))
        aa=a.iloc[rng.choice(len(a),n_real,replace=False)]
        bb=b.iloc[rng.choice(len(b),n_fake,replace=False)]
        pick=pd.concat([aa,bb]);drop=sub.drop(index=pick.index)
        kept.append(pick);removed.append(drop)
        report.append(dict(split=part,real_before=len(a),fake_before=len(b),
            real_after=n_real,fake_after=n_fake,excluded=len(drop),
            real_fraction_after=n_real/(n_real+n_fake),
            groups_after=pick.group_id.nunique(),parents_after=pick.parent_file_id.nunique()))
    final=pd.concat(kept,ignore_index=True).sort_values(['split','relative_path']).reset_index(drop=True)
    dropped=pd.concat(removed,ignore_index=True) if removed else segments.iloc[:0].copy()
    assert final.groupby('group_id').split.nunique().max()==1
    assert final.groupby('parent_file_id').split.nunique().max()==1
    final.to_csv(out/'V5_02_BALANCED_SEGMENTS.csv',index=False)
    dropped.to_csv(out/'V5_02_EXCLUDED_SEGMENTS.csv',index=False)
    info=pd.DataFrame(report)
    info.to_csv(out/'V5_02_BALANCE_REPORT.csv',index=False)
    return final,info


def _unique_adaptation_subset(df,max_per_class=24,seed=42):
    # Parent-aware, not dominated by long recordings. Scenario evaluation unchanged.
    chunks=[];rng=np.random.default_rng(seed)
    for label in (0,1):
        x=df[df.label.astype(int)==label].copy()
        x=x.iloc[rng.permutation(len(x))].copy()
        x['parent_rank']=x.groupby('parent_file_id').cumcount()
        x=x.sort_values(['parent_rank','relative_path']).head(max_per_class).drop(columns='parent_rank')
        chunks.append(x)
    return pd.concat(chunks,ignore_index=True)


def _difficulty_policy(predictions,prior=10.,temperature=.12,uniform_floor=.20):
    """Group-wise confusion-balanced scenario difficulty -> category and level probabilities.

    Groups in adaptation (not signal windows) each contribute equally to difficulty.
    Shrink tiny cells toward 0.5. Uniform floor prevents collapsing the search.
    """
    x=predictions[predictions.category.isin(S.CATEGORIES)].copy()
    x['incorrect']=(x.pred.astype(int)!=x.label.astype(int)).astype(float)
    levels=list(S.available_levels('v2_5level'))
    risk={}; counts={}
    for cat in S.CATEGORIES:
        for lv in levels:
            cell=x[(x.category==cat)&(x.level==lv)]
            er=[]
            for lab in (0,1):
                xlab=cell[cell.label==lab]
                if len(xlab):
                    # One mean per independently identified group, not per duplicate window.
                    er.append(float(xlab.groupby('group_id').incorrect.mean().mean()))
            v=float(np.mean(er)) if len(er)==2 else .5
            support=int(len(cell))
            risk[(cat,lv)]=(support*v+prior*.5)/(support+prior)
            counts[(cat,lv)]=support
    overall=np.array([np.mean([risk[(c,l)] for l in levels]) for c in S.CATEGORIES],float)
    def soft(vals):
        z=np.asarray(vals,dtype=float);z=(z-z.max())/max(.03,temperature)
        p=np.exp(z);p=p/p.sum()
        return (1-uniform_floor)*p+uniform_floor/len(p)
    cat_prob=soft(overall)
    cat_weights=dict(zip(S.CATEGORIES,[float(j) for j in cat_prob]))
    lev_weights={c:dict(zip(levels,[float(v) for v in soft([risk[(c,l)] for l in levels])])) for c in S.CATEGORIES}
    table=pd.DataFrame([dict(category=c,level=l,group_balanced_error=risk[(c,l)],
        n_scenario_rows=counts[(c,l)],category_probability=cat_weights[c],level_probability=lev_weights[c][l])
        for c in S.CATEGORIES for l in levels])
    return cat_weights,lev_weights,table


class CurriculumAudioDataset(S.AudioDataset):
    def __init__(self,records,cfg,regime,cat_weights=None,level_weights=None):
        super().__init__(records,cfg,train=True,regime=regime,
          adaptive_weights=cat_weights if regime=='adaptive' else None)
        self.cat_weights=cat_weights
        self.level_weights=level_weights
    def __getitem__(self,i):
        r=self.records.iloc[i]
        seed=S.deterministic_seed(self.cfg.seed,self.epoch,i,r.relative_path)
        wav=S.read_fixed_segment(r.path,self.cfg.sample_rate,self.cfg.seconds,
            start_sec=r.get('segment_start_sec',None),seed=seed,train=True)
        if self.regime!='static':
            rng=np.random.default_rng(seed+123)
            if rng.random()<self.cfg.augmentation_probability:
                categories=S.CATEGORIES
                p=np.array([self.cat_weights[c] for c in categories]) if self.regime=='adaptive' else None
                cat=str(rng.choice(categories,p=p))
                levels=list(S.available_levels(self.cfg.scenario_profile))
                probs=(np.array([self.level_weights[cat][l] for l in levels])
                       if self.regime=='adaptive' else None)
                lv=int(rng.choice(levels,p=probs))
                val=S.scenario_severity(cat,lv,self.cfg.scenario_profile)
                wav=S.transform(wav,cat,lv,self.cfg.sample_rate,seed,severity_override=val)
        return np.asarray(wav,dtype='float32'),int(r.label)


def _model_predictions_on_adaptation(model,name,regime,df,cfg,max_per_class,seed):
    subset=_unique_adaptation_subset(df[df.split=='adaptation'],max_per_class,seed)
    # Original split tag stays adaptation, hidden_test and validation are never accessed.
    yy,ss=S.get_validation_scores(model,name,subset,cfg)
    thr=S.best_threshold(yy,ss)
    p=S.run_scenario_predictions(subset,model,name,regime,cfg,threshold=thr,split='adaptation')
    return float(thr),p


def train_curriculum_model(splits,cfg,name,regime,seed,epochs,update_every=2,
                           initial_policy=None,adapt_max_per_class=24):
    """Train fixed epochs, identical updates/augmentation count across regimes.

    Adaptive distribution changes ONLY using `adaptation` predictions. Does not
    touch validation nor hidden_test for threshold selection or policy updates.
    """
    import torch
    from torch import nn
    from torch.utils.data import DataLoader
    out=_out(cfg);mdir=out/'models';mdir.mkdir(exist_ok=True)
    if name not in ('spectrogram_cnn','raw_wave_cnn','mfcc_svm','aasist','rawnet2','wavlm'):
        raise ValueError(name)
    tr=splits[splits.split=='train'].reset_index(drop=True)
    adaptation=splits[splits.split=='adaptation'].reset_index(drop=True)
    if tr.label.nunique()!=2 or adaptation.label.nunique()!=2:
        raise ValueError('Train and adaptation must each contain two classes')
    trial_cfg=replace(cfg,seed=int(seed))
    S.seed_everything(seed+S.deterministic_seed(name)%1000)
    model=S.create_model(name,trial_cfg)
    equal={c:.25 for c in S.CATEGORIES};lev={c:{l:.2 for l in range(1,6)} for c in S.CATEGORIES}
    cat_weights,lv_weights=(initial_policy if initial_policy else (equal,lev))
    if regime!='adaptive':cat_weights,lv_weights=equal,lev
    ds=CurriculumAudioDataset(tr,trial_cfg,regime,cat_weights,lv_weights)
    logs=[]; policy_history=[]; draws=0
    if name=='mfcc_svm':
        all_x=[];all_y=[]
    else:
        device='cuda' if torch.cuda.is_available() else 'cpu'
        model=model.to(device)
        opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=trial_cfg.lr)
        pos=torch.tensor([int((tr.label==0).sum())/max(1,int((tr.label==1).sum()))],device=device,dtype=torch.float32)
        loss_fn=nn.BCEWithLogitsLoss(pos_weight=pos)
    for ep in range(int(epochs)):
        ds.epoch=ep
        ds.cat_weights=cat_weights;ds.level_weights=lv_weights
        if name=='mfcc_svm':
            for i in range(len(ds)):
                wav,label=ds[i];all_x.append(S.mfcc_features(wav,trial_cfg.sample_rate));all_y.append(label)
            model.fit(np.stack(all_x),np.asarray(all_y))
            loss=np.nan
        else:
            # Reset same deterministic shuffle epoch per regime, seed and model.
            g=torch.Generator();g.manual_seed(seed+ep*101)
            loader=DataLoader(ds,batch_size=trial_cfg.batch_size,shuffle=True,
                              num_workers=0,collate_fn=S._batch_tensors,generator=g)
            model.train();losses=[]
            for a,b in loader:
                a=a.to(device);b=b.to(device)
                opt.zero_grad(set_to_none=True)
                logits=model(a).view(-1)
                L=loss_fn(logits,b)
                if not torch.isfinite(L):raise FloatingPointError('NaN train loss')
                L.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),5.);opt.step()
                losses.append(float(L.detach().cpu()))
            loss=float(np.mean(losses))
        draws+=len(ds)
        logs.append(dict(model=name,regime=regime,seed=seed,epoch=ep+1,
                         n_training_draws=len(ds),loss=loss))
        if regime=='adaptive' and ep+1<epochs and ((ep+1)%max(1,int(update_every))==0):
            threshold,pred=_model_predictions_on_adaptation(model,name,regime,splits,trial_cfg,adapt_max_per_class,seed)
            new_cat,new_levels,table=_difficulty_policy(pred)
            # Momentum stabilizes small adaptation sets; fixed independent fold.
            cat_weights={c:.5*cat_weights[c]+.5*new_cat[c] for c in S.CATEGORIES}
            lv_weights={c:{lv:.5*lv_weights[c][lv]+.5*new_levels[c][lv] for lv in range(1,6)} for c in S.CATEGORIES}
            for r in table.itertuples():
                policy_history.append(dict(model=name,seed=seed,epoch=ep+1,
                    category=r.category,level=r.level,risk=r.group_balanced_error,
                    n_scenario_rows=r.n_scenario_rows,category_weight=cat_weights[r.category],
                    level_weight=lv_weights[r.category][r.level]))
        print(f'  {name} {regime} seed={seed} epoch {ep+1}/{epochs}: train loss {loss:.4f}' if np.isfinite(loss)
              else f'  {name} {regime} seed={seed} epoch {ep+1}/{epochs}: MFCC-SVM fitted')
    threshold,adapt_preds=_model_predictions_on_adaptation(model,name,regime,splits,trial_cfg,adapt_max_per_class,seed)
    metrics=S.metric_row(adapt_preds.loc[adapt_preds.category=='baseline','label'],
                         adapt_preds.loc[adapt_preds.category=='baseline','score'],threshold)
    sub=mdir/f'{name}_{regime}_seed{seed}'
    if name=='mfcc_svm':
        import joblib
        joblib.dump(model,str(sub)+'.joblib')
    else:torch.save(model.state_dict(),str(sub)+'.pt')
    meta=dict(model=name,regime=regime,seed=int(seed),epochs=int(epochs),
        threshold=float(threshold),adaptation_metrics=metrics,training_draws=int(draws),
        augmentation_probability=(0. if regime=='static' else float(cfg.augmentation_probability)),
        final_category_weights=cat_weights,final_level_weights=lv_weights,
        caveat='Threshold chosen ONLY on adaptation. Validation untouched until all models are trained.',
        model_path=str(sub)+('.joblib' if name=='mfcc_svm' else '.pt'))
    S.save_json(str(sub)+'_meta.json',meta)
    pd.DataFrame(logs).to_csv(str(sub)+'_train_history.csv',index=False)
    if policy_history:pd.DataFrame(policy_history).to_csv(str(sub)+'_policy_history.csv',index=False)
    return model,meta,adapt_preds


def evaluate_validation_only(splits,cfg,trained,seed):
    """Read validation only. Do not modify policy, checkpoint, threshold, hyperparams."""
    result=[]
    for (model_name,regime), (model,meta) in trained.items():
        print('VAL',model_name,regime,seed)
        p=S.run_scenario_predictions(splits,model,model_name,regime,cfg,
                            threshold=meta['threshold'],split='validation')
        p['seed']=int(seed)
        result.append(p)
    return pd.concat(result,ignore_index=True)


def evaluate_all_seeds(splits,cfg,models=('spectrogram_cnn','raw_wave_cnn','mfcc_svm'),
                      seeds=(42,123,777),epochs=12,update_every=3,adapt_max_per_class=24,
                      require_preflight=True):
    out=_out(cfg)
    if splits.groupby('group_id').split.nunique().gt(1).any():raise AssertionError('Group leakage')
    if require_preflight and not set(PARTITIONS).issubset(splits.split.unique()):
        raise ValueError('Expected train/adaptation/validation/hidden_test')
    if cfg.scenario_profile!='v2_5level':raise ValueError('Required 21-scenario profile')
    all_preds=[];adapt_reports=[];all_metas=[]
    for seed in seeds:
        trial_cfg=replace(cfg,seed=int(seed),out=str(out))
        trained={}
        for name in models:
            e1,meta,p1=train_curriculum_model(splits,trial_cfg,name,'static',int(seed),epochs,
                  update_every,adapt_max_per_class=adapt_max_per_class)
            trained[(name,'static')]=(e1,meta);all_metas.append(meta)
            policy=_difficulty_policy(p1)[:2]  # ONLY adaptation risk, no validation use
            e2,meta,p2=train_curriculum_model(splits,trial_cfg,name,'random',int(seed),epochs,
                  update_every,adapt_max_per_class=adapt_max_per_class)
            trained[(name,'random')]=(e2,meta);all_metas.append(meta)
            e3,meta,p3=train_curriculum_model(splits,trial_cfg,name,'adaptive',int(seed),epochs,
                  update_every,initial_policy=policy,adapt_max_per_class=adapt_max_per_class)
            trained[(name,'adaptive')]=(e3,meta);all_metas.append(meta)
            adapt_reports.extend([p1.assign(seed=seed),p2.assign(seed=seed),p3.assign(seed=seed)])
        val=evaluate_validation_only(splits,trial_cfg,trained,int(seed))
        all_preds.append(val)
        val.to_csv(out/f'V5_04_VALIDATION_PREDICTIONS_SEED_{seed}.csv',index=False)
    preds=pd.concat(all_preds,ignore_index=True)
    adapts=pd.concat(adapt_reports,ignore_index=True)
    preds.to_csv(out/'V5_04_VALIDATION_PREDICTIONS_ALL.csv',index=False)
    adapts.to_csv(out/'V5_03_ADAPTATION_PREDICTIONS_ALL.csv',index=False)
    pd.DataFrame(all_metas).drop(columns=['final_level_weights'],errors='ignore').to_csv(
        out/'V5_03_MODEL_RUNS.csv',index=False)
    return preds,adapts


def metric_tables(preds,cfg):
    """Per-seed scenario metrics and per-seed ASRI; never pool repeats as independent audio."""
    o=_out(cfg);sm=[];ari=[]
    for seed,g in preds.groupby('seed'):
        scores,asri=S.summarize_predictions(g,cfg,report_tag=f'v5_seed_{seed}')
        scores['seed']=int(seed);asri['seed']=int(seed)
        sm.append(scores);ari.append(asri)
    metrics=pd.concat(sm,ignore_index=True);table=pd.concat(ari,ignore_index=True)
    metrics.to_csv(o/'V5_05_SCENARIO_METRICS.csv',index=False)
    table.to_csv(o/'V5_05_ASRI_BY_SEED.csv',index=False)
    # Paired comparison: each seed/model E3 vs E2; mean and variation across seeds.
    pv=table.pivot(index=['model','seed'],columns='regime',values='asri').reset_index()
    pv['delta_adaptive_minus_random']=pv['adaptive']-pv['random']
    pv['delta_adaptive_minus_static']=pv['adaptive']-pv['static']
    pv.to_csv(o/'V5_05_PAIRED_SEED_DELTAS.csv',index=False)
    stats=pv.groupby('model').agg(n_seeds=('seed','nunique'),
       mean_delta_E3_E2=('delta_adaptive_minus_random','mean'),
       std_delta_E3_E2=('delta_adaptive_minus_random','std'),
       n_seeds_E3_wins=('delta_adaptive_minus_random',lambda x:int(sum(x>0)))).reset_index()
    stats.to_csv(o/'V5_05_DELTA_SUMMARY.csv',index=False)
    return metrics,table,pv,stats


def generate_tts_job_queue(parents,cfg,max_real=400,seed=42):
    """Export matched text synthesis plan. Does not silently call any vendor/network.

    The queue provides original transcript, parent, original speaker and locked fold.
    Human transcript is NOT verified unless its provenance says so. Edit/review
    before optional Piper generation. TTS files do not become real independent voices.
    """
    out=_out(cfg)
    x=parents[(parents.label.astype(int)==0)&parents.transcription.fillna('').astype(str).str.strip().ne('')].copy()
    x['text_normalized']=x.transcription.astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
    x=x[x.text_normalized.str.len().between(20,350)]
    x=x.drop_duplicates(subset=['source_record_id','text_normalized','split'])
    # Hold each exact normalized transcript in ONE fold to prevent text leakage.
    x['text_key']=x.text_normalized.str.casefold().str.replace(r'\W+','',regex=True)
    x['split_priority']=x.split.map({'train':0,'adaptation':1,'validation':2,'hidden_test':3}).fillna(9)
    x=x.sort_values(['split_priority','relative_path']).drop_duplicates('text_key',keep='first')
    # Drop unsafe ambiguous source attribution.
    x=x[x.source.isin(['CommonVoice','FLEURS','AUA_own'])]
    rng=np.random.default_rng(seed)
    x=x.iloc[rng.permutation(len(x))].head(max_real)
    q=x[['path','relative_path','source','source_record_id','speaker_id','group_id','split','text_normalized']].copy()
    q['source_license']=x.get('license',pd.Series('',index=x.index)).astype(str).values
    q['source_approval_status']=x.get('approval_status',pd.Series('',index=x.index)).astype(str).values
    q['pair_id']=q.apply(lambda r:'PAIR_'+hashlib.sha256((r['relative_path']+'|'+r['text_normalized']).encode()).hexdigest()[:15],axis=1)
    q['tts_status']='PENDING_REVIEW'
    q['tts_model_id']=''
    q['tts_voice_id']=''
    q['generated_path']=''
    queue_path=out/'V5_00_TTS_GENERATION_QUEUE.csv'
    if queue_path.exists():
        saved=pd.read_csv(queue_path,dtype=str,keep_default_na=False)
        if set(saved.pair_id)!=set(q.pair_id):
            raise RuntimeError('TTS queue parents/transcripts have changed. Preserve existing approvals; '
                               'start a new experiment directory and review the changes.')
        return saved
    q.to_csv(queue_path,index=False)
    q.groupby('split',dropna=False).size().rename('queued_texts').reset_index().to_csv(
        out/'V5_00_TTS_QUEUE_SPLIT_COUNTS.csv',index=False)
    S.save_json(out/'V5_00_TTS_QUEUE_INFO.json',dict(rows=len(q),
      warning='User must review text and licenses, assign voice models and keep voice identities disjoint between locked splits. No audio is generated by this function.'))
    return q


def synthesize_piper_reviewed(queue_csv,cfg,model_by_voice,limit=None):
    """Optional local TTS using preinstalled Piper CLI and explicitly supplied voice models.

    Input CSV must have tts_status=APPROVED, tts_voice_id, tts_model_id,
    and rows with valid human transcripts and consent/license. Each model_by_voice
    maps tts_voice_id -> local .onnx model path. No internet calls or fake claims.
    """
    out=_out(cfg)
    from shutil import which
    if not which('piper'):raise FileNotFoundError('piper CLI not found in PATH; TTS step is OPTIONAL')
    q=pd.read_csv(queue_csv,dtype=str,keep_default_na=False)
    required=['tts_status','tts_voice_id','tts_model_id','pair_id','text_normalized','split']
    missing=[c for c in required if c not in q.columns]
    if missing:raise ValueError('Missing columns: '+','.join(missing))
    approved=q[q.tts_status.eq('APPROVED')].copy()
    if limit:approved=approved.head(limit)
    if approved.empty:raise ValueError('No APPROVED TTS jobs')
    # A global voice cannot appear in two data partitions under a voice-disjoint protocol.
    repeats=approved.groupby('tts_voice_id').split.nunique()
    if repeats.gt(1).any():raise ValueError('Same TTS voice assigned to multiple folds: '+str(repeats[repeats>1].index.tolist()))
    # Distinct voice IDs MUST NOT point at the same ONNX model (aliases are not new voices).
    resolved={}
    model_hashes={}
    for voice in approved.tts_voice_id.unique():
        path=Path(model_by_voice.get(voice,'')).expanduser()
        if not path.is_file() or path.suffix.lower()!='.onnx':
            raise FileNotFoundError(f'Piper .onnx model missing for {voice}: {path}')
        resolved[voice]=path
        model_hashes[voice]=_sha256_file(path)
    inverted=defaultdict(set)
    for voice, digest in model_hashes.items():inverted[digest].add(voice)
    aliases={digest:sorted(names) for digest,names in inverted.items() if len(names)>1}
    if aliases:
        raise ValueError('Same Piper ONNX file is aliased as multiple independent TTS voices: '+str(aliases))
    outdir=out/'paired_tts_generated';outdir.mkdir(exist_ok=True)
    receipts=[]
    for r in approved.itertuples():
        path=resolved[r.tts_voice_id]
        dest=outdir/f'{r.pair_id}_{S.safe_key(r.tts_voice_id)}.wav'
        proc=subprocess.run(['piper','--model',str(path),'--output_file',str(dest)],
                  input=r.text_normalized,capture_output=True,text=True,timeout=240)
        if proc.returncode or not dest.is_file():
            raise RuntimeError(f'Piper failed for {r.pair_id}: {proc.stderr[-350:]}')
        digest=hashlib.sha256(dest.read_bytes()).hexdigest()
        receipts.append(dict(pair_id=r.pair_id,original_path=r.path,generated_path=str(dest),
            original_group_id=r.group_id,split=r.split,source=r.source,model_id=r.tts_model_id,
            voice_id=r.tts_voice_id,text=r.text_normalized,sha256=digest,
            tts_model_file_sha256=model_hashes[r.tts_voice_id]))
    pd.DataFrame(receipts).to_csv(out/'V5_00_PIPER_GENERATION_RECEIPTS.csv',index=False)
    return pd.DataFrame(receipts)


def source_and_split_diagnostics(splits,cfg):
    out=_out(cfg)
    d=splits.copy()
    a=d.groupby(['split','source','label']).agg(segments=('path','size'),
               independent_groups=('group_id','nunique'),parents=('parent_file_id','nunique')).reset_index()
    a.to_csv(out/'V5_02_SPLIT_SOURCE_CLASS.csv',index=False)
    risks=[]
    for part in ['adaptation','validation','hidden_test']:
        sub=d[d.split==part]
        n=int(sub[sub.label==1].group_id.nunique())
        risks.append(dict(partition=part,independent_fake_groups=n,
          adequate_for_generalization=bool(n>=3)))
    pd.DataFrame(risks).to_csv(out/'V5_02_FAKE_GROUP_RISKS.csv',index=False)
    print('IMPORTANT: A class=source confound remains until matched TTS audio is actually generated and included.')
    return a,pd.DataFrame(risks)


def integrate_tts_pairs(original_parents,receipts_csv,cfg,min_pairs_per_split=2):
    """Replace current experimental parents with SAME-TEXT real/fake pairs.

    This is opt-in; creates a new dataset manifest only in result directory.
    Preserves the original parent split and the verified human speaker ID.
    Voices must be split-disjoint; each pair has exactly one human and one TTS file.
    """
    out=_out(cfg)
    receipts=pd.read_csv(receipts_csv,dtype=str,keep_default_na=False)
    needed={'pair_id','original_path','generated_path','split','voice_id','model_id','sha256','text'}
    if needed-set(receipts.columns):raise ValueError(f'Missing receipts fields: {sorted(needed-set(receipts.columns))}')
    receipts=receipts.drop_duplicates('pair_id',keep=False).copy()
    if receipts.empty:raise ValueError('No uniquely identifiable paired TTS examples')
    if receipts.assign(_voice_key=receipts.voice_id.str.casefold().str.strip()).groupby('_voice_key').split.nunique().gt(1).any():
        raise AssertionError('TTS voice crosses train/validation/test; assign distinct voices per partition')
    if 'tts_model_file_sha256' in receipts.columns:
        dig=receipts[receipts.tts_model_file_sha256.fillna('').astype(str).str.strip().ne('')].copy()
        if not dig.empty:
            aliases=dig.groupby('tts_model_file_sha256').voice_id.nunique()
            if aliases.gt(1).any():
                raise AssertionError('One TTS model audio hash is assigned as multiple independent voices')
    orig=original_parents.copy()
    by_path=orig.drop_duplicates('path').set_index('path',drop=False)
    rows=[]
    for r in receipts.itertuples():
        if r.original_path not in by_path.index:
            raise ValueError(f'Original recording not in locked parent split: {r.original_path}')
        rr=by_path.loc[r.original_path]
        if isinstance(rr,pd.DataFrame):raise ValueError(f'Ambiguous original record: {r.original_path}')
        if int(rr.label)!=0 or str(rr.split)!=str(r.split):
            raise ValueError(f'Incorrect matched-pair label or split: {r.pair_id}')
        dest=Path(r.generated_path).resolve()
        if not dest.is_file() or out!=dest.parent and out not in dest.parents:
            raise ValueError('Generated WAV missing or outside results directory')
        sha=hashlib.sha256(dest.read_bytes()).hexdigest()
        if sha.lower()!=r.sha256.lower():raise ValueError('Generated TTS SHA mismatch')
        if not str(r.voice_id).strip() or not str(r.model_id).strip():
            raise ValueError('TTS model and voice IDs required')
        # A real matched pair must contain the SAME utterance, not just a shared ID.
        import re
        def _norm_text(t):
            return re.sub(r'[^\w]+','',str(t).casefold(),flags=re.UNICODE)
        original_transcript=str(rr.get('transcription','')).strip()
        if not original_transcript or _norm_text(original_transcript)!=_norm_text(r.text):
            raise ValueError('TTS receipt text does not match original AUA transcript: '+str(r.pair_id))
        h=rr.copy();h['pair_id']=r.pair_id;h['paired_source']=f'PAIR::{rr.source}'
        h['source_original']=rr.source;h['source']=h['paired_source'];h['transcription']=r.text
        h['input_text_id']=r.pair_id
        f=h.copy();f['label']=1;f['path']=str(dest)
        f['relative_path']=f'generated_pairs/{dest.name}'
        f['group_id']=f'VOICE::{r.model_id}::{r.voice_id}'
        f['identity_status']='synthetic_voice_confirmed';f['split_role']='eligible'
        f['group_confidence']='metadata_voice'
        f['speaker_id']='';f['voice_id']=r.voice_id;f['generator']=r.model_id
        f['source_record_id']='GENERATED_TTS_'+r.pair_id
        f['source_sha256']=sha;f['working_sha256']=sha
        f['duration_sec']=S.audio_info_seconds(dest)
        if not np.isfinite(f['duration_sec']) or f['duration_sec']<cfg.min_seconds:
            raise ValueError('Paired synthesized audio invalid/short')
        f['pair_id']=r.pair_id
        rows.extend([h,f])
    paired=pd.DataFrame(rows).reset_index(drop=True)
    paired['text_key']=paired.transcription.fillna('').astype(str).str.casefold().str.replace(r'\W+','',regex=True)
    if (paired[paired.text_key.ne('')].groupby('text_key').split.nunique()>1).any():
        raise AssertionError('Matched text is reused across data folds')
    if ((paired.groupby('pair_id').label.nunique()!=2) | (paired.groupby('pair_id').size()!=2)).any():
        raise AssertionError('Each pair must contain exactly one REAL and one FAKE')
    if paired.groupby('group_id').split.nunique().gt(1).any():
        raise AssertionError('Paired voices or speakers cross splits')
    for split in ['train','validation','hidden_test']:
        if paired[paired.split==split].pair_id.nunique()<min_pairs_per_split:
            raise ValueError(f'Not enough matched pairs in {split}: requires {min_pairs_per_split}')
    paired.to_csv(out/'V5_01_MATCHED_PAIRS_PARENTS.csv',index=False)
    S.save_json(out/'V5_01_MATCHED_PAIR_REPORT.json',dict(pairs=len(paired)//2,
        train=int(paired[paired.split=='train'].pair_id.nunique()),
        validation=int(paired[paired.split=='validation'].pair_id.nunique()),
        hidden_test=int(paired[paired.split=='hidden_test'].pair_id.nunique()),
        caution='Same-text matching does NOT match capture channel, codec, room or mic.'))
    return paired


def balance_pairs(paired_segments,cfg,seed=42):
    """Preserve real/fake paired structure and parent group independence.

    If one class has more extracted windows, take the same random number per pair.
    This is not an independence guarantee for windows from the same parent.
    """
    out=_out(cfg);rng=np.random.default_rng(seed);kept=[];lost=[]
    for (split,pairid),g in paired_segments.groupby(['split','pair_id'],sort=True):
        a=g[g.label.astype(int)==0];b=g[g.label.astype(int)==1]
        n=min(len(a),len(b))
        if n<=0:lost.extend(g.index.tolist());continue
        kept.extend(a.iloc[rng.choice(len(a),n,replace=False)].index.tolist())
        kept.extend(b.iloc[rng.choice(len(b),n,replace=False)].index.tolist())
        lost.extend(g.index.difference(kept).tolist())
    res=paired_segments.loc[sorted(set(kept))].reset_index(drop=True)
    assert res.groupby(['split','pair_id','label']).size().unstack(fill_value=0).pipe(lambda d:(d[0]==d[1]).all())
    assert res.groupby('group_id').split.nunique().max()==1
    res.to_csv(out/'V5_02_PAIRED_SEGMENTS.csv',index=False)
    return res


def stratified_voice_bootstrap(preds,cfg,B=300,seed=42,min_groups_per_class=3):
    """Paired E3−E2 ASRI cluster bootstrap, stratified by TRUE class.

    The sampling unit is an entire independent speaker/voice group. The same sampled
    groups evaluate both regimes and all 20 acoustic scenarios. A bootstrap is NOT
    reported when there are fewer than `min_groups_per_class` groups in either class.
    Evaluates seeds separately; this does not treat random seeds as audio examples.
    """
    out=_out(cfg);rng=np.random.default_rng(seed);results=[]
    levels=list(S.available_levels(cfg.scenario_profile));cats=S.CATEGORIES
    keys=[(c,l) for c in cats for l in levels];K=len(keys)
    for (model,seed_value),g in preds.groupby(['model','seed']):
        g=g[g.regime.isin(['random','adaptive'])&g.category.isin(cats)].copy()
        info={lab:sorted(g[g.label==lab].group_id.unique()) for lab in (0,1)}
        common=dict(model=model,seed=int(seed_value),real_groups=len(info[0]),fake_groups=len(info[1]))
        if min(len(info[0]),len(info[1]))<min_groups_per_class:
            results.append(dict(**common,status='INSUFFICIENT_INDEPENDENT_VOICES',
                   delta_asri=np.nan,ci95_lower=np.nan,ci95_upper=np.nan,n_valid=0))
            continue
        arrays={}
        for label in (0,1):
            keyrows=[]
            for group in info[label]:
                one=g[(g.label==label)&(g.group_id==group)]
                vals=[]
                for cat,lv in keys:
                    cell=one[(one.category==cat)&(one.level==lv)]
                    dat=[]
                    for reg in ('random','adaptive'):
                        part=cell[cell.regime==reg]
                        dat.extend([float((part.pred==label).sum()),float(len(part))])
                    vals.append(dat)
                keyrows.append(vals)
            arrays[label]=np.asarray(keyrows,float) # [group,20,4] correct,count × regimes
        def calc(ix0,ix1):
            z0=arrays[0][ix0].sum(axis=0);z1=arrays[1][ix1].sum(axis=0)
            if (z0[:,1]==0).any() or (z0[:,3]==0).any() or (z1[:,1]==0).any() or (z1[:,3]==0).any():
                return np.nan
            ba_r=.5*(z0[:,0]/z0[:,1]+z1[:,0]/z1[:,1])
            ba_a=.5*(z0[:,2]/z0[:,3]+z1[:,2]/z1[:,3])
            return float(np.mean(np.maximum(0,2*ba_a-1)-np.maximum(0,2*ba_r-1)))
        obs=calc(np.arange(len(info[0])),np.arange(len(info[1])))
        sims=[]
        for _ in range(int(B)):
            ii=rng.integers(0,len(info[0]),len(info[0]))
            jj=rng.integers(0,len(info[1]),len(info[1]))
            v=calc(ii,jj)
            if np.isfinite(v):sims.append(v)
        result=dict(**common,status='OK' if len(sims)>=.9*B else 'INCOMPLETE',
            delta_asri=obs,ci95_lower=float(np.quantile(sims,.025)) if sims else np.nan,
            ci95_upper=float(np.quantile(sims,.975)) if sims else np.nan,n_valid=len(sims))
        results.append(result)
    report=pd.DataFrame(results)
    report.to_csv(out/'V5_05_BOOTSTRAP_STRATIFIED_VOICES.csv',index=False)
    return report


def research_readiness(splits,cfg,mode='pilot_existing',min_voices=3,min_seeds=3,trained_seeds=()):
    """Conservative report. A positive score never overrides missing provenance."""
    out=_out(cfg)
    rows=[]
    for part in PARTITIONS:
        x=splits[splits.split.eq(part)]
        for label in (0,1):
            z=x[x.label.astype(int).eq(label)]
            rows.append(dict(split=part,label=label,segments=len(z),parents=z.parent_file_id.nunique(),
                             independent_groups=z.group_id.nunique(),voices=z.voice_id.fillna('').astype(str).loc[lambda a:a.str.strip().ne('')].nunique() if 'voice_id' in z else 0))
    info=pd.DataFrame(rows)
    info.to_csv(out/'V5_READINESS_CLASS_GROUPS.csv',index=False)
    matched=(mode=='matched_tts')
    groups_ok=all(int(info.loc[(info.split==part)&(info.label==label),'independent_groups'].iloc[0])>=min_voices
                  for part in ('adaptation','validation','hidden_test') for label in (0,1))
    text_ok=True
    if matched:
        if 'text_key' not in splits or not splits.text_key.fillna('').astype(str).str.strip().ne('').all():
            text_ok=False
        else:
            text_ok=not splits.groupby('text_key').split.nunique().gt(1).any()
    voice_ok=True
    if matched and 'voice_id' in splits:
        v=splits[splits.label.astype(int).eq(1)].copy()
        v['_k']=v.voice_id.fillna('').astype(str).str.casefold().str.strip()
        voice_ok=v._k.ne('').all() and not v.groupby('_k').split.nunique().gt(1).any()
    leak_ok=not splits.groupby('group_id').split.nunique().gt(1).any()
    if matched:leak_ok=leak_ok and not splits.groupby('pair_id').split.nunique().gt(1).any()
    checks=dict(mode=mode,matched_tts=matched,independent_groups_ready=bool(groups_ok),
        independent_tts_voices_across_folds=bool(voice_ok),no_cross_split_text=bool(text_ok),
        no_cross_split_group_or_pair=bool(leak_ok),n_seeds=len(trained_seeds),min_seeds=int(min_seeds),
        hidden_test_opened=False,expert_assessment_included=False)
    checks['candidate_for_strict_review']=bool(matched and groups_ok and voice_ok and text_ok and leak_ok
        and len(trained_seeds)>=min_seeds)
    checks['warning']='Candidate status is NOT proof of channel matching, statistical significance, or causal benefit.'
    S.save_json(out/'V5_RESEARCH_READINESS.json',checks)
    return info,checks


def _sha256_file(path):
    digest=hashlib.sha256()
    with open(path,'rb') as stream:
        for piece in iter(lambda:stream.read(2**20),b''):
            digest.update(piece)
    return digest.hexdigest()


def freeze_for_hidden_once(splits,cfg,models,seeds,epochs,mode='matched_tts',min_voice_groups=3):
    """Lock V5 model checkpoints and pre-selected thresholds BEFORE hidden audio is read.

    Requires a sufficient paired-text design. Pilot existing-AUA cannot claim finality.
    Existing protocol in the same output directory must match byte-for-byte.
    """
    out=_out(cfg)
    info,readiness=research_readiness(splits,cfg,mode=mode,min_voices=min_voice_groups,
                                      trained_seeds=seeds)
    if mode!='matched_tts' or not readiness['candidate_for_strict_review']:
        raise RuntimeError('Not ready for dissertation hidden test: see V5_RESEARCH_READINESS.json')
    if len(seeds)<3 or len(seeds)>5:
        raise ValueError('At least 3 and at most 5 declared seeds required')
    checkpoint_info=[]
    for seed in seeds:
        for model in models:
            for regime in ('static','random','adaptive'):
                stem=f'{model}_{regime}_seed{seed}'
                meta_path=out/'models'/f'{stem}_meta.json'
                if not meta_path.is_file():raise FileNotFoundError(meta_path)
                meta=json.loads(meta_path.read_text(encoding='utf-8'))
                if meta.get('epochs')!=epochs or meta.get('seed')!=seed:
                    raise RuntimeError('Mismatch between declared and trained epochs/seeds: '+stem)
                cp=Path(meta['model_path']).resolve()
                if not cp.is_file() or not cp.is_relative_to(out):
                    raise FileNotFoundError(f'Model checkpoint outside experiment folder: {cp}')
                checkpoint_info.append(dict(model=model,regime=regime,seed=seed,
                    model_path=str(cp),sha256=_sha256_file(cp),threshold=float(meta['threshold']),
                    epochs=int(epochs),model_meta_sha256=_sha256_file(meta_path)))
    val=out/'V5_04_VALIDATION_PREDICTIONS_ALL.csv'
    if not val.is_file():raise FileNotFoundError('Validation predictions missing: '+str(val))
    conf=dict(protocol_version='SAADE_V5_FINAL',split_sha256=S.frozen_hash(splits),
        checkpoints=checkpoint_info,mode=mode,models=list(models),seeds=list(seeds),
        epochs=int(epochs),scenario_profile=cfg.scenario_profile,
        validation_predictions_sha256=_sha256_file(val),
        channel_matching_not_guaranteed=True,
        notes='Thresholds and E3 policies were selected using adaptation, NOT hidden test.')
    path=out/'V5_HIDDEN_FROZEN_PROTOCOL.json'
    if path.exists():
        old=json.loads(path.read_text(encoding='utf-8'))
        if old!=conf:
            raise RuntimeError('Already frozen protocol differs; use a NEW experiment directory')
    else:
        S.save_json(path,conf)
    return path


def run_hidden_once(splits,cfg,authorization=False):
    """Perform ONE hidden evaluation per locked checkpoint; no training or tuning.

    Authorization must be explicit; one-shot marker is created atomically BEFORE
    reading any hidden audio. The marker is not deleted after an interrupted run.
    """
    if not authorization:
        raise PermissionError('Set authorization=True ONLY after protocol and data acceptance')
    out=_out(cfg)
    pp=out/'V5_HIDDEN_FROZEN_PROTOCOL.json'
    if not pp.is_file():raise RuntimeError('Call freeze_for_hidden_once first')
    frozen=json.loads(pp.read_text(encoding='utf-8'))
    if frozen['split_sha256']!=S.frozen_hash(splits):
        raise RuntimeError('Locked split has changed')
    for entry in frozen['checkpoints']:
        p=Path(entry['model_path'])
        if not p.is_file() or _sha256_file(p)!=entry['sha256']:
            raise RuntimeError('Model checkpoint changed after freeze: '+str(p))
        meta=out/'models'/f"{entry['model']}_{entry['regime']}_seed{entry['seed']}_meta.json"
        if _sha256_file(meta)!=entry['model_meta_sha256']:
            raise RuntimeError('Model meta changed after freeze')
    if _sha256_file(out/'V5_04_VALIDATION_PREDICTIONS_ALL.csv') != frozen['validation_predictions_sha256']:
        raise RuntimeError('Validation results changed after freeze')
    marker=out/'V5_HIDDEN_TEST_STARTED.json'
    with marker.open('x',encoding='utf-8') as fh:
        json.dump({'protocol_sha256':_sha256_file(pp),'one_shot':True,
            'started_utc':pd.Timestamp.utcnow().isoformat()},fh,ensure_ascii=False,indent=2)
    from dataclasses import replace
    results=[]
    for item in frozen['checkpoints']:
        name=item['model']; path=Path(item['model_path'])
        if name=='mfcc_svm':
            import joblib
            model=joblib.load(path)
        else:
            import torch
            c=replace(cfg,seed=int(item['seed']))
            model=S.create_model(name,c)
            try:weights=torch.load(path,map_location='cpu',weights_only=True)
            except TypeError:weights=torch.load(path,map_location='cpu')
            model.load_state_dict(weights)
            model=model.to('cuda' if torch.cuda.is_available() else 'cpu')
        p=S.run_scenario_predictions(splits,model,name,item['regime'],cfg,
            threshold=float(item['threshold']),split='hidden_test')
        p['seed']=int(item['seed'])
        results.append(p)
    preds=pd.concat(results,ignore_index=True)
    preds.to_csv(out/'V5_07_HIDDEN_PREDICTIONS_ALL.csv',index=False)
    # Keep per-seed summaries independent.
    tab=[]
    for seed,g in preds.groupby('seed'):
        _,a=S.summarize_predictions(g,cfg,report_tag=f'hidden_seed_{seed}')
        a['seed']=int(seed);tab.append(a)
    summary=pd.concat(tab,ignore_index=True)
    summary.to_csv(out/'V5_07_HIDDEN_ASRI_BY_SEED.csv',index=False)
    return summary
'''
_module_name = 'saade_final_adaptive'
_mod = types.ModuleType(_module_name)
_mod.__file__ = '<SAADE_ALL_IN_ONE>/' + _module_name + '.py'
sys.modules[_module_name] = _mod
exec(compile(_module_source, _mod.__file__, 'exec'), _mod.__dict__)
print('Підключено: saade_final_adaptive')

In [ ]:
import saade_final_core as S
import saade_final_data as D
import saade_final_methods as M
import saade_final_visuals as V
import saade_final_adaptive as A

assert len(V.all_scenario_metadata(S.Config(scenario_profile='v2_5level'))) == 21
cfg_tts = S.Config(root=str(AUA), out=str(MATCHED_OUT), scenario_profile='v2_5level',
                   seed=42, seconds=8.0, min_seconds=2., augmentation_probability=.5,
                   batch_size=12, lr=.0005, epochs=EPOCHS, use_cache=True)
print('Модулі підключено з цього ноутбука. 21 акустичний сценарій доступний.')

## 2. Перевірка первинних метаданих і незалежності людських мовців

Використовуємо авторитетні таблиці AUA через виправлений адаптер. Вихідний `speaker_id` Common Voice та `source_record_id` зберігаються; аудіо з `05_splits` не додається повторно. **FLEURS поки не включаємо в matched-пари**, бо в цьому AUA не підтверджено незалежні ID мовців.

In [ ]:
catalog_tts = D.build_verified_catalog(cfg_tts)
print('У каталозі:', len(catalog_tts), '| придатних:', int(catalog_tts.usable.sum()))
display(pd.crosstab(catalog_tts.source, [catalog_tts.label, catalog_tts.usable], dropna=False))
print('DRSSU з підтвердженим джерелом:',
      int((catalog_tts.source.eq('DRSSU') & catalog_tts.primary_source_meta_found).sum()))
parents_tts, preflight_report = D.prepare_parent_splits(
    catalog_tts, cfg_tts, max_parent_files=MAX_PARENT_FILES,
    unknown_train_cap=UNKNOWN_TRAIN_CAP, policy='exploratory')
display(preflight_report)
display(D.data_readiness_report(catalog_tts, parents_tts, cfg_tts, policy='exploratory'))
print('Доступні незалежні групи:')
display(parents_tts.groupby(['split','label']).group_id.nunique().rename('n_groups').reset_index())
assert not parents_tts.groupby('group_id').split.nunique().gt(1).any()

## 3. Формуємо реальну чергу текстів Common Voice

Відбираються **тільки фактично наявні** файли з текстом, ліцензією CC0-1.0, підтвердженим джерелом і достовірним ID мовця. Кожен оригінальний запис має один `pair_id`; один і той самий нормалізований текст не може бути в кількох розділах. Всі рядки зберігаються в `V6_TTS_QUEUE.csv`.

Черга **не означає**, що синтетичне аудіо вже створено. Лише справжні WAV із перевіреним SHA-256 потрапляють у `V6_TTS_RECEIPTS.csv`.

In [ ]:
import unicodedata

def _norm_text_v6(value):
    s=unicodedata.normalize('NFC',str(value or '')).casefold()
    return re.sub(r'[^\w]+','',s,flags=re.UNICODE)

def make_v6_queue(catalog,out,max_utterances=600,seed=42):
    # Use ALL verified Common Voice records; do not downsample via old pilot parents.
    r=catalog.copy()
    r=r[(r.label.astype(int)==0)&r.source.eq('CommonVoice')&r.usable].copy()
    r=r[r.group_confidence.astype(str).eq('metadata_speaker_verified_basis')]
    r=r[r.approval_status.astype(str).str.casefold().eq('confirmed')]
    r=r[r.license.astype(str).str.upper().eq('CC0-1.0')]
    r['text_normalized']=r.transcription.fillna('').astype(str).str.replace(r'\s+',' ',regex=True).str.strip()
    r=r[r.text_normalized.str.len().between(20,200)]
    r=r[pd.to_numeric(r.duration_sec,errors='coerce').between(MIN_SOURCE_DURATION_SEC,MAX_SOURCE_DURATION_SEC)]
    r=r[r.path.map(lambda p:Path(p).is_file())]
    if VERIFY_REAL_WAV_SHA:
        def _verified(row):
            expected=str(row.get('working_sha256','')).lower().strip()
            if not re.fullmatch(r'[0-9a-f]{64}',expected):return False
            h=hashlib.sha256()
            with open(row.path,'rb') as stream:
                for chunk in iter(lambda:stream.read(1024*1024),b''):h.update(chunk)
            return h.hexdigest()==expected
        before=len(r)
        r=r[r.apply(_verified,axis=1)].copy()
        print('Working WAV SHA-256 перевірено:',len(r),'/',before)
    r['text_key']=r.text_normalized.map(_norm_text_v6)
    r=r[r.text_key.str.len().ge(12)]
    # Each human speaker appears ONCE, so no speaker connects different TTS voices.
    rng=np.random.default_rng(int(seed))
    r['_random']=rng.random(len(r))
    r=r.sort_values('_random').drop_duplicates('source_sha256').drop_duplicates('text_key').drop_duplicates('group_id')
    r=r.head(int(max_utterances)).copy().reset_index(drop=True)
    if len(r)<12:raise RuntimeError(f'Знайдено лише {len(r)} підтверджених REAL-записів, потрібно хоча б 12')
    order=rng.permutation(len(r))
    n_train=int(round(len(r)*.65));n_val=int(round(len(r)*.18))
    r['split']='hidden_test'
    r.loc[order[:n_train],'split']='train'
    r.loc[order[n_train:n_train+n_val],'split']='validation'
    if (r.groupby('split').size()<3).any() or r.split.nunique()!=3:
        raise ValueError('Недостатньо записів для трьох незалежних груп розподілу')
    r['parent_file_id']=r.relative_path
    r['pair_id']=r.apply(lambda x:'PAIR_'+hashlib.sha256((str(x.relative_path)+'|'+str(x.text_normalized)).encode()).hexdigest()[:16],axis=1)
    r['tts_model_id']='Piper (rhasspy/piper-voices)'
    r['tts_voice_id']=''
    for fold,vs in VOICE_ALLOCATION.items():
        ix=r.index[r.split.eq(fold)].tolist();rng.shuffle(ix)
        for j,i in enumerate(ix):r.loc[i,'tts_voice_id']=vs[j%len(vs)]
    r['tts_status']='APPROVED_FROM_CONFIRMED_CC0_METADATA'
    if r.groupby('group_id').split.nunique().gt(1).any() or r.groupby('text_key').split.nunique().gt(1).any():
        raise AssertionError('DATA LEAKAGE in matched candidate set')
    qcols=['pair_id','path','relative_path','source','source_record_id','source_sha256','working_sha256',
           'speaker_id','group_id','split','text_normalized','text_key','license','approval_status',
           'duration_sec','tts_model_id','tts_voice_id','tts_status']
    q=r[qcols].copy().sort_values(['split','pair_id']).reset_index(drop=True)
    target=out/'V6_TTS_QUEUE.csv'
    parent_target=out/'V6_LOCKED_REAL_PARENTS.csv'
    if target.exists():
        old=pd.read_csv(target,dtype=str,keep_default_na=False)
        if set(old.pair_id)!=set(q.pair_id):
            raise RuntimeError('Склад підкорпусу змінився. Для нового дослідження створіть новий OUTPUT, не перезаписуйте наявний.')
        if not parent_target.is_file():raise RuntimeError('Втрачені збережені REAL parents для існуючої черги')
        return old,pd.read_csv(parent_target,low_memory=False)
    q.to_csv(target,index=False)
    r.drop(columns=['_random'],errors='ignore').to_csv(parent_target,index=False)
    return q,r.drop(columns=['_random'],errors='ignore')

q,matched_parent_catalog=make_v6_queue(catalog_tts,MATCHED_OUT,max_utterances=TTS_MAX_UTTERANCES,seed=SEED_SELECTION)
print('REAL-текстів для генерації:',len(q))
display(q.groupby(['split','tts_voice_id']).agg(texts=('pair_id','nunique'),real_speakers=('speaker_id','nunique')).reset_index())
display(q[['pair_id','split','text_normalized','tts_voice_id']].head(10))
assert q.groupby('group_id').split.nunique().max()==1
assert q.groupby('text_key').split.nunique().max()==1
assert q.groupby('group_id').size().max()==1
print('Черга:',MATCHED_OUT/'V6_TTS_QUEUE.csv')

## 4. Реальні українські голоси Piper: перевірка та надійне отримання моделей

Голоси `lada`, `mykyta`, `oleksa`, `tetiana`, `ukrainian_tts` доступні в публічному репозиторії Piper. Код читає **справжній список файлів** у Hugging Face, а не вгадує назви ONNX, і фіксує SHA-256 та commit репозиторію. Якщо файли вже є локально, вкажіть їх у `LOCAL_PIPER_VOICES`.

⚠️ Це **п'ять голосів одного сімейства TTS**, а не п'ять незалежних генераторів. До публікації підкорпусу потрібно перевірити `MODEL_CARD` та умови ліцензування кожної голосової моделі. При відсутності Інтернету генерація неможлива без локальних ONNX.

**Виправлення v6.1:** конфігурації `.onnx.json` завантажуються без механізму HTTP/Brotli; вже отримані `.onnx` повторно не завантажуються. Якщо попередній запуск зупинився тут, достатньо перевиконати цю комірку й наступні.

In [ ]:
def _require_optional_tts_package(module, pip_name):
    if importlib.util.find_spec(module) is None:
        if AUTO_INSTALL_TTS_PACKAGES:
            subprocess.run([sys.executable,'-m','pip','install',pip_name],check=True)
        else:
            raise RuntimeError(f'Потрібен {pip_name}. У Jupyter виконайте: %pip install {pip_name}')

def _file_sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for chunk in iter(lambda:stream.read(2**20),b''):
            h.update(chunk)
    return h.hexdigest()

def _piper_cli():
    cli=shutil.which('piper')
    if cli:return [cli]
    if importlib.util.find_spec('piper') is not None:return [sys.executable,'-m','piper']
    if AUTO_INSTALL_TTS_PACKAGES:
        subprocess.run([sys.executable,'-m','pip','install','piper-tts'],check=True)
        cli=shutil.which('piper')
        return [cli] if cli else [sys.executable,'-m','piper']
    raise RuntimeError('Piper не знайдено. Виконайте %pip install piper-tts і перезапустіть ядро.')

def _piper_json_ok(path):
    """Валідний конфіг Piper, без HTML-помилок і порожніх/часткових завантажень."""
    path=Path(path)
    if not path.is_file() or path.stat().st_size < 80:
        return False
    try:
        data=json.loads(path.read_text(encoding='utf-8'))
        return isinstance(data,dict) and isinstance(data.get('audio'),dict) and bool(data.get('phoneme_type'))
    except (OSError, ValueError, UnicodeError):
        return False


def _download_hf_small_file(repo_id, revision, relative_path, root):
    """Завантаження JSON/README стандартним urllib, минаючи httpx2/Brotli.

    Записує атомарно. Використовує pinned revision. Ніколи не пише в AUA.
    """
    from urllib.parse import quote
    from urllib.request import Request, urlopen
    root=Path(root).resolve()
    dest=(root/relative_path).resolve()
    if not dest.is_relative_to(root):
        raise ValueError('Заборонений шлях за межами папки TTS моделей: '+relative_path)
    if dest.suffix == '.json' and _piper_json_ok(dest):
        return dest
    if dest.suffix != '.json' and dest.is_file() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True,exist_ok=True)
    url='https://huggingface.co/'+quote(repo_id,safe='/')+'/resolve/'+quote(revision,safe='')+'/'+quote(relative_path,safe='/')
    request=Request(url,headers={'User-Agent':'SAADE-AUA-v6.1/1.0','Accept-Encoding':'identity'})
    tmp=dest.with_name(dest.name+'.saade-download')
    try:
        with urlopen(request,timeout=120) as response, open(tmp,'wb') as output:
            # JSON і картки моделі — малі файли, не дозволяємо неконтрольований HTML/великі відповіді
            limit=2*1024*1024
            total=0
            while True:
                block=response.read(min(65536,limit+1-total))
                if not block:break
                total+=len(block)
                if total>limit:raise RuntimeError('Файл має неочікуваний великий розмір: '+relative_path)
                output.write(block)
        if relative_path.endswith('.json') and not _piper_json_ok(tmp):
            raise RuntimeError('Отриманий файл не є валідною Piper ONNX JSON-конфігурацією: '+relative_path)
        os.replace(tmp,dest)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise
    return dest


def setup_piper_models(voices):
    meta=[]; local={}
    local_root=MATCHED_OUT/'tts_models'
    local_root.mkdir(parents=True,exist_ok=True)
    needs=[name for name in voices if not (name in LOCAL_PIPER_VOICES and Path(LOCAL_PIPER_VOICES[name]).is_file())]
    if needs and not AUTO_DOWNLOAD_PIPER_VOICES:
        raise FileNotFoundError('Немає локальних ONNX для '+str(needs))
    if needs:
        _require_optional_tts_package('huggingface_hub','huggingface_hub')
        from huggingface_hub import HfApi, hf_hub_download
        api=HfApi()
        commit=api.model_info(TTS_REPO).sha
        listing=set(api.list_repo_files(TTS_REPO,revision=commit))
    else:
        commit='LOCAL'; listing=set()

    for name in sorted(set(voices)):
        if name in LOCAL_PIPER_VOICES and Path(LOCAL_PIPER_VOICES[name]).is_file():
            onnx=Path(LOCAL_PIPER_VOICES[name]).expanduser().resolve()
            cfg_path=Path(str(onnx)+'.json')
            model_rel='local:'+onnx.name
        else:
            options=[p for p in listing if p.startswith('uk/uk_UA/'+name+'/') and p.endswith('.onnx')]
            if not options:raise RuntimeError('Голос відсутній у HF: '+name)
            priorities=['medium','high','low','x_low']
            def rank(path):
                parts=path.split('/')
                quality=parts[3] if len(parts)>3 else ''
                return (priorities.index(quality) if quality in priorities else 99,path)
            model_rel=sorted(options,key=rank)[0]
            config_rel=model_rel+'.json'
            if config_rel not in listing:raise RuntimeError('Немає .onnx.json для '+model_rel)

            # Не завантажуємо .onnx повторно, коли він уже успішно існує.
            existing=local_root/model_rel
            if existing.is_file() and existing.stat().st_size > 1024*1024:
                onnx=existing.resolve()
                print(f'  [{name}] використовую наявний ONNX: {onnx.name}')
            else:
                onnx=Path(hf_hub_download(TTS_REPO,model_rel,revision=commit,local_dir=local_root)).resolve()
            if not onnx.is_file() or onnx.stat().st_size < 1024*1024:
                raise RuntimeError('Неповна або відсутня ONNX-модель: '+str(onnx))

            # Малий JSON через urllib, без несумісного httpx2/Brotli.
            cfg_path=_download_hf_small_file(TTS_REPO,commit,config_rel,local_root)
            prefix=model_rel.rsplit('/',1)[0]
            for card in [prefix+'/MODEL_CARD',prefix+'/MODEL_CARD.md',prefix+'/MODEL_CARD.txt']:
                if card in listing:
                    try:
                        _download_hf_small_file(TTS_REPO,commit,card,local_root)
                    except Exception as exc:
                        print(f'  Попередження: картку голосу {name} не отримано: {exc}')
                    break

        if not _piper_json_ok(cfg_path):
            raise RuntimeError('Конфіг Piper відсутній або некоректний: '+str(cfg_path))
        local[name]=onnx
        meta.append({'voice':name,'model_file':str(onnx),'model_sha256':_file_sha256(onnx),
                     'config_file':str(cfg_path),'config_sha256':_file_sha256(cfg_path),
                     'source_repo':TTS_REPO,'revision':commit,'hf_path':model_rel})
        print(f'  [{name}] OK: ONNX + JSON')

    mm=pd.DataFrame(meta)
    if mm.model_sha256.nunique()!=len(mm):
        raise RuntimeError('Використано один ONNX під різними голосовими ID!')
    mm.to_csv(MATCHED_OUT/'V6_TTS_MODEL_PROVENANCE.csv',index=False)
    return local,mm

requested_voices=sorted(q.tts_voice_id.unique())
print('Потрібні голоси:',requested_voices)
PIPER_MODEL_FILES, piper_provenance = setup_piper_models(requested_voices) if RUN_TTS_GENERATION else ({},pd.DataFrame())
if RUN_TTS_GENERATION:
    _piper_command=_piper_cli()
    display(piper_provenance[['voice','source_repo','revision','model_sha256']])
else:
    print('RUN_TTS_GENERATION=False: створено лише чергу; WAV ще не генерувалися.')

## 5. Text → TTS: генерація WAV, перевірка тривалості та SHA-256

Кожен файл записується спершу до тимчасового WAV, перевіряється `soundfile` і лише потім атомарно переноситься до каталогу пар. **Результати попередніх запусків перевіряються й повторно не синтезуються.** Некоректні або надто довгі аудіо отримують статус `REJECTED_QC`, а не потрапляють до навчання. Для кожної успішної пари зберігаємо текст, мовець-джерело, TTS-голос, SHA моделі, SHA WAV, тривалість та split.

In [ ]:
import soundfile as sf

def produce_v6_tts_receipts(queue, model_files):
    dest_dir=MATCHED_OUT/'paired_tts_generated'
    dest_dir.mkdir(exist_ok=True)
    receipts_path=MATCHED_OUT/'V6_TTS_RECEIPTS.csv'
    statuses_path=MATCHED_OUT/'V6_TTS_GENERATION_STATUS.csv'
    old=pd.read_csv(receipts_path,dtype=str,keep_default_na=False) if receipts_path.exists() else pd.DataFrame()
    previously={str(x.pair_id):x for x in old.itertuples()} if not old.empty else {}
    found=[];statuses=[]
    cli=_piper_cli()
    for i,r in enumerate(queue.itertuples(),1):
        voice=str(r.tts_voice_id)
        onnx=model_files[voice]
        model_sha=_file_sha256(onnx)
        wav=dest_dir/f'{r.pair_id}_{voice}.wav'
        tmp=dest_dir/f'.{r.pair_id}_{voice}.tmp.wav'
        if tmp.exists():tmp.unlink()
        old_row=previously.get(str(r.pair_id))
        if old_row and wav.is_file() and _file_sha256(wav)==str(old_row.sha256) and str(old_row.tts_model_file_sha256)==model_sha:
            receipt=old_row._asdict()
            found.append(receipt)
            statuses.append({'pair_id':r.pair_id,'status':'REUSED_VERIFIED','reason':''})
            continue
        try:
            if wav.exists() and old_row is not None:
                raise RuntimeError('WAV вже існує, але контрольний SHA/модель не відповідає receipts; не перезаписуємо.')
            if wav.exists() and old_row is None:
                raise RuntimeError('WAV уже існує без receipts. Потрібна ручна перевірка; не перезаписуємо.')
            proc=subprocess.run(cli+['--model',str(onnx),'--config',str(Path(str(onnx)+'.json')),
                                    '--output_file',str(tmp)],input=str(r.text_normalized),
                               text=True,capture_output=True,timeout=240)
            if proc.returncode != 0 or not tmp.exists():
                raise RuntimeError(f'Piper exit={proc.returncode}; '+proc.stderr[-350:])
            audio_info=sf.info(str(tmp))
            dur=float(audio_info.duration)
            duration_real=float(r.duration_sec)
            dur_ratio=dur/max(duration_real,.001)
            if not MIN_TTS_DURATION_SEC <= dur <= MAX_TTS_DURATION_SEC:
                raise ValueError(f'TTS duration {dur:.2f}s outside [{MIN_TTS_DURATION_SEC},{MAX_TTS_DURATION_SEC}]')
            if not (1/MAX_DURATION_RATIO <= dur_ratio <= MAX_DURATION_RATIO):
                raise ValueError(f'Extreme REAL/TTS duration ratio {dur_ratio:.2f}')
            if not int(audio_info.frames)>0 or not int(audio_info.channels)>=1:
                raise ValueError('Empty or invalid generated audio')
            tmp.replace(wav)
            receipt={'pair_id':str(r.pair_id),'original_path':str(r.path),'generated_path':str(wav),
                     'original_group_id':str(r.group_id),'split':str(r.split), 'source':str(r.source),
                     'model_id':'Piper@'+TTS_REPO,'voice_id':voice,'text':str(r.text_normalized),
                     'sha256':_file_sha256(wav),'tts_model_file_sha256':model_sha,
                     'duration_real_sec':duration_real,'duration_tts_sec':dur,
                     'source_license':str(r.license),'source_approval_status':str(r.approval_status)}
            found.append(receipt)
            statuses.append({'pair_id':r.pair_id,'status':'CREATED','reason':''})
        except Exception as exc:
            statuses.append({'pair_id':r.pair_id,'status':'REJECTED_QC','reason':str(exc)[:300]})
            if tmp.exists():tmp.unlink()
        if i%10==0 or i==len(queue):
            pd.DataFrame(found).to_csv(receipts_path,index=False) if found else None
            pd.DataFrame(statuses).to_csv(statuses_path,index=False)
            print(f'TTS: {i}/{len(queue)}, accepted={len(found)}, rejected={sum(x["status"]=="REJECTED_QC" for x in statuses)}')
    if not found: raise RuntimeError('Жоден WAV не пройшов перевірку. Див. V6_TTS_GENERATION_STATUS.csv')
    receipts=pd.DataFrame(found)
    if receipts.duplicated('pair_id').any():raise AssertionError('DUPLICATE pair_id in receipts')
    assert receipts.groupby('voice_id').split.nunique().le(1).all()
    receipts.to_csv(receipts_path,index=False)
    return receipts,pd.DataFrame(statuses)

if RUN_TTS_GENERATION:
    receipts, generation_status=produce_v6_tts_receipts(q,PIPER_MODEL_FILES)
    print('REAL–TTS пари з готовим WAV:',len(receipts))
    display(receipts.groupby(['split','voice_id']).size().rename('pairs').reset_index())
else:
    receipts_path=MATCHED_OUT/'V6_TTS_RECEIPTS.csv'
    if not receipts_path.exists():
        raise RuntimeError('Реальної генерації TTS ще немає. Увімкніть RUN_TTS_GENERATION=True або додайте перевірені WAV.')
    receipts=pd.read_csv(receipts_path,dtype=str,keep_default_na=False)


## 6. Аудит сформованого підкорпусу та прослуховування

Тут відображаються пари *REAL ↔ FAKE того самого тексту* й графіки двох сигналів. Аудіоплеєри працюють із реальними WAV на вашому сервері. За недоступності голосів у validation/hidden ця перевірка зупиняється; **не копіюйте один голос під різними іменами**.

In [ ]:
from scipy.signal import stft
import matplotlib.pyplot as plt

if len(receipts) < 12:
    raise RuntimeError(f'Замало підтверджених REAL–TTS пар: {len(receipts)} (<12).')
q_check=q[q.pair_id.isin(receipts.pair_id)].copy()
assert not q_check.groupby('group_id').split.nunique().gt(1).any()
assert not q_check.groupby('text_key').split.nunique().gt(1).any()
assert not receipts.groupby('voice_id').split.nunique().gt(1).any()
assert q_check.pair_id.nunique()==receipts.pair_id.nunique()
matched_report=q_check.groupby('split').agg(pairs=('pair_id','nunique'),real_speakers=('speaker_id','nunique'),
                                            texts=('text_key','nunique'),fake_voices=('tts_voice_id','nunique')).reset_index()
matched_report.to_csv(MATCHED_OUT/'V6_MATCHED_PAIR_COUNTS.csv',index=False)
print('Фактичний matched-підкорпус:');display(matched_report)
print('ВАЖЛИВО: незалежних TTS-голосів adaptation/validation/hidden може бути замало для голосового bootstrap.')

def _listen_pair(pair_id):
    from IPython.display import Audio
    r=q_check[q_check.pair_id.eq(pair_id)].iloc[0]
    f=receipts[receipts.pair_id.eq(pair_id)].iloc[0]
    print('Текст:',r.text_normalized,'|','TTS:',f.voice_id)
    print('REAL');display(Audio(filename=str(r.path)))
    print('FAKE');display(Audio(filename=str(f.generated_path)))
    fig,axes=plt.subplots(2,2,figsize=(12,5),constrained_layout=True)
    for i,p in enumerate([r.path,f.generated_path]):
        x,sr=sf.read(str(p),always_2d=True)
        sig=x.mean(axis=1)
        axes[i,0].plot(np.arange(len(sig))/sr,sig,lw=.5)
        axes[i,0].set(title=['REAL waveform','FAKE waveform'][i],xlabel='Час, с')
        _,tt,zz=stft(sig,sr,nperseg=min(512,len(sig)))
        axes[i,1].pcolormesh(tt,np.arange(zz.shape[0])*sr/min(512,len(sig)),
                             20*np.log10(np.abs(zz)+1e-8),shading='auto',cmap='viridis',vmin=-100,vmax=-20)
        axes[i,1].set(title='Спектрограма',xlabel='Час, с',ylabel='Гц',ylim=(0,min(8000,sr/2)))
    plt.show()

for idx in q_check[q_check.split.eq('train')].pair_id.head(2):_listen_pair(idx)

## 7. Незалежні групи та парне балансування 50/50

Беремо **лише реально згенеровані** й перевірені CSV receipts. Для кожної пари обидва класи знаходяться в одному split, а фрагменти отримують однакову кількість елементів REAL/FAKE. Далі adaptation формується тільки з train; validation та hidden лишаються окремими.

In [ ]:
selected_mode='matched_tts'
OUT=MATCHED_OUT
cfg=replace(cfg_tts,out=str(OUT),seconds=8.0,epochs=EPOCHS)
receipts_file=OUT/'V6_TTS_RECEIPTS.csv'
parents=A.integrate_tts_pairs(matched_parent_catalog,receipts_file,cfg,min_pairs_per_split=3)
folds=A.create_matched_adaptation_fold(parents,cfg,seed=SEED_SELECTION,target_fraction=.22)
segments,segment_counts,segment_report=M.expand_nonoverlapping_segments(
    folds,cfg,target=TARGET_SEGMENTS,max_per_parent=1)
splits=A.balance_pairs(segments,cfg,seed=SEED_SELECTION)
assert splits.groupby('pair_id').split.nunique().max()==1
assert splits.groupby('group_id').split.nunique().max()==1
assert splits.groupby('parent_file_id').split.nunique().max()==1
assert splits.groupby('text_key').split.nunique().max()==1
counts=splits.groupby(['split','label']).size().unstack(fill_value=0).reindex(columns=[0,1],fill_value=0)
assert (counts[0]==counts[1]).all(), 'Баланс REAL/FAKE порушено'
splits.to_csv(OUT/'V6_LOCKED_MATCHED_SPLITS.csv',index=False)
splits.to_csv(OUT/'V5_02_LOCKED_EXPERIMENT_SEGMENTS.csv',index=False)
S.lock_hidden_test(splits,cfg)
print('Баланс CLASS REAL=0 / FAKE=1:');display(counts)
print('Підтверджені незалежні групи (а не кількість фрагментів):')
display(splits.groupby(['split','label']).group_id.nunique().rename('groups').reset_index())
A.source_and_split_diagnostics(splits,cfg)
print('⚠️ З 5 доступних Piper-голосів неможливо отримати >=3 незалежні fake-голоси в adaptation, validation та hidden. Це контрольований matched-пілот.')

## 8. Усі 21 акустичний сценарій: графіки та відтворення REAL/FAKE

Два види сценаріїв: **синтез мовлення з того самого тексту** (реальний Piper WAV) і **акустичні впливи** — 5 рівнів шуму, MP3-компресії, реверберації та фільтрації. Тут прослуховуємо також приклади впливів.

In [ ]:
scenario_guide = V.draw_scenario_guide(cfg,tag='unified_scenario_guide')
display(scenario_guide)
if SHOW_AUDIO_AND_PLOTS:
    examples = V.pick_examples(splits, 'validation')
    for record in examples:
        print('REAL' if int(record.label)==0 else 'FAKE', record.relative_path)
        V.show_audio_gallery(record,cfg,tag='unified_gallery',
                             show_players=True,show_figures=True,max_players=None)
    try:
        V.make_interactive_player(splits,cfg,'validation')
    except ImportError:
        print('Інтерактивний вибір аудіо потребує ipywidgets; звичайні аудіоплеєри вже доступні.')
else:
    print('Плеєри та великі ілюстрації вимкнено. Для показу встановіть SHOW_AUDIO_AND_PLOTS=True.')

## 9. Дисертаційне порівняння E1 / E2 / E3

E3 справді **переналаштовує розподіл сценаріїв протягом навчання** на adaptation, E2 використовує рівномірні випадкові впливи, E1 — без аугментації. У всіх трьох однакові seeds, епохи, моделі, сценарії та бюджет. Результати для matched-Piper з недостатньою різноманітністю генераторів називаємо **контрольованим пілотом**, а не остаточним доказом узагальнення.

In [ ]:
validation_predictions = None
if RUN_TRAINING:
    if (OUT/'V5_HIDDEN_FROZEN_PROTOCOL.json').exists() or (OUT/'V5_HIDDEN_TEST_STARTED.json').exists():
        raise RuntimeError('Протокол hidden зафіксовано/відкрито: повторне навчання заборонене. ' 
                           'Встановіть RUN_TRAINING=False для перегляду результатів.')
    print('Починаю фактичне навчання:', selected_mode, MODELS, SEEDS, EPOCHS, 'епох')
    validation_predictions, adaptation_predictions = A.evaluate_all_seeds(
        splits,cfg,models=MODELS,seeds=SEEDS,epochs=EPOCHS,
        update_every=UPDATE_EVERY,adapt_max_per_class=ADAPT_MAX_PER_CLASS)
    print('Отримано прогнозів validation:',len(validation_predictions))
else:
    file=OUT/'V5_04_VALIDATION_PREDICTIONS_ALL.csv'
    if file.exists():
        validation_predictions=pd.read_csv(file,low_memory=False)
        print('Навчання не виконувалось: завантажено існуючі validation-прогнози.')
    else:
        print('Навчання вимкнене, збережених прогнозів ще немає.')

## 7. ГОЛОВНИЙ РЕЗУЛЬТАТ ДИСЕРТАЦІЇ — ΔASRI = ASRI(E3) − ASRI(E2)

Сценарні метрики та ASRI рахуються окремо за кожним seed. Позитивний ΔASRI — лише спостереження; незалежність голосів і груповий bootstrap перевіряються окремо. За недостатньої кількості груп програма повертає `INSUFFICIENT_INDEPENDENT_VOICES`, а не вигадану статистичну значущість.

In [ ]:
if validation_predictions is not None:
    scenario_metrics,asri_by_seed,paired_deltas,delta_summary = A.metric_tables(validation_predictions,cfg)
    display(asri_by_seed.sort_values(['model','seed','regime']))
    display(paired_deltas)
    display(delta_summary)
    coverage=(validation_predictions.groupby(['seed','model','regime'])
       .agg(n_scenarios=('scenario_id',lambda x:validation_predictions.loc[x.index,['category','level']].drop_duplicates().shape[0]),
            n_predictions=('scenario_id','size')).reset_index())
    coverage.to_csv(OUT/'V5_05_SCENARIO_COVERAGE.csv',index=False)
    display(coverage)
    assert coverage.n_scenarios.eq(21).all(), 'Не всі 21 сценарій оцінено'
    assert asri_by_seed.complete_cells.eq(20).all(), 'Неповна матриця ASRI'
    bootstrap = A.stratified_voice_bootstrap(validation_predictions,cfg,
                  B=BOOTSTRAPS,seed=2026,min_groups_per_class=3)
    display(bootstrap)
    if (bootstrap.status != 'OK').any():
        print('⚠️ Недостатньо незалежних голосів для підтверджувального групового bootstrap.')
    group_counts=validation_predictions.groupby('label').group_id.nunique()
    if len(group_counts)==2 and group_counts.min()>=3:
        for seed,g in validation_predictions.groupby('seed'):
            status=M.interaction_statistics(g,cfg,tag=f'unified_validation_seed{seed}')
            print('GEE seed',seed, status.get('status'))
    else:
        print('GEE не використовується: недостатньо незалежних REAL/FAKE груп.')
else:
    print('Метрики недоступні: спершу потрібен завершений етап навчання.')

## 8. ГРАФІКИ ВСІХ СЦЕНАРІЇВ, АДАПТИВНИХ ВАГ І ПОРІВНЯННЯ МОДЕЛЕЙ

Відтворення аудіо доступне вище; тут — теплові карти, падіння точності, ASRI, матриці помилок і журнал зміни ваг E3. Це **ті самі validation**, що використані в обчисленнях вище.

In [ ]:
if validation_predictions is not None and SHOW_FULL_REPORT_PLOTS:
    import matplotlib.pyplot as plt
    for seed,g in validation_predictions.groupby('seed'):
        V.report_scenario_tests(g,cfg,tag=f'unified_validation_seed{seed}',show=True)
    fig,ax=plt.subplots(figsize=(9,4.5))
    for name,g in paired_deltas.groupby('model'):
        ax.plot(g.seed,g.delta_adaptive_minus_random,marker='o',label=name)
    ax.axhline(0,color='black',lw=1)
    ax.set(title='Дисертаційна гіпотеза: ΔASRI E3 − E2 за seeds',
           xlabel='Seed',ylabel='ΔASRI')
    ax.legend();ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(OUT/'V5_06_ADAPTIVE_EFFECT_BY_SEED.png',dpi=160)
    plt.show()
    for model in MODELS:
        for seed in SEEDS:
            h=OUT/'models'/f'{model}_adaptive_seed{seed}_policy_history.csv'
            if h.exists():
                history=pd.read_csv(h)
                if not history.empty:
                    weights=history.drop_duplicates(['epoch','category']).pivot(
                        index='epoch',columns='category',values='category_weight')
                    ax=weights.plot(marker='o',figsize=(8,3.8),
                      title=f'Адаптивні ваги E3: {model}, seed={seed}')
                    ax.set(xlabel='Епоха',ylabel='Вага категорії')
                    ax.set_ylim(0,1);ax.grid(alpha=.2)
                    plt.tight_layout()
                    plt.savefig(OUT/f'V5_06_POLICY_{model}_seed{seed}.png',dpi=150)
                    plt.show()
else:
    print('Графіки моделей недоступні або вимкнені.')

## 12. Підсумки, оцінка готовності та один ZIP для перевірки

ZIP містить таблиці, графіки, походження голосових моделей та оцінювання за всіма сценаріями. **Самі вихідні WAV, персональні аудіо AUA та великі файли моделей не експортуємо.** Звіти зберігаються тільки в `audio_real_fake/results`.

In [ ]:
info,readiness=A.research_readiness(splits,cfg,mode=selected_mode,
      trained_seeds=(SEEDS if validation_predictions is not None else ()))
print('Обраний режим:',selected_mode)
print('Готовність до строгого дослідження:')
display(pd.DataFrame([readiness]).T.rename(columns={0:'value'}))
if validation_predictions is not None:
    wins=int(paired_deltas.delta_adaptive_minus_random.gt(0).sum())
    print('Позитивних model×seed порівнянь E3−E2:',wins,'/',len(paired_deltas))
if selected_mode!='matched_tts':
    print('УВАГА: це PILOT, не незалежне підтвердження дисертаційної гіпотези.')
report_zip=RESULTS_ROOT/f'SAADE_MATCHED_TTS_v6_{selected_mode}_FOR_REVIEW.zip'
include_ext={'.csv','.json','.png','.txt','.md'}
report_files=[p for p in OUT.rglob('*') if p.is_file() and p.suffix.lower() in include_ext
              and 'cache' not in p.parts and 'paired_tts_generated' not in p.parts]
with ZipFile(report_zip,'w',ZIP_DEFLATED) as archive:
    for p in report_files:
        archive.write(p,arcname=str(p.relative_to(OUT)))
print('АРХІВ ДЛЯ ПЕРЕВІРКИ:',report_zip)
print('Результати:',OUT,'| файлів у звіті:',len(report_files))
print('Hidden test:', 'ЩЕ НЕ ВІДКРИТО')

## 13. Захищений одноразовий hidden test

**Залиште його закритим** для пілотного matched-підкорпусу з менш ніж трьома незалежними голосами FAKE у validation/hidden. Статистична значущість за фрагментами не компенсує нестачу незалежних голосів.

In [ ]:
if not FREEZE_FOR_HIDDEN and not RUN_HIDDEN_ONCE:
    print('Hidden test закрито. Це правильний стан під час підготовки та валідації.')
else:
    if selected_mode != 'matched_tts':
        raise RuntimeError('Hidden test дозволено лише для підтвердженого REAL–TTS експерименту.')
    if FREEZE_FOR_HIDDEN:
        assert readiness['candidate_for_strict_review'], 'Немає наукової готовності. Перевірте V5_RESEARCH_READINESS.json'
        frozen=A.freeze_for_hidden_once(splits,cfg,models=MODELS,seeds=SEEDS,
                  epochs=EPOCHS,mode='matched_tts',min_voice_groups=3)
        print('Контрольні суми протоколу та ваг зафіксовано:',frozen)
    if RUN_HIDDEN_ONCE:
        if HIDDEN_CONFIRMATION != 'ПІДТВЕРДЖУЮ_ОДНОРАЗОВИЙ_HIDDEN_TEST':
            raise PermissionError('Для одноразового hidden test потрібна точна фраза HIDDEN_CONFIRMATION.')
        if not (OUT/'V5_HIDDEN_FROZEN_PROTOCOL.json').exists():
            raise RuntimeError('Спочатку зафіксуйте протокол і ваги.')
        hidden_summary=A.run_hidden_once(splits,cfg,authorization=True)
        display(hidden_summary)
        print('Hidden test виконано ОДИН РАЗ. Повторний запуск заборонено.')

---

### Статус доказовості

- REAL–TTS пари синтезуються **справжнім Piper**; якщо генерації немає, навчання в `matched_tts` не виконується.
- Класовий баланс **50/50**, текст і людський мовець не перетинаються між fold.
- Текстова тотожність не усуває повністю відмінності каналу/запису. Тут усі REAL із Common Voice, всі FAKE з Piper: **конфаундинг джерела/генератора ще залишається**.
- П'ять Piper голосів розподілені між train, adaptation, validation, hidden; у validation та hidden по одному голосу. Це **не достатньо** для достовірних CI за голосами та переконливого обґрунтування широкого узагальнення.
- Для строгого дисертаційного тесту додайте незалежні голоси/моделі з іншими TTS-рушіями, підтверджуйте license/model hashes, повторіть експеримент у новому каталозі та виконуйте hidden test після фіксації протоколу.
